# MLB Race to October — Data Foundation

Stages all data assets needed by the **MLB Race to October** lab into GCS.
This notebook is the source of truth for how lab data was prepared. **Students do not run this notebook** — they consume its outputs.

## Data sources

| Source | What | Stewardship |
|---|---|---|
| Lahman Baseball Database | Historical season-level batting, pitching, team records | [Chadwick Bureau](https://github.com/chadwickbureau/baseballdatabank), CC BY-SA 3.0 |
| MLB Rulebook | Official rules PDF for retrieval grounding | MLB.com (used under partnership) |
| MLB Stats API | Live current-season stats | statsapi.mlb.com (used under partnership) |

## Outputs (in `gs://class-demo/mlb-race-to-october/`)

- `lahman/` — Parquet files with agent-friendly column names
- `rulebook/` — official MLB rulebook PDF(s)
- `profiles/` — generated player and team profile documents
- `schema/` — column descriptions for the `ALTER TABLE` Task 1 emits

## Attribution

The Lahman Baseball Database is licensed under CC BY-SA 3.0. Original collection by Sean Lahman, currently maintained by the Chadwick Baseball Bureau.


## 1. Configuration

Set your project before running anything else. The bucket and prefix are fixed by the lab; region is `US` for GCS+BigQuery (Gemini Enterprise / CX Agent Studio data stores will live in `global` separately).


In [ ]:
# --- Project / GCS ---
import google.auth
_, PROJECT_ID = google.auth.default()
GCS_BUCKET = "class-demo"
GCS_PREFIX = "mlb-race-to-october"
BQ_LOCATION = "US"  # where BigQuery dataset will live in Task 1

# --- Local working dir (inside the Colab Enterprise runtime VM) ---
WORK_DIR = "/content/lahman"

# --- Lahman source ---
# Lahman is now maintained by SABR (sabr.org/lahman-database/), distributed via Box.
# We stage the archive in GCS once so the notebook has a stable, project-controlled source.
LAHMAN_GCS_SOURCE_PREFIX = f"{GCS_PREFIX}/lahman-source"

# --- Tables we're keeping (decided in planning conversation) ---
# Core 5: People, Batting, Pitching, Teams, Appearances
# Narrative: AllstarFull, AwardsPlayers
# Postseason: BattingPost, PitchingPost
LAHMAN_TABLES = [
    "People",
    "Batting",
    "Pitching",
    "Teams",
    "Appearances",
    "AllstarFull",
    "AwardsPlayers",
    "BattingPost",
    "PitchingPost",
]

assert PROJECT_ID, "Set PROJECT_ID above before running the rest of the notebook."
print(f"Project:        {PROJECT_ID}")
print(f"GCS target:     gs://{GCS_BUCKET}/{GCS_PREFIX}/")
print(f"BQ location:    {BQ_LOCATION}")
print(f"Tables to stage: {len(LAHMAN_TABLES)}")


Project:        qwiklabs-gcp-00-dd506b084f93
GCS target:     gs://class-demo/mlb-race-to-october/
BQ location:    US
Tables to stage: 9


In [ ]:
from google.cloud.storage import Client
from urllib3.util import Retry
from urllib3 import PoolManager

# Increase HTTP pool size to match our worker count
import google.auth.transport.requests
google.auth.transport.requests.requests.adapters.DEFAULT_POOLSIZE = 50

## 2. Imports and GCS sanity check

Colab Enterprise runtimes auth as a service account that has access to your project. We confirm GCS is reachable up front so we fail fast on permissions, not after a long download.


In [ ]:
import os
import tarfile
import urllib.request
from pathlib import Path

import pandas as pd
from google.cloud import storage

os.makedirs(WORK_DIR, exist_ok=True)

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(GCS_BUCKET)
assert bucket.exists(), f"Bucket gs://{GCS_BUCKET} not reachable from project {PROJECT_ID}."

# Reload to populate metadata (location, storage class) from the API.
bucket.reload()
print(f"Bucket gs://{GCS_BUCKET} OK")
print(f"  location:      {bucket.location}")
print(f"  storage_class: {bucket.storage_class}")
print(f"Working dir:    {WORK_DIR}")


Bucket gs://class-demo OK
  location:      US-CENTRAL1
  storage_class: STANDARD
Working dir:    /content/lahman


## 3. Download the Lahman release

Source CSVs were staged in GCS by the instructor from SABR's January 2026 release. We fetch only the 9 tables we need."


In [ ]:
CORE_DIR = Path(WORK_DIR)
CORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Fetching CSVs from gs://{GCS_BUCKET}/{LAHMAN_GCS_SOURCE_PREFIX}/")
for table in LAHMAN_TABLES:
    blob = bucket.blob(f"{LAHMAN_GCS_SOURCE_PREFIX}/{table}.csv")
    if not blob.exists():
        raise FileNotFoundError(
            f"Missing: gs://{GCS_BUCKET}/{LAHMAN_GCS_SOURCE_PREFIX}/{table}.csv\n"
            f"Did you upload the SABR CSVs to that prefix? See section 3 markdown."
        )
    blob.download_to_filename(CORE_DIR / f"{table}.csv")
    print(f"  ✓ {table}.csv")

print(f"\nDownloaded {len(LAHMAN_TABLES)} CSVs to {CORE_DIR}")

Fetching CSVs from gs://class-demo/mlb-race-to-october/lahman-source/
  ✓ People.csv
  ✓ Batting.csv
  ✓ Pitching.csv
  ✓ Teams.csv
  ✓ Appearances.csv
  ✓ AllstarFull.csv
  ✓ AwardsPlayers.csv
  ✓ BattingPost.csv
  ✓ PitchingPost.csv

Downloaded 9 CSVs to /content/lahman


## 4. Inspect the raw release

Before we transform anything, look at what Chadwick actually shipped:

1. **All 9 target tables present?** Sanity check the table list survives whatever Chadwick has done lately.
2. **How fresh is the data?** Max `yearID` across tables answers our "how stale is Lahman?" question — this is the deciding input for the MLB API backfill discussion later.
3. **Approximate row counts** — quick gut check on what we're about to transform.


In [ ]:
# What tables does the release actually contain?
all_csvs = sorted(b.name.split("/")[-1].replace(".csv", "")
                  for b in client.list_blobs(GCS_BUCKET, prefix=LAHMAN_GCS_SOURCE_PREFIX)
                  if b.name.endswith(".csv"))
print(f"Total CSVs in release: {len(all_csvs)}")
print(f"  {', '.join(all_csvs)}\n")

missing = [t for t in LAHMAN_TABLES if t not in all_csvs]
if missing:
    print(f"\u26a0\ufe0f  MISSING from release: {missing}")
else:
    print(f"\u2713 All {len(LAHMAN_TABLES)} target tables present.\n")

# Per-table summary: rows, cols, year range
print(f"{'Table':<16} {'Rows':>10} {'Cols':>6} {'Year range':>14}")
print("-" * 50)
for t in LAHMAN_TABLES:
    df = pd.read_csv(CORE_DIR / f"{t}.csv", low_memory=False)
    yr = f"{df['yearID'].min()}\u2013{df['yearID'].max()}" if "yearID" in df.columns else "\u2014"
    print(f"{t:<16} {len(df):>10,} {len(df.columns):>6} {yr:>14}")


Total CSVs in release: 27
  AllstarFull, Appearances, AwardsManagers, AwardsPlayers, AwardsShareManagers, AwardsSharePlayers, Batting, BattingPost, CollegePlaying, Fielding, FieldingOF, FieldingOFsplit, FieldingPost, HallOfFame, HomeGames, Managers, ManagersHalf, Parks, People, Pitching, PitchingPost, Salaries, Schools, SeriesPost, Teams, TeamsFranchises, TeamsHalf

✓ All 9 target tables present.

Table                  Rows   Cols     Year range
--------------------------------------------------
People               24,270     25              —
Batting             128,598     22      1871–2025
Pitching             57,630     30      1871–2025
Teams                 3,614     48      1871–2025
Appearances         128,512     21      1871–2025
AllstarFull           6,425      8      1933–2025
AwardsPlayers        12,667      6      1877–2025
BattingPost          18,687     22      1884–2025
PitchingPost          7,474     30      1884–2025


In [ ]:
# Headers only — much faster than reading the data.
for t in LAHMAN_TABLES:
    cols = pd.read_csv(CORE_DIR / f"{t}.csv", nrows=0).columns.tolist()
    print(f"\n{t}  ({len(cols)} cols)")
    print("  " + ", ".join(cols))


People  (25 cols)
  ID, playerID, birthYear, birthMonth, birthDay, birthCity, birthCountry, birthState, deathYear, deathMonth, deathDay, deathCountry, deathState, deathCity, nameFirst, nameLast, nameGiven, weight, height, bats, throws, debut, bbrefID, finalGame, retroID

Batting  (22 cols)
  playerID, yearID, stint, teamID, lgID, G, AB, R, H, 2B, 3B, HR, RBI, SB, CS, BB, SO, IBB, HBP, SH, SF, GIDP

Pitching  (30 cols)
  playerID, yearID, stint, teamID, lgID, W, L, G, GS, CG, SHO, SV, IPouts, H, ER, HR, BB, SO, BAOpp, ERA, IBB, WP, HBP, BK, BFP, GF, R, SH, SF, GIDP

Teams  (48 cols)
  yearID, lgID, teamID, franchID, divID, Rank, G, Ghome, W, L, DivWin, WCWin, LgWin, WSWin, R, AB, H, 2B, 3B, HR, BB, SO, SB, CS, HBP, SF, RA, ER, ERA, CG, SHO, SV, IPouts, HA, HRA, BBA, SOA, E, DP, FP, name, park, attendance, BPF, PPF, teamIDBR, teamIDlahman45, teamIDretro

Appearances  (21 cols)
  yearID, teamID, lgID, playerID, G_all, GS, G_batting, G_defense, G_p, G_c, G_1b, G_2b, G_3b, G_ss, G_lf, G_cf

## 5. Schema definition — agent-friendly column names

Lahman's column names are optimized for compactness, not for an LLM doing NL→SQL. Names like `IPouts`, `BAOpp`, and `G_c` work fine for someone fluent in baseball stat abbreviations, but cost the agent extra reasoning steps and create silent footguns when the same abbreviation means different things in different tables (Pitching's `H` is hits-allowed, Batting's `H` is hits).

The rename map below applies the following conventions:

- **Canonical IDs preserved**: `playerID`, `teamID`, `yearID`, `lgID`, `franchID`, `divID`, `awardID`, `gameID`. These are the universally-recognized Lahman keys; cross-referencing external Lahman materials still works. External IDs (`bbrefID`, `retroID`, `teamIDBR`, etc.) likewise stay as-is — they're literal identifiers in other systems.
- **Snake_case everywhere else**: `home_runs`, `outs_pitched`, `games_catcher`.
- **Self-documenting names**: `RBI` → `runs_batted_in`, `BB` → `walks`, `IPouts` → `outs_pitched`, `BAOpp` → `opponent_batting_avg`.
- **Units made explicit where ambiguous**: `weight` → `weight_lbs`, `height` → `height_inches`.
- **`_allowed` suffix on Pitching's negative-valence stats**: `hits_allowed`, `home_runs_allowed`, `walks_allowed`, `runs_allowed`, etc. Without this an agent could silently sum `b.home_runs + p.home_runs` and produce nonsense — batter HRs and HRs-allowed are not the same quantity.
- **Type conversions deferred to the transform cell**: `Y`/`N` flags become BOOL, `debut`/`finalGame` become DATE. Done at transform time, not here, because the rename map is just text.
- **Synthesized columns**: a `name_full` column is added to People in the transform step (concat of `name_first` + `name_last`) for natural-language queries like "Smith pitchers with 100+ wins."
- **Dropped columns**: any column not in this map is dropped. Notably, People's leading `ID` (an auto-increment artifact in SABR's build) is dropped — `playerID` is the natural key.

In [ ]:
# Per-table rename maps. Any column not listed is dropped during transformation.
# Apply with: df.rename(columns=RENAME_MAPS[t])[list(RENAME_MAPS[t].values())]
# (the slice both drops unmapped columns and orders to match)

RENAME_MAPS = {
    "People": {
        # leading "ID" dropped — playerID is the natural key
        "playerID": "playerID",
        "birthYear": "birth_year",
        "birthMonth": "birth_month",
        "birthDay": "birth_day",
        "birthCity": "birth_city",
        "birthState": "birth_state",
        "birthCountry": "birth_country",
        "deathYear": "death_year",
        "deathMonth": "death_month",
        "deathDay": "death_day",
        "deathCity": "death_city",
        "deathState": "death_state",
        "deathCountry": "death_country",
        "nameFirst": "name_first",
        "nameLast": "name_last",
        "nameGiven": "name_given",
        "weight": "weight_lbs",
        "height": "height_inches",
        "bats": "bats",                       # 'L'/'R'/'B' — described in column comments
        "throws": "throws",                    # 'L'/'R'
        "debut": "debut_date",                 # cast to DATE during transform
        "finalGame": "final_game_date",        # cast to DATE during transform
        "bbrefID": "bbrefID",
        "retroID": "retroID",
        # name_full is synthesized post-rename in the transform cell
    },

    "Batting": {
        "playerID": "playerID",
        "yearID": "yearID",
        "stint": "stint",                      # 1, 2, ... within a season if traded mid-year
        "teamID": "teamID",
        "lgID": "lgID",
        "G": "games",
        "AB": "at_bats",
        "R": "runs",
        "H": "hits",
        "2B": "doubles",
        "3B": "triples",
        "HR": "home_runs",
        "RBI": "runs_batted_in",
        "SB": "stolen_bases",
        "CS": "caught_stealing",
        "BB": "walks",
        "SO": "strikeouts",
        "IBB": "intentional_walks",
        "HBP": "hit_by_pitch",
        "SH": "sacrifice_hits",
        "SF": "sacrifice_flies",
        "GIDP": "grounded_into_double_play",
    },

    "Pitching": {
        "playerID": "playerID",
        "yearID": "yearID",
        "stint": "stint",
        "teamID": "teamID",
        "lgID": "lgID",
        "W": "wins",
        "L": "losses",
        "G": "games",
        "GS": "games_started",
        "CG": "complete_games",
        "SHO": "shutouts",
        "SV": "saves",
        "IPouts": "outs_pitched",              # innings × 3
        "H": "hits_allowed",
        "ER": "earned_runs_allowed",
        "HR": "home_runs_allowed",
        "BB": "walks_allowed",
        "SO": "strikeouts",                    # pitcher's Ks — positive for pitcher, no _allowed
        "BAOpp": "opponent_batting_avg",
        "ERA": "earned_run_avg",
        "IBB": "intentional_walks_allowed",
        "WP": "wild_pitches",
        "HBP": "batters_hit_by_pitch",         # batters this pitcher hit
        "BK": "balks",
        "BFP": "batters_faced",
        "GF": "games_finished",
        "R": "runs_allowed",
        "SH": "sacrifice_hits_allowed",
        "SF": "sacrifice_flies_allowed",
        "GIDP": "double_plays_induced",
    },

    "Teams": {
        "yearID": "yearID",
        "lgID": "lgID",
        "teamID": "teamID",
        "franchID": "franchID",
        "divID": "divID",
        "Rank": "division_rank",
        "G": "games",
        "Ghome": "home_games",
        "W": "wins",
        "L": "losses",
        "DivWin": "won_division",              # 'Y'/'N' → BOOL during transform
        "WCWin": "won_wild_card",              # 'Y'/'N' → BOOL during transform
        "LgWin": "won_league",                 # 'Y'/'N' → BOOL during transform
        "WSWin": "won_world_series",           # 'Y'/'N' → BOOL during transform
        # Batting (team-level offense)
        "R": "runs_scored",
        "AB": "at_bats",
        "H": "hits",
        "2B": "doubles",
        "3B": "triples",
        "HR": "home_runs",
        "BB": "walks",
        "SO": "strikeouts",                    # team batters' Ks
        "SB": "stolen_bases",
        "CS": "caught_stealing",
        "HBP": "hit_by_pitch",
        "SF": "sacrifice_flies",
        # Pitching (team-level defense)
        "RA": "runs_allowed",
        "ER": "earned_runs_allowed",
        "ERA": "earned_run_avg",
        "CG": "complete_games",
        "SHO": "shutouts",
        "SV": "saves",
        "IPouts": "outs_pitched",
        "HA": "hits_allowed",
        "HRA": "home_runs_allowed",
        "BBA": "walks_allowed",
        "SOA": "strikeouts_pitched",           # team pitchers' Ks (distinct from batters')
        # Fielding
        "E": "errors",
        "DP": "double_plays",
        "FP": "fielding_pct",
        # Park / metadata
        "name": "team_name",
        "park": "park_name",
        "attendance": "home_attendance",
        "BPF": "batting_park_factor",
        "PPF": "pitching_park_factor",
        "teamIDBR": "teamIDBR",
        "teamIDlahman45": "teamIDlahman45",
        "teamIDretro": "teamIDretro",
    },

    "Appearances": {
        "yearID": "yearID",
        "teamID": "teamID",
        "lgID": "lgID",
        "playerID": "playerID",
        "G_all": "games_total",
        "GS": "games_started",
        "G_batting": "games_batting",
        "G_defense": "games_defense",
        "G_p": "games_pitcher",
        "G_c": "games_catcher",
        "G_1b": "games_first_base",
        "G_2b": "games_second_base",
        "G_3b": "games_third_base",
        "G_ss": "games_shortstop",
        "G_lf": "games_left_field",
        "G_cf": "games_center_field",
        "G_rf": "games_right_field",
        "G_of": "games_outfield",              # LF + CF + RF total
        "G_dh": "games_designated_hitter",
        "G_ph": "games_pinch_hitter",
        "G_pr": "games_pinch_runner",
    },

    "AllstarFull": {
        "playerID": "playerID",
        "yearID": "yearID",
        "gameNum": "game_num",                 # 1 or 2 (some seasons had two ASGs)
        "gameID": "gameID",
        "teamID": "teamID",
        "lgID": "lgID",
        "GP": "games_played",                  # 1 if appeared, 0/NULL if selected only
        "startingPos": "starting_position",    # 1-9 if started, NULL otherwise
    },

    "AwardsPlayers": {
        "playerID": "playerID",
        "awardID": "awardID",
        "yearID": "yearID",
        "lgID": "lgID",
        "tie": "tie",                          # 'Y'/'N' for shared awards
        "notes": "notes",
    },

    "BattingPost": {
        "yearID": "yearID",
        "round": "round",                      # WS, NLCS, ALCS, NLDS, ALDS, NLWC, ALWC, etc.
        "playerID": "playerID",
        "teamID": "teamID",
        "lgID": "lgID",
        "G": "games",
        "AB": "at_bats",
        "R": "runs",
        "H": "hits",
        "2B": "doubles",
        "3B": "triples",
        "HR": "home_runs",
        "RBI": "runs_batted_in",
        "SB": "stolen_bases",
        "CS": "caught_stealing",
        "BB": "walks",
        "SO": "strikeouts",
        "IBB": "intentional_walks",
        "HBP": "hit_by_pitch",
        "SH": "sacrifice_hits",
        "SF": "sacrifice_flies",
        "GIDP": "grounded_into_double_play",
    },

    "PitchingPost": {
        "playerID": "playerID",
        "yearID": "yearID",
        "round": "round",
        "teamID": "teamID",
        "lgID": "lgID",
        "W": "wins",
        "L": "losses",
        "G": "games",
        "GS": "games_started",
        "CG": "complete_games",
        "SHO": "shutouts",
        "SV": "saves",
        "IPouts": "outs_pitched",
        "H": "hits_allowed",
        "ER": "earned_runs_allowed",
        "HR": "home_runs_allowed",
        "BB": "walks_allowed",
        "SO": "strikeouts",
        "BAOpp": "opponent_batting_avg",
        "ERA": "earned_run_avg",
        "IBB": "intentional_walks_allowed",
        "WP": "wild_pitches",
        "HBP": "batters_hit_by_pitch",
        "BK": "balks",
        "BFP": "batters_faced",
        "GF": "games_finished",
        "R": "runs_allowed",
        "SH": "sacrifice_hits_allowed",
        "SF": "sacrifice_flies_allowed",
        "GIDP": "double_plays_induced",
    },
}

# Quick sanity print: every table in LAHMAN_TABLES has a rename map
missing_maps = [t for t in LAHMAN_TABLES if t not in RENAME_MAPS]
assert not missing_maps, f"No rename map for: {missing_maps}"

total_cols = sum(len(m) for m in RENAME_MAPS.values())
print(f"Loaded rename maps for {len(RENAME_MAPS)} tables, {total_cols} columns total.")

Loaded rename maps for 9 tables, 211 columns total.


## 5. Schema definition — agent-friendly column names

Lahman's column names are optimized for compactness, not for an LLM doing NL→SQL. Names like `IPouts`, `BAOpp`, and `G_c` work fine for someone fluent in baseball stat abbreviations, but cost the agent extra reasoning steps and create silent footguns when the same abbreviation means different things in different tables (Pitching's `H` is hits-allowed, Batting's `H` is hits).

The rename map below applies the following conventions:

- **Canonical IDs preserved**: `playerID`, `teamID`, `yearID`, `lgID`, `franchID`, `divID`, `awardID`, `gameID`. These are the universally-recognized Lahman keys; cross-referencing external Lahman materials still works. External IDs (`bbrefID`, `retroID`, `teamIDBR`, etc.) likewise stay as-is — they're literal identifiers in other systems.
- **Snake_case everywhere else**: `home_runs`, `outs_pitched`, `games_catcher`.
- **Self-documenting names**: `RBI` → `runs_batted_in`, `BB` → `walks`, `IPouts` → `outs_pitched`, `BAOpp` → `opponent_batting_avg`.
- **Units made explicit where ambiguous**: `weight` → `weight_lbs`, `height` → `height_inches`.
- **`_allowed` suffix on Pitching's negative-valence stats**: `hits_allowed`, `home_runs_allowed`, `walks_allowed`, `runs_allowed`, etc. Without this an agent could silently sum `b.home_runs + p.home_runs` and produce nonsense — batter HRs and HRs-allowed are not the same quantity.
- **Type conversions deferred to the transform cell**: `Y`/`N` flags become BOOL, `debut`/`finalGame` become DATE. Done at transform time, not here, because the rename map is just text.
- **Synthesized columns**: a `name_full` column is added to People in the transform step (concat of `name_first` + `name_last`) for natural-language queries like "Smith pitchers with 100+ wins."
- **Dropped columns**: any column not in this map is dropped. Notably, People's leading `ID` (an auto-increment artifact in SABR's build) is dropped — `playerID` is the natural key.

In [ ]:
import numpy as np
from datetime import date

PARQUET_DIR = Path("/content/lahman-parquet")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

# Columns that hold 'Y'/'N' flags and should become BOOL.
# Keyed by table; values are the *renamed* column names.
BOOL_COLUMNS = {
    "Teams": ["won_division", "won_wild_card", "won_league", "won_world_series"],
    "AwardsPlayers": ["tie"],
}

# Columns that hold ISO-ish date strings and should become DATE.
DATE_COLUMNS = {
    "People": ["debut_date", "final_game_date"],
}


def yn_to_bool(series: pd.Series) -> pd.Series:
    """'Y'/'N' → True/False, preserving NaN as <NA>."""
    mapping = {"Y": True, "N": False}
    return series.map(mapping).astype("boolean")  # nullable BooleanDtype


def transform_table(table: str) -> pd.DataFrame:
    df = pd.read_csv(CORE_DIR / f"{table}.csv", low_memory=False)
    raw_rows = len(df)

    # Rename + drop unmapped columns + order to match the map
    rename = RENAME_MAPS[table]
    df = df.rename(columns=rename)[list(rename.values())]

    # BOOL casts
    for col in BOOL_COLUMNS.get(table, []):
        df[col] = yn_to_bool(df[col])

    # DATE casts (errors='coerce' so any unparseable value becomes NaT,
    # which we'll catch in the validation block below)
    for col in DATE_COLUMNS.get(table, []):
        df[col] = pd.to_datetime(df[col], errors="coerce").dt.date

    # People only: synthesize name_full
    if table == "People":
        df["name_full"] = (
            df["name_first"].fillna("") + " " + df["name_last"].fillna("")
        ).str.strip()

    assert len(df) == raw_rows, (
        f"{table}: row count changed during transform "
        f"({raw_rows} → {len(df)})"
    )
    return df


# Run the transform for every table, write Parquet, collect a summary
summary = []
for t in LAHMAN_TABLES:
    df = transform_table(t)
    out = PARQUET_DIR / f"{t}.parquet"
    df.to_parquet(out, engine="pyarrow", index=False)
    summary.append({
        "table": t,
        "rows": len(df),
        "cols": len(df.columns),
        "parquet_mb": round(out.stat().st_size / 1024 / 1024, 2),
    })

# Print the summary
print(f"{'Table':<16} {'Rows':>10} {'Cols':>6} {'Parquet MB':>12}")
print("-" * 48)
for r in summary:
    print(f"{r['table']:<16} {r['rows']:>10,} {r['cols']:>6} {r['parquet_mb']:>12}")
print(f"\nTotal Parquet size: {sum(r['parquet_mb'] for r in summary):.2f} MB")

Table                  Rows   Cols   Parquet MB
------------------------------------------------
People               24,270     25         1.57
Batting             128,598     22         1.91
Pitching             57,630     30         1.41
Teams                 3,614     48         0.25
Appearances         128,512     21         1.54
AllstarFull           6,425      8         0.05
AwardsPlayers        12,667      6         0.08
BattingPost          18,687     22          0.2
PitchingPost          7,474     30         0.14

Total Parquet size: 7.15 MB


### Validate the type casts

The transform did three non-trivial things: `Y`/`N` → BOOL, date strings → DATE, and a synthesized `name_full`. We read the Parquet files back from disk (not the in-memory dfs) so we're validating the round-trip through PyArrow, not just pandas state. Things to verify:

- BOOL columns have `boolean` dtype with sensible True/False/NULL distributions. `won_world_series` should have ~one True per year (gaps for 1904 and 1994 strikes); `won_wild_card` should be NULL for everything before 1994.
- DATE columns have `object` dtype (Python `date` objects) with min/max in plausible ranges (debuts span 1871 to current season; some active players have NULL `final_game_date`).
- `name_full` has no leading/trailing whitespace artifacts and no empty strings.

In [ ]:
print("BOOL columns (after Parquet round-trip)")
print("-" * 70)
for table, cols in BOOL_COLUMNS.items():
    df = pd.read_parquet(PARQUET_DIR / f"{table}.parquet")
    for col in cols:
        counts = df[col].value_counts(dropna=False).to_dict()
        print(f"  {table}.{col}: dtype={df[col].dtype}, {counts}")

print()
print("DATE columns (after Parquet round-trip)")
print("-" * 70)
for table, cols in DATE_COLUMNS.items():
    df = pd.read_parquet(PARQUET_DIR / f"{table}.parquet")
    for col in cols:
        s = df[col]
        non_null = s.dropna()
        print(f"  {table}.{col}: dtype={s.dtype}, "
              f"non-null={len(non_null):,}, null={s.isna().sum():,}, "
              f"range=[{non_null.min()} \u2192 {non_null.max()}]")

print()
print("Synthesized name_full (People)")
print("-" * 70)
people = pd.read_parquet(PARQUET_DIR / "People.parquet")
sample = people[["playerID", "name_first", "name_last", "name_full"]].sample(5, random_state=42)
print(sample.to_string(index=False))

weird_ws = (people["name_full"] != people["name_full"].str.strip()).sum()
empty = (people["name_full"] == "").sum()
print(f"\n  Names with leading/trailing whitespace: {weird_ws}")
print(f"  Empty name_full: {empty}")

BOOL columns (after Parquet round-trip)
----------------------------------------------------------------------
  Teams.won_division: dtype=boolean, {<NA>: 2054, np.False_: 1270, np.True_: 290}
  Teams.won_wild_card: dtype=boolean, {<NA>: 2690, np.False_: 820, np.True_: 104}
  Teams.won_league: dtype=boolean, {np.False_: 2785, <NA>: 488, np.True_: 341}
  Teams.won_world_series: dtype=boolean, {np.False_: 2622, <NA>: 855, np.True_: 137}
  AwardsPlayers.tie: dtype=boolean, {<NA>: 12008, np.True_: 659}

DATE columns (after Parquet round-trip)
----------------------------------------------------------------------
  People.debut_date: dtype=object, non-null=21,240, null=3,030, range=[1871-05-04 → 2025-10-01]
  People.final_game_date: dtype=object, non-null=19,338, null=4,932, range=[1871-05-05 → 2024-09-30]

Synthesized name_full (People)
----------------------------------------------------------------------
 playerID name_first name_last    name_full
rondojo01      Jorge    Rondon Jorge Ron

## 7. Upload Parquet to GCS

Push the 9 validated Parquet files to `gs://class-demo/mlb-race-to-october/lahman/`. Students will load these into BigQuery in Task 1 with `LOAD DATA OVERWRITE` — that SQL is captured in section 8 (next cell after upload).

In [ ]:
LAHMAN_GCS_DEST_PREFIX = f"{GCS_PREFIX}/lahman"

print(f"Uploading to gs://{GCS_BUCKET}/{LAHMAN_GCS_DEST_PREFIX}/")
total_bytes = 0
for table in LAHMAN_TABLES:
    src = PARQUET_DIR / f"{table}.parquet"
    blob_path = f"{LAHMAN_GCS_DEST_PREFIX}/{table}.parquet"
    blob = bucket.blob(blob_path)
    blob.upload_from_filename(str(src))
    size_mb = src.stat().st_size / 1024 / 1024
    total_bytes += src.stat().st_size
    print(f"  \u2713 {table}.parquet ({size_mb:.2f} MB)")

print(f"\nTotal uploaded: {total_bytes / 1024 / 1024:.2f} MB across {len(LAHMAN_TABLES)} files")

Uploading to gs://class-demo/mlb-race-to-october/lahman/
  ✓ People.parquet (1.57 MB)
  ✓ Batting.parquet (1.91 MB)
  ✓ Pitching.parquet (1.41 MB)
  ✓ Teams.parquet (0.25 MB)
  ✓ Appearances.parquet (1.54 MB)
  ✓ AllstarFull.parquet (0.05 MB)
  ✓ AwardsPlayers.parquet (0.08 MB)
  ✓ BattingPost.parquet (0.20 MB)
  ✓ PitchingPost.parquet (0.14 MB)

Total uploaded: 7.14 MB across 9 files


## 8. Generate Task 1 SQL — LOAD DATA + column descriptions

Task 1 of the lab loads these Parquet files into BigQuery and adds column descriptions. This cell generates the exact SQL the lab content embeds, so the lab author isn't writing it by hand.

The output is two parts per table:

1. **`LOAD DATA OVERWRITE ... FROM FILES(...)`** — creates the BigQuery table from the staged Parquet, schema autodetected.
2. **`ALTER TABLE ... ALTER COLUMN ... SET OPTIONS(description='...')`** — adds column descriptions the ADK agent's BigQuery tool will read when generating SQL.

Two pedagogical points the lab content hammers on:

- Step 1 alone produces a working table. The lab could stop there. But the agent's NL→SQL works dramatically better when columns have descriptions — that's the whole point of step 2. We're not adding descriptions because BigQuery requires them; we're adding them because the agent's prompt grounding reads them.
- Multi-column `ALTER TABLE` syntax — one statement, many `ALTER COLUMN` clauses — is the cleanest way to apply descriptions in bulk.

This turn seeds **People** and **Teams** as exemplars to validate description style. Once those look right, we'll fill in the other seven tables.

In [ ]:
# BigQuery target — student labs create this dataset in their Qwiklabs project.
# The generated SQL hardcodes this name so it's copy-pasteable.
DATASET_NAME = "mlb_race_to_october"

# Map Lahman CSV/Parquet names to BigQuery table names (snake_case for consistency
# with our column convention; mixed case in tables + snake in cols would be jarring).
BQ_TABLE_NAMES = {
    "People": "people",
    "Batting": "batting",
    "Pitching": "pitching",
    "Teams": "teams",
    "Appearances": "appearances",
    "AllstarFull": "allstar_full",
    "AwardsPlayers": "awards_players",
    "BattingPost": "batting_post",
    "PitchingPost": "pitching_post",
}

# Column descriptions, keyed by Lahman table name then renamed column name.
# Goal: every description gives the agent enough to (a) pick this column over
# similar ones, (b) understand sentinel values, (c) handle nullability correctly.
# Apostrophes are fine here — SQL escaping happens in the generation step below.

COLUMN_DESCRIPTIONS = {
    "People": {
        "playerID": "Lahman's universal player identifier (e.g., 'aaronha01' for Hank Aaron, 'ruthba01' for Babe Ruth). Preserved in original camelCase as the canonical Lahman key. The primary join column linking People to all stat tables.",
        "birth_year": "Year of birth (4-digit). NULL if unknown.",
        "birth_month": "Month of birth (1-12). NULL if unknown.",
        "birth_day": "Day of birth (1-31). NULL if unknown.",
        "birth_city": "City of birth. NULL if unknown.",
        "birth_state": "State, province, or other country subdivision of birth. NULL if unknown.",
        "birth_country": "Country of birth. NULL if unknown.",
        "death_year": "Year of death (4-digit). NULL if the player is still living or death year is unknown.",
        "death_month": "Month of death (1-12). NULL if living or unknown.",
        "death_day": "Day of death (1-31). NULL if living or unknown.",
        "death_city": "City of death. NULL if living or unknown.",
        "death_state": "State or country subdivision of death. NULL if living or unknown.",
        "death_country": "Country of death. NULL if living or unknown.",
        "name_first": "First name as commonly known.",
        "name_last": "Last name.",
        "name_given": "Full given name(s) at birth, including middle names.",
        "weight_lbs": "Listed playing weight in pounds. NULL if unknown.",
        "height_inches": "Listed playing height in inches. NULL if unknown.",
        "bats": "Batting handedness: 'L' (left), 'R' (right), 'B' (both / switch hitter). NULL if unknown.",
        "throws": "Throwing handedness: 'L' (left), 'R' (right). NULL if unknown.",
        "debut_date": "Date of first MLB appearance. NULL for non-players included in this table (managers, executives) who never played.",
        "final_game_date": "Date of last MLB appearance. NULL for currently active players AND for very recent debuts — Lahman stamps this only after a player is officially retired (didn't play the following season), so 2025 debuters who are still active will have NULL here.",
        "bbrefID": "Baseball Reference player identifier (different format from playerID). Used for cross-referencing to baseball-reference.com.",
        "retroID": "Retrosheet player identifier. Used for cross-referencing to Retrosheet play-by-play data.",
        "name_full": "Concatenation of name_first and name_last for natural-language matching (e.g., 'Babe Ruth', 'Hank Aaron'). Synthesized at staging time, not present in source Lahman.",
    },

    "Teams": {
        "yearID": "Season year (4-digit). Preserved in original camelCase as the canonical Lahman key.",
        "lgID": "League ID: 'AL' (American League), 'NL' (National League), or historical leagues — 'AA' (American Association), 'UA' (Union Association), 'PL' (Players League), 'FL' (Federal League), 'NA' (National Association). Preserved in original camelCase as the canonical Lahman key.",
        "teamID": "Team identifier for this specific franchise-season. Note: teamID can change when a franchise relocates (e.g., the Brooklyn Dodgers and Los Angeles Dodgers have different teamIDs across years). Use franchID to track a franchise across moves.",
        "franchID": "Franchise identifier, stable across team relocations. The Dodgers franchise has franchID 'LAD' regardless of city.",
        "divID": "Division: 'E' (East), 'W' (West), 'C' (Central). NULL before 1969 (pre-divisional play).",
        "division_rank": "Final standing within division (1 = first place). For pre-1969 seasons, this is rank within the league.",
        "games": "Total games played in the regular season.",
        "home_games": "Games played at home (Ghome in Lahman). Useful when computing home/road splits.",
        "wins": "Regular season wins.",
        "losses": "Regular season losses.",
        "won_division": "True if this team won its division. NULL before 1969 (pre-divisional play).",
        "won_wild_card": "True if this team won a wild card berth. NULL before 1994 (Wild Card era began in 1994). Number of wild cards per league increased from 1 (1994-2011) to 2 (2012-2021) to 3 (2022 onwards).",
        "won_league": "True if this team won the pennant (league championship; for the World Series era this means making the World Series).",
        "won_world_series": "True if this team won the World Series. The World Series began in 1903; NULL for 1904 and 1994 (no WS played) and for most pre-1903 seasons. Some 19th-century postseason events (1880s AA-NL championship series, Temple Cup) ARE coded as won_world_series=True, so a small number of pre-1903 entries will be True.",
        "runs_scored": "Total runs scored by this team's offense in the regular season.",
        "at_bats": "Team total at-bats.",
        "hits": "Team total hits (offensive). For hits allowed by team pitching see hits_allowed.",
        "doubles": "Team total doubles hit (offensive).",
        "triples": "Team total triples hit (offensive).",
        "home_runs": "Team total home runs hit (offensive). For HRs allowed by pitching see home_runs_allowed.",
        "walks": "Team total walks drawn by batters (offensive). For walks issued by pitchers see walks_allowed.",
        "strikeouts": "Team total strikeouts BY batters (negative for offense). For strikeouts recorded BY pitchers see strikeouts_pitched.",
        "stolen_bases": "Team total stolen bases.",
        "caught_stealing": "Team total times caught stealing.",
        "hit_by_pitch": "Team total times batters were hit by a pitch (offensive). For batters hit BY this team's pitchers, that data is not in Teams.",
        "sacrifice_flies": "Team total sacrifice flies (offensive).",
        "runs_allowed": "Total runs allowed by this team's pitching staff. Includes both earned and unearned runs.",
        "earned_runs_allowed": "Total earned runs allowed by team pitching (excludes runs scored due to fielding errors).",
        "earned_run_avg": "Team ERA: earned runs allowed per 9 innings pitched.",
        "complete_games": "Team complete games (a single pitcher faced every batter from start to finish without relief).",
        "shutouts": "Team shutouts (pitching staff allowed zero runs in a 9+ inning game).",
        "saves": "Team saves recorded by relief pitchers.",
        "outs_pitched": "Total outs recorded while pitching. Innings pitched = outs_pitched / 3.",
        "hits_allowed": "Total hits allowed by team pitching.",
        "home_runs_allowed": "Total home runs allowed by team pitching.",
        "walks_allowed": "Total walks (bases on balls) issued by team pitching.",
        "strikeouts_pitched": "Total strikeouts recorded BY team pitchers (positive for defense). Distinct from 'strikeouts' which is batters' strikeouts.",
        "errors": "Total fielding errors committed.",
        "double_plays": "Total double plays turned defensively.",
        "fielding_pct": "Team fielding percentage: (putouts + assists) / (putouts + assists + errors).",
        "team_name": "Full team name as known in this season (e.g., 'New York Yankees', 'Brooklyn Dodgers').",
        "park_name": "Home park name for this season.",
        "home_attendance": "Total home attendance for the season.",
        "batting_park_factor": "Park factor for batting effects (BPF in Lahman). 100 is league-neutral; values >100 indicate the park favors offense, <100 favors pitching. Computed by Lahman from home/road run-scoring ratios.",
        "pitching_park_factor": "Park factor for pitching effects (PPF in Lahman). 100 is league-neutral. Computed similarly to batting_park_factor.",
        "teamIDBR": "Baseball Reference team ID for this season.",
        "teamIDlahman45": "Legacy Lahman v4.5 team ID. Used for cross-referencing older Lahman datasets.",
        "teamIDretro": "Retrosheet team ID for this season.",
    },

    "Batting": {
    "playerID": "Lahman's universal player identifier. Joins to people.playerID. Preserved in original camelCase as the canonical Lahman key.",
    "yearID": "Season year (4-digit). Preserved in original camelCase as the canonical Lahman key.",
    "stint": "Within-season stint number (1, 2, ...). A player traded mid-season has multiple rows for the same yearID, one per team. To get a player's full season totals, SUM across stints with GROUP BY playerID, yearID.",
    "teamID": "Team for this stint. Joins to teams.teamID. Note teamID changes when a franchise relocates; use franchID via teams to track a franchise across moves.",
    "lgID": "League ID for this stint: 'AL', 'NL', or historical leagues. Preserved in original camelCase.",
    "games": "Games played by this batter in this stint.",
    "at_bats": "Plate appearances excluding walks, hit-by-pitch, sacrifices, and interference. Used as the denominator for batting average.",
    "runs": "Runs scored by this batter.",
    "hits": "Hits accumulated. Singles + doubles + triples + home runs.",
    "doubles": "Doubles hit. Lahman calls this 2B.",
    "triples": "Triples hit. Lahman calls this 3B.",
    "home_runs": "Home runs hit by this batter. For batters this is offensive output; do NOT confuse with pitching.home_runs_allowed.",
    "runs_batted_in": "Runs batted in. Sometimes abbreviated RBI in baseball commentary.",
    "stolen_bases": "Bases stolen successfully.",
    "caught_stealing": "Times caught stealing.",
    "walks": "Bases on balls drawn by this batter (sometimes abbreviated BB). Includes intentional walks.",
    "strikeouts": "Times this batter struck out. Negative for offense.",
    "intentional_walks": "Intentional walks drawn (subset of walks). Tracked in Lahman from 1955 onwards; NULL for earlier seasons.",
    "hit_by_pitch": "Times hit by a pitch (offensive).",
    "sacrifice_hits": "Sacrifice bunts (sometimes called 'sacrifices'). Distinct from sacrifice_flies.",
    "sacrifice_flies": "Sacrifice flies. Tracked from 1954 onwards; NULL for earlier seasons.",
    "grounded_into_double_play": "Times grounded into a double play. Tracked from 1933 (NL) and 1939 (AL) onwards; NULL for earlier seasons.",
},

"Pitching": {
    "playerID": "Lahman's universal player identifier. Joins to people.playerID.",
    "yearID": "Season year (4-digit).",
    "stint": "Within-season stint number for pitchers traded mid-season. SUM across stints for full-season totals.",
    "teamID": "Team for this stint.",
    "lgID": "League ID for this stint.",
    "wins": "Pitcher's wins (W).",
    "losses": "Pitcher's losses (L).",
    "games": "Games pitched in (any role: starter, reliever, closer).",
    "games_started": "Games started as the starting pitcher. games - games_started gives relief appearances.",
    "complete_games": "Complete games (pitcher faced every opposing batter from start to finish).",
    "shutouts": "Complete-game shutouts (no runs allowed in a 9+ inning complete game).",
    "saves": "Saves recorded under MLB save rules. Tracked from 1969 onwards; earlier values are retroactive estimates.",
    "outs_pitched": "Total outs recorded while pitching. Innings pitched = outs_pitched / 3.0. To compute IP exactly: FLOOR(outs_pitched/3) + (outs_pitched MOD 3)/10.0 in standard baseball notation.",
    "hits_allowed": "Hits allowed by this pitcher. Distinct from batting.hits.",
    "earned_runs_allowed": "Earned runs allowed (excludes runs scored due to fielding errors).",
    "home_runs_allowed": "Home runs allowed. Distinct from batting.home_runs.",
    "walks_allowed": "Walks issued (bases on balls). Distinct from batting.walks.",
    "strikeouts": "Strikeouts recorded BY this pitcher (positive for pitcher). Distinct from batting.strikeouts.",
    "opponent_batting_avg": "Opposing batters' batting average against this pitcher (BAOpp in Lahman). Lower is better.",
    "earned_run_avg": "Earned Run Average: (earned_runs_allowed * 9) / innings_pitched. Lower is better.",
    "intentional_walks_allowed": "Intentional walks issued. Tracked from 1955 onwards.",
    "wild_pitches": "Wild pitches thrown.",
    "batters_hit_by_pitch": "Batters this pitcher hit with a pitch.",
    "balks": "Balks called against this pitcher.",
    "batters_faced": "Total batters faced (BFP in Lahman). The pitching equivalent of plate appearances.",
    "games_finished": "Games finished as the last pitcher (relief appearances completing a game). Used in older save calculations.",
    "runs_allowed": "Total runs allowed (earned + unearned).",
    "sacrifice_hits_allowed": "Sacrifice bunts allowed by this pitcher.",
    "sacrifice_flies_allowed": "Sacrifice flies allowed.",
    "double_plays_induced": "Double plays induced by this pitcher (GIDP). Distinct from batting.grounded_into_double_play which tracks the batter's perspective.",
},

"Appearances": {
    "yearID": "Season year (4-digit).",
    "teamID": "Team for this row.",
    "lgID": "League ID.",
    "playerID": "Lahman's universal player identifier.",
    "games_total": "Total games appeared in any capacity for this team this season (G_all in Lahman).",
    "games_started": "Games started in any defensive position.",
    "games_batting": "Games in which the player batted.",
    "games_defense": "Games played defensively (any position including DH).",
    "games_pitcher": "Games appeared as a pitcher.",
    "games_catcher": "Games appeared as a catcher.",
    "games_first_base": "Games appeared at first base.",
    "games_second_base": "Games appeared at second base.",
    "games_third_base": "Games appeared at third base.",
    "games_shortstop": "Games appeared at shortstop.",
    "games_left_field": "Games appeared in left field.",
    "games_center_field": "Games appeared in center field.",
    "games_right_field": "Games appeared in right field.",
    "games_outfield": "Total outfield games (G_OF). Equals games_left_field + games_center_field + games_right_field. Useful for queries that don't care about the specific outfield position.",
    "games_designated_hitter": "Games as DH. The DH was introduced in the AL in 1973 and adopted by the NL in 2022.",
    "games_pinch_hitter": "Games as a pinch hitter.",
    "games_pinch_runner": "Games as a pinch runner.",
},

"AllstarFull": {
    "playerID": "Lahman's universal player identifier.",
    "yearID": "Season year of the All-Star Game.",
    "game_num": "Game number within the season: 1 or 2. Most years have a single ASG (game_num=1). From 1959-1962 MLB held two All-Star Games per season; those rows have game_num=2 for the second game.",
    "gameID": "Retrosheet-style identifier for the specific All-Star Game. Preserved in camelCase.",
    "teamID": "The player's team during the season they were selected (not the AL/NL ASG roster).",
    "lgID": "League the player represented in the ASG ('AL' or 'NL').",
    "games_played": "1 if the player actually appeared in the ASG, 0 or NULL if they were selected but didn't play (injury, manager's discretion, etc.). Counts of All-Star selections should NOT filter on this column; counts of All-Star appearances should.",
    "starting_position": "Position number (1-9) if the player started the game, NULL otherwise. 1=P, 2=C, 3=1B, 4=2B, 5=3B, 6=SS, 7=LF, 8=CF, 9=RF, 10=DH (DH used in some interleague-rule years).",
},

"AwardsPlayers": {
    "playerID": "Lahman's universal player identifier.",
    "awardID": "Award name as a string. Common values: 'Most Valuable Player', 'Cy Young Award', 'Rookie of the Year', 'Gold Glove', 'Silver Slugger', 'Manager of the Year', 'World Series MVP', 'ALCS MVP', 'NLCS MVP', 'Reliever of the Year'. Use DISTINCT awardID to enumerate possibilities. Preserved in camelCase as the canonical Lahman key.",
    "yearID": "Season the award was given for.",
    "lgID": "League the award was given in: 'AL', 'NL', or 'ML' for awards given league-agnostically (e.g., World Series MVP).",
    "tie": "True if the award was shared (ties between multiple winners), NULL otherwise. NOT False — Lahman uses NULL for the non-tie case rather than 'N'.",
    "notes": "Free-text notes about the award (e.g., position for Gold Gloves like 'CF', 'P'; specific designation like 'pitcher' or 'outfield' for Silver Slugger). NULL for most awards.",
},

"BattingPost": {
    "yearID": "Season year. Postseason follows the regular season of the same yearID.",
    "round": "Postseason round identifier. Common values: 'WS' (World Series), 'NLCS' / 'ALCS' (League Championship Series), 'NLDS' / 'ALDS' (League Division Series), 'NLWC' / 'ALWC' (Wild Card round, 2012 onwards), 'WC' (single Wild Card game, 2012-2019), and historical rounds like 'CS' (19th-century championship series).",
    "playerID": "Lahman's universal player identifier.",
    "teamID": "Team during this postseason round.",
    "lgID": "League ID.",
    "games": "Games played in this postseason round.",
    "at_bats": "Postseason at-bats in this round.",
    "runs": "Postseason runs scored in this round.",
    "hits": "Postseason hits in this round.",
    "doubles": "Doubles hit in this round.",
    "triples": "Triples hit in this round.",
    "home_runs": "Home runs hit in this round (offensive).",
    "runs_batted_in": "RBI in this round.",
    "stolen_bases": "Stolen bases in this round.",
    "caught_stealing": "Caught stealing in this round.",
    "walks": "Walks drawn in this round.",
    "strikeouts": "Strikeouts in this round.",
    "intentional_walks": "Intentional walks drawn.",
    "hit_by_pitch": "Times hit by pitch in this round.",
    "sacrifice_hits": "Sacrifice bunts in this round.",
    "sacrifice_flies": "Sacrifice flies in this round.",
    "grounded_into_double_play": "Grounded into double plays in this round.",
},

"PitchingPost": {
    "playerID": "Lahman's universal player identifier.",
    "yearID": "Season year.",
    "round": "Postseason round identifier (see batting_post.round for full list of values).",
    "teamID": "Team during this postseason round.",
    "lgID": "League ID.",
    "wins": "Pitcher's wins in this round.",
    "losses": "Pitcher's losses in this round.",
    "games": "Games pitched in this round.",
    "games_started": "Games started in this round.",
    "complete_games": "Complete games in this round.",
    "shutouts": "Shutouts in this round.",
    "saves": "Saves in this round.",
    "outs_pitched": "Total outs recorded pitching in this round. IP = outs_pitched / 3.",
    "hits_allowed": "Hits allowed in this round.",
    "earned_runs_allowed": "Earned runs allowed in this round.",
    "home_runs_allowed": "Home runs allowed in this round.",
    "walks_allowed": "Walks issued in this round.",
    "strikeouts": "Strikeouts BY this pitcher in this round.",
    "opponent_batting_avg": "Opposing batting average against this pitcher in this round.",
    "earned_run_avg": "ERA in this round.",
    "intentional_walks_allowed": "Intentional walks issued in this round.",
    "wild_pitches": "Wild pitches in this round.",
    "batters_hit_by_pitch": "Batters hit by this pitcher in this round.",
    "balks": "Balks in this round.",
    "batters_faced": "Batters faced in this round.",
    "games_finished": "Games finished in this round (relief appearances closing out games).",
    "runs_allowed": "Total runs allowed in this round.",
    "sacrifice_hits_allowed": "Sacrifice bunts allowed in this round.",
    "sacrifice_flies_allowed": "Sacrifice flies allowed in this round.",
    "double_plays_induced": "Double plays induced in this round.",
},
}

# Sanity check: every described column exists in the rename map's outputs
for table, descs in COLUMN_DESCRIPTIONS.items():
    if not descs:
        continue
    expected_cols = set(RENAME_MAPS[table].values())
    if table == "People":
        expected_cols.add("name_full")  # synthesized
    described = set(descs.keys())
    missing = expected_cols - described
    extra = described - expected_cols
    if missing or extra:
        print(f"\u26a0\ufe0f  {table}: missing={missing}, extra={extra}")
    else:
        print(f"\u2713 {table}: all {len(described)} columns described")

✓ People: all 25 columns described
✓ Teams: all 48 columns described
✓ Batting: all 22 columns described
✓ Pitching: all 30 columns described
✓ Appearances: all 21 columns described
✓ AllstarFull: all 8 columns described
✓ AwardsPlayers: all 6 columns described
✓ BattingPost: all 22 columns described
✓ PitchingPost: all 30 columns described


In [ ]:
SQL_OUT = Path("/content/load_lahman.sql")
SCHEMA_GCS_PREFIX = f"{GCS_PREFIX}/schema"

def sql_escape(s: str) -> str:
    """Escape backslashes and single quotes for BigQuery single-quoted strings.
    Order matters: backslashes first, then quotes."""
    return s.replace("\\", "\\\\").replace("'", "\\'")

def gen_load_sql(lahman_table: str, bq_table: str) -> str:
    parquet_uri = f"gs://{GCS_BUCKET}/{LAHMAN_GCS_DEST_PREFIX}/{lahman_table}.parquet"
    return (
        f"LOAD DATA OVERWRITE `{DATASET_NAME}.{bq_table}`\n"
        f"FROM FILES (\n"
        f"  format = 'PARQUET',\n"
        f"  uris = ['{parquet_uri}']\n"
        f");"
    )

def gen_alter_sql(lahman_table: str, bq_table: str) -> str:
    descs = COLUMN_DESCRIPTIONS[lahman_table]
    if not descs:
        return f"-- TODO: descriptions for {bq_table} not yet defined"
    clauses = [
        f"  ALTER COLUMN {col} SET OPTIONS(description = '{sql_escape(desc)}')"
        for col, desc in descs.items()
    ]
    return (
        f"ALTER TABLE `{DATASET_NAME}.{bq_table}`\n"
        + ",\n".join(clauses)
        + ";"
    )

# Build the full SQL file
parts = [
    f"-- MLB Race to October — Task 1: Load Lahman data into BigQuery",
    f"-- Generated by mlb_data_foundation.ipynb",
    f"-- Target dataset: {DATASET_NAME} (create this in your project before running)",
    "",
    "-- =====================================================================",
    "-- Part 1: LOAD DATA — create tables from staged Parquet",
    "-- =====================================================================",
    "",
]
for lahman, bq in BQ_TABLE_NAMES.items():
    parts.append(f"-- {lahman} \u2192 {bq}")
    parts.append(gen_load_sql(lahman, bq))
    parts.append("")

parts.extend([
    "-- =====================================================================",
    "-- Part 2: ALTER TABLE — add column descriptions for the agent",
    "-- =====================================================================",
    "",
])
for lahman, bq in BQ_TABLE_NAMES.items():
    parts.append(f"-- {bq} column descriptions")
    parts.append(gen_alter_sql(lahman, bq))
    parts.append("")

sql_text = "\n".join(parts)
SQL_OUT.write_text(sql_text)
print(f"Wrote {SQL_OUT} ({len(sql_text):,} chars, {sql_text.count(chr(10)):,} lines)")

# Upload to GCS
blob_path = f"{SCHEMA_GCS_PREFIX}/load_lahman.sql"
bucket.blob(blob_path).upload_from_filename(str(SQL_OUT))
print(f"Uploaded to gs://{GCS_BUCKET}/{blob_path}")

# Structured list for the validator — avoids re-parsing the SQL file.
GENERATED_STATEMENTS = []
for lahman, bq in BQ_TABLE_NAMES.items():
    GENERATED_STATEMENTS.append((f"LOAD {bq}", gen_load_sql(lahman, bq)))
for lahman, bq in BQ_TABLE_NAMES.items():
    if COLUMN_DESCRIPTIONS[lahman]:
        GENERATED_STATEMENTS.append((f"ALTER {bq}", gen_alter_sql(lahman, bq)))

print(f"Tracked {len(GENERATED_STATEMENTS)} executable statements for validation.")

# Show the People LOAD + first few ALTER lines so we can spot-check format
print("\n" + "=" * 70)
print("Preview: people LOAD DATA + first lines of ALTER TABLE")
print("=" * 70)
print(gen_load_sql("People", "people"))
print()
preview = gen_alter_sql("People", "people").splitlines()
print("\n".join(preview[:6] + ["  -- (... rest elided ...)", preview[-1]]))

Wrote /content/load_lahman.sql (29,552 chars, 313 lines)
Uploaded to gs://class-demo/mlb-race-to-october/schema/load_lahman.sql
Tracked 18 executable statements for validation.

Preview: people LOAD DATA + first lines of ALTER TABLE
LOAD DATA OVERWRITE `mlb_race_to_october.people`
FROM FILES (
  format = 'PARQUET',
  uris = ['gs://class-demo/mlb-race-to-october/lahman/People.parquet']
);

ALTER TABLE `mlb_race_to_october.people`
  ALTER COLUMN playerID SET OPTIONS(description = 'Lahman\'s universal player identifier (e.g., \'aaronha01\' for Hank Aaron, \'ruthba01\' for Babe Ruth). Preserved in original camelCase as the canonical Lahman key. The primary join column linking People to all stat tables.'),
  ALTER COLUMN birth_year SET OPTIONS(description = 'Year of birth (4-digit). NULL if unknown.'),
  ALTER COLUMN birth_month SET OPTIONS(description = 'Month of birth (1-12). NULL if unknown.'),
  ALTER COLUMN birth_day SET OPTIONS(description = 'Day of birth (1-31). NULL if unknown.'),
 

### Validate the generated SQL end-to-end

Execute the LOAD + ALTER SQL we just generated, against a throwaway dataset, to confirm:

1. The Parquet schema autodetect produces the BigQuery types we expect (especially the nullable BOOLs and DATEs).
2. The `ALTER TABLE ... ALTER COLUMN ... SET OPTIONS(description=...)` syntax actually applies — we query `INFORMATION_SCHEMA.COLUMNS` to verify descriptions landed.
3. The full SQL runs without error in the order the lab will execute it.

Validation runs in dataset `mlb_validation_<timestamp>` to avoid colliding with anything in the project, and the dataset is dropped at the end of the cell whether validation succeeded or failed. Any failure raises before the drop so we still see what broke; the drop is in a `finally` block.

In [ ]:
import time
from google.cloud import bigquery

VALIDATION_DATASET = f"mlb_validation_{int(time.time())}"
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

# Tables we generated SQL for so far (full list in BQ_TABLE_NAMES; descriptions
# only filled in for People and Teams, but LOAD will run for all 9)
TABLES_WITH_DESCRIPTIONS = [t for t, d in COLUMN_DESCRIPTIONS.items() if d]

print(f"Validation dataset: {PROJECT_ID}.{VALIDATION_DATASET}")
print(f"Location:           {BQ_LOCATION}\n")

# Create the dataset
ds_ref = bigquery.Dataset(f"{PROJECT_ID}.{VALIDATION_DATASET}")
ds_ref.location = BQ_LOCATION
bq.create_dataset(ds_ref)
print(f"\u2713 Created dataset\n")

try:
    print(f"Executing {len(GENERATED_STATEMENTS)} statements against {VALIDATION_DATASET}...\n")

    load_count = 0
    alter_count = 0
    for label, stmt in GENERATED_STATEMENTS:
        # Swap the production dataset name for our throwaway one
        sql = stmt.replace(f"`{DATASET_NAME}.", f"`{VALIDATION_DATASET}.")
        try:
            bq.query(sql).result()
        except Exception as e:
            print(f"  \u2717 {label} FAILED: {e}")
            raise
        if label.startswith("LOAD"):
            load_count += 1
        elif label.startswith("ALTER"):
            alter_count += 1

    print(f"\u2713 {load_count} LOAD statements succeeded")
    print(f"\u2713 {alter_count} ALTER statements succeeded\n")

    # Verify rows landed correctly — compare against our local Parquet row counts
    print("Row count check (BigQuery vs local Parquet)")
    print("-" * 60)
    for lahman_t, bq_t in BQ_TABLE_NAMES.items():
        expected = pd.read_parquet(PARQUET_DIR / f"{lahman_t}.parquet").shape[0]
        result = bq.query(
            f"SELECT COUNT(*) AS n FROM `{VALIDATION_DATASET}.{bq_t}`"
        ).result()
        actual = list(result)[0]["n"]
        status = "\u2713" if actual == expected else "\u2717"
        print(f"  {status} {bq_t:<18} BQ={actual:,}  Parquet={expected:,}")

    # Verify descriptions stuck — query INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
    print("\nDescription check (INFORMATION_SCHEMA.COLUMN_FIELD_PATHS)")
    print("-" * 60)
    for lahman_t in TABLES_WITH_DESCRIPTIONS:
        bq_t = BQ_TABLE_NAMES[lahman_t]
        expected_described = len(COLUMN_DESCRIPTIONS[lahman_t])
        result = bq.query(f"""
            SELECT COUNT(*) AS n
            FROM `{VALIDATION_DATASET}`.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
            WHERE table_name = '{bq_t}' AND description IS NOT NULL
        """).result()
        actual = list(result)[0]["n"]
        status = "\u2713" if actual == expected_described else "\u2717"
        print(f"  {status} {bq_t:<18} described={actual}  expected={expected_described}")

    # Sample a couple of descriptions back to confirm content (not just presence)
    print("\nSample descriptions readback")
    print("-" * 60)
    sample_q = f"""
        SELECT table_name, field_path, description
        FROM `{VALIDATION_DATASET}`.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
        WHERE table_name IN ('people', 'teams')
          AND field_path IN ('playerID', 'won_world_series', 'name_full')
        ORDER BY table_name, field_path
    """
    for row in bq.query(sample_q).result():
        snippet = row["description"][:90] + ("..." if len(row["description"]) > 90 else "")
        print(f"  {row['table_name']}.{row['field_path']}:")
        print(f"    {snippet}")

finally:
    print(f"\nDropping validation dataset {VALIDATION_DATASET}...")
    bq.delete_dataset(VALIDATION_DATASET, delete_contents=True, not_found_ok=True)
    print("\u2713 Dropped.")

Validation dataset: qwiklabs-gcp-00-dd506b084f93.mlb_validation_1778103423
Location:           US

✓ Created dataset

Executing 18 statements against mlb_validation_1778103423...

✓ 9 LOAD statements succeeded
✓ 9 ALTER statements succeeded

Row count check (BigQuery vs local Parquet)
------------------------------------------------------------
  ✓ people             BQ=24,270  Parquet=24,270
  ✓ batting            BQ=128,598  Parquet=128,598
  ✓ pitching           BQ=57,630  Parquet=57,630
  ✓ teams              BQ=3,614  Parquet=3,614
  ✓ appearances        BQ=128,512  Parquet=128,512
  ✓ allstar_full       BQ=6,425  Parquet=6,425
  ✓ awards_players     BQ=12,667  Parquet=12,667
  ✓ batting_post       BQ=18,687  Parquet=18,687
  ✓ pitching_post      BQ=7,474  Parquet=7,474

Description check (INFORMATION_SCHEMA.COLUMN_FIELD_PATHS)
------------------------------------------------------------
  ✓ people             described=25  expected=25
  ✓ teams              described=48  expected

## 9. BQML playoff probability model — validation

In Task 2 of the lab, students train a logistic regression model in BigQuery ML to predict whether a team will make the playoffs based on their regular-season performance. This section generates the exact `CREATE MODEL` SQL the lab embeds, then validates the model trains cleanly and produces sensible metrics.

**Modeling decisions:**

- **Target**: `played_postseason` — any of `won_division`, `won_wild_card`, `won_league`, or `won_world_series` is True. The last flag catches pre-1969 pennant winners but in our 1994+ training window it's a strict subset of `won_league`, so it's belt-and-suspenders. Computed inline in the training query rather than added as a column to the staged data.
- **Training window**: 1994–2024. The Wild Card era began in 1994; restricting to this window means a single competitive regime (no rule changes large enough to require era controls). 31 seasons × ~30 teams = ~930 training rows.
- **Holdout**: 2025. Single most recent season, not in training. Lets students ask "how did our model do this year?" and sanity-check predictions against their own memory of the season — concrete, current, narratively tight.
- **Features** (all from `teams`, all standardized internally by BQML):
  - `winning_pct` = wins / games
  - `run_differential` = runs_scored − runs_allowed
  - `earned_run_avg` (team ERA)
- **AUC sanity range**: 0.75–0.99 acceptable. Below 0.75 means the model isn't learning enough; above 0.99 means we've leaked the target somehow.

**What this section produces:**

- `gs://class-demo/mlb-race-to-october/schema/train_playoff_model.sql` — the `CREATE MODEL` and `ML.EVALUATE` statements Task 2 embeds, paired so students see training and evaluation as one workflow.
- A validation run that creates a throwaway dataset, loads the `teams` parquet, trains the model, evaluates against 2025, prints metrics, and drops the dataset.

A note on why the model uses inline derived columns rather than a pre-computed view or staged feature table: the lab is teaching BQML, and showing students that you can compute features directly in the training query is part of the lesson. A separate feature pipeline would be appropriate for production but adds machinery the lab doesn't need.

In [ ]:
# BQML model training + evaluation SQL.
# Like the Lahman SQL, this is the verbatim text that will appear in the lab content.

MODEL_NAME = "playoff_probability"

CREATE_MODEL_SQL = f"""\
CREATE OR REPLACE MODEL `{DATASET_NAME}.{MODEL_NAME}`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['played_postseason'],
  auto_class_weights = TRUE
) AS
SELECT
  -- Target: did this team play in the postseason?
  COALESCE(won_division, FALSE)
    OR COALESCE(won_wild_card, FALSE)
    OR COALESCE(won_league, FALSE)
    OR COALESCE(won_world_series, FALSE)
    AS played_postseason,
  -- Features
  wins / games AS winning_pct,
  runs_scored - runs_allowed AS run_differential,
  earned_run_avg
FROM `{DATASET_NAME}.teams`
WHERE yearID BETWEEN 1994 AND 2024;"""

EVALUATE_SQL = f"""\
SELECT *
FROM ML.EVALUATE(
  MODEL `{DATASET_NAME}.{MODEL_NAME}`,
  (
    SELECT
      COALESCE(won_division, FALSE)
        OR COALESCE(won_wild_card, FALSE)
        OR COALESCE(won_league, FALSE)
        OR COALESCE(won_world_series, FALSE)
        AS played_postseason,
      wins / games AS winning_pct,
      runs_scored - runs_allowed AS run_differential,
      earned_run_avg
    FROM `{DATASET_NAME}.teams`
    WHERE yearID = 2025
  )
);"""

PREDICT_SQL = f"""\
SELECT
  team_name,
  yearID,
  ROUND(predicted_played_postseason_probs[OFFSET(0)].prob, 3) AS playoff_probability,
  predicted_played_postseason AS predicted_postseason,
  COALESCE(won_division, FALSE)
    OR COALESCE(won_wild_card, FALSE)
    OR COALESCE(won_league, FALSE)
    OR COALESCE(won_world_series, FALSE)
    AS actual_postseason
FROM ML.PREDICT(
  MODEL `{DATASET_NAME}.{MODEL_NAME}`,
  (
    SELECT
      team_name,
      yearID,
      won_division, won_wild_card, won_league, won_world_series,
      wins / games AS winning_pct,
      runs_scored - runs_allowed AS run_differential,
      earned_run_avg
    FROM `{DATASET_NAME}.teams`
    WHERE yearID = 2025
  )
)
ORDER BY playoff_probability DESC;"""

# Note on PREDICT_SQL:
# predicted_played_postseason_probs is a STRUCT array with entries for each class.
# OFFSET(0) is the TRUE class (sorted by label value, TRUE > FALSE alphabetically? actually
# BQML returns them in label-sort order; we'll verify in validation that OFFSET(0) is the
# postseason=TRUE prob and adjust if not).

# Build the SQL file
bqml_parts = [
    "-- MLB Race to October — Task 2: Train and evaluate the playoff probability model",
    "-- Generated by mlb_data_foundation.ipynb",
    f"-- Target dataset: {DATASET_NAME} (must contain the teams table from Task 1)",
    "",
    "-- =====================================================================",
    "-- Step 1: Train the model",
    "-- =====================================================================",
    "",
    CREATE_MODEL_SQL,
    "",
    "-- =====================================================================",
    "-- Step 2: Evaluate on the 2025 holdout",
    "-- =====================================================================",
    "",
    EVALUATE_SQL,
    "",
    "-- =====================================================================",
    "-- Step 3: Per-team 2025 predictions (for sanity-checking against memory)",
    "-- =====================================================================",
    "",
    PREDICT_SQL,
    "",
]
bqml_sql_text = "\n".join(bqml_parts)

BQML_SQL_OUT = Path("/content/train_playoff_model.sql")
BQML_SQL_OUT.write_text(bqml_sql_text)
print(f"Wrote {BQML_SQL_OUT} ({len(bqml_sql_text):,} chars, {bqml_sql_text.count(chr(10)):,} lines)")

# Upload to GCS alongside load_lahman.sql
bqml_blob = f"{SCHEMA_GCS_PREFIX}/train_playoff_model.sql"
bucket.blob(bqml_blob).upload_from_filename(str(BQML_SQL_OUT))
print(f"Uploaded to gs://{GCS_BUCKET}/{bqml_blob}")

# Structured statement list for the validator (mirrors GENERATED_STATEMENTS pattern)
BQML_STATEMENTS = [
    ("CREATE MODEL playoff_probability", CREATE_MODEL_SQL),
    ("ML.EVALUATE playoff_probability (2025)", EVALUATE_SQL),
    ("ML.PREDICT playoff_probability (2025)", PREDICT_SQL),
]
print(f"Tracked {len(BQML_STATEMENTS)} BQML statements for validation.")

Wrote /content/train_playoff_model.sql (2,582 chars, 79 lines)
Uploaded to gs://class-demo/mlb-race-to-october/schema/train_playoff_model.sql
Tracked 3 BQML statements for validation.


In [ ]:
import time
from google.cloud import bigquery

VALIDATION_DATASET = f"mlb_validation_{int(time.time())}"
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

print(f"Validation dataset: {PROJECT_ID}.{VALIDATION_DATASET}")
print(f"Location:           {BQ_LOCATION}\n")

# Create the dataset
ds_ref = bigquery.Dataset(f"{PROJECT_ID}.{VALIDATION_DATASET}")
ds_ref.location = BQ_LOCATION
bq.create_dataset(ds_ref)
print(f"\u2713 Created dataset\n")

try:
    # --- Step 1: Load just the teams table (BQML training only needs that one) ---
    teams_load = next(
        stmt for label, stmt in GENERATED_STATEMENTS if label == "LOAD teams"
    )
    sql = teams_load.replace(f"`{DATASET_NAME}.", f"`{VALIDATION_DATASET}.")
    bq.query(sql).result()
    print("\u2713 Loaded teams table\n")

    # --- Step 2: Train the model ---
    print("Training model (logistic regression on 1994-2024)...")
    sql = CREATE_MODEL_SQL.replace(f"`{DATASET_NAME}.", f"`{VALIDATION_DATASET}.")
    train_job = bq.query(sql)
    train_job.result()
    print(f"\u2713 Model trained ({train_job.ended - train_job.started})\n")

    # --- Step 3: Evaluate on 2025 ---
    print("Evaluating on 2025 holdout...")
    sql = EVALUATE_SQL.replace(f"`{DATASET_NAME}.", f"`{VALIDATION_DATASET}.")
    eval_rows = list(bq.query(sql).result())
    metrics = dict(eval_rows[0])

    print("ML.EVALUATE metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:<24} {v:.4f}")
        else:
            print(f"  {k:<24} {v}")

    auc = metrics.get("roc_auc")
    if auc is None:
        print("\n\u26a0\ufe0f  No roc_auc in metrics — check the keys above.")
    elif 0.75 <= auc <= 0.99:
        print(f"\n\u2713 AUC {auc:.3f} is in the expected 0.75–0.99 range.")
    elif auc < 0.75:
        print(f"\n\u2717 AUC {auc:.3f} below 0.75 — model is not learning enough. Investigate features.")
    else:
        print(f"\n\u2717 AUC {auc:.3f} above 0.99 — suspicious, possible target leak.")

    # --- Step 4: Per-team 2025 predictions for eyeball validation ---
    print("\n2025 predictions (sorted by playoff probability)")
    print("-" * 70)
    sql = PREDICT_SQL.replace(f"`{DATASET_NAME}.", f"`{VALIDATION_DATASET}.")
    pred_rows = list(bq.query(sql).result())

    print(f"{'Team':<28} {'Prob':>7} {'Pred':>6} {'Actual':>7}")
    print("-" * 70)
    for row in pred_rows:
        team = row["team_name"][:27]
        prob = row["playoff_probability"]
        pred = "\u2713" if row["predicted_postseason"] else " "
        actual = "\u2713" if row["actual_postseason"] else " "
        marker = "  \u2190 miss" if row["predicted_postseason"] != row["actual_postseason"] else ""
        print(f"{team:<28} {prob:>7.3f} {pred:>6} {actual:>7}{marker}")

    # Summary: how many predictions were correct?
    correct = sum(1 for r in pred_rows if r["predicted_postseason"] == r["actual_postseason"])
    total = len(pred_rows)
    print(f"\n2025 prediction accuracy: {correct}/{total} = {correct/total:.1%}")

finally:
    print(f"\nDropping validation dataset {VALIDATION_DATASET}...")
    bq.delete_dataset(VALIDATION_DATASET, delete_contents=True, not_found_ok=True)
    print("\u2713 Dropped.")

Validation dataset: qwiklabs-gcp-00-dd506b084f93.mlb_validation_1778103474
Location:           US

✓ Created dataset

✓ Loaded teams table

Training model (logistic regression on 1994-2024)...
✓ Model trained (0:00:53.636000)

Evaluating on 2025 holdout...
ML.EVALUATE metrics:
  precision                1.0000
  recall                   0.7500
  accuracy                 0.9000
  f1_score                 0.8571
  log_loss                 0.2237
  roc_auc                  0.9870

✓ AUC 0.987 is in the expected 0.75–0.99 range.

2025 predictions (sorted by playoff probability)
----------------------------------------------------------------------
Team                            Prob   Pred  Actual
----------------------------------------------------------------------
Milwaukee Brewers              0.961      ✓       ✓
Philadelphia Phillies          0.930      ✓       ✓
New York Yankees               0.919      ✓       ✓
Los Angeles Dodgers            0.880      ✓       ✓
Chicago Cubs     

## 10. MLB Rulebook — validation

The rulebook lives in a Gemini Enterprise unstructured data store as one of the documents the front-office chat agent grounds on. For that to work the PDF needs to be text-extractable — a scanned image won't index, even if the file opens fine in a viewer.

The official 2025 MLB Official Baseball Rules PDF lives at:

  https://mktg.mlbstatic.com/mlb/official-information/2025-official-baseball-rules.pdf

The instructor downloads it once and stages it in GCS at:

  gs://class-demo/mlb-race-to-october/rulebook/2025-official-baseball-rules.pdf

This cell validates that:

1. The PDF is present at the expected GCS path.
2. The file is a valid PDF (correct magic bytes).
3. Text extraction works — every sampled page yields non-trivial text, not just whitespace or single-character noise that would indicate a scanned image.
4. A sample of extracted text reads as actual rulebook content (not garbled, not just headers/footers).

When MLB releases a 2026 rulebook (typically early in the calendar year if there are rule changes), the instructor refresh is just: download the new PDF, replace the GCS object, rerun this cell.

In [ ]:
# pypdf is included in Colab Enterprise; pip install on the off chance it isn't.
try:
    import pypdf
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "pypdf"])
    import pypdf

from io import BytesIO

RULEBOOK_GCS_PATH = f"{GCS_PREFIX}/rulebook/2026-official-baseball-rules.pdf"
LOCAL_RULEBOOK = Path(WORK_DIR).parent / "rulebook.pdf"

# 1. Confirm presence in GCS
print(f"Checking gs://{GCS_BUCKET}/{RULEBOOK_GCS_PATH}")
blob = bucket.blob(RULEBOOK_GCS_PATH)
if not blob.exists():
    raise FileNotFoundError(
        f"Rulebook not found at gs://{GCS_BUCKET}/{RULEBOOK_GCS_PATH}\n"
        f"Did you upload it? See section 10 markdown for the manual prep step."
    )
blob.reload()
size_mb = blob.size / 1024 / 1024
print(f"  \u2713 Present, {size_mb:.2f} MB, content-type={blob.content_type}\n")

# 2. Download and validate magic bytes
blob.download_to_filename(str(LOCAL_RULEBOOK))
with open(LOCAL_RULEBOOK, "rb") as f:
    magic = f.read(5)
assert magic == b"%PDF-", f"Not a PDF — magic bytes were {magic!r}"
print(f"  \u2713 Valid PDF magic bytes ({magic.decode('ascii', errors='replace')})\n")

# 3. Open and inspect
reader = pypdf.PdfReader(str(LOCAL_RULEBOOK))
n_pages = len(reader.pages)
print(f"  \u2713 Opened cleanly, {n_pages} pages\n")

# 4. Sample text extraction across the document
sample_indices = [0, n_pages // 4, n_pages // 2, 3 * n_pages // 4, n_pages - 1]
print("Text extraction sample (chars per sampled page):")
print("-" * 60)
text_lengths = []
for i in sample_indices:
    text = reader.pages[i].extract_text() or ""
    text_lengths.append(len(text))
    print(f"  page {i+1:>3} of {n_pages}: {len(text):>5} chars")

# What matters for retrieval: the body of the rulebook extracts cleanly.
# Front matter (title page) and back matter (covers, blank pages) are legitimately
# near-empty in born-digital PDFs and don't affect indexing. We sample 5 pages
# spanning the document and require the 3 interior samples (25%, 50%, 75% through)
# to all have substantial text — that's the rules content.
interior = text_lengths[1:4]  # the three quartile-position pages
min_interior = min(interior)
if min_interior < 200:
    print(f"\n\u26a0\ufe0f  Interior pages thin (min {min_interior} chars) — possible extraction trouble.")
else:
    cover_note = ""
    if text_lengths[0] < 200 or text_lengths[-1] < 200:
        cover_note = f" (first/last page near-empty, normal for cover/back matter)"
    print(f"\n\u2713 Interior pages extract cleanly (min {min_interior} chars across body samples){cover_note}")

# 5. Eyeball a sample — first ~600 chars from a mid-document page,
# which should be in the meat of the rules rather than TOC or index.
mid_page = reader.pages[n_pages // 2]
sample_text = (mid_page.extract_text() or "")[:600].strip()
print(f"\nSample from page {n_pages // 2 + 1}:")
print("-" * 60)
print(sample_text)
print("-" * 60)

# Cleanup local copy — GCS is the source of truth
LOCAL_RULEBOOK.unlink()

Checking gs://class-demo/mlb-race-to-october/rulebook/2026-official-baseball-rules.pdf
  ✓ Present, 1.26 MB, content-type=application/pdf

  ✓ Valid PDF magic bytes (%PDF-)

  ✓ Opened cleanly, 192 pages

Text extraction sample (chars per sampled page):
------------------------------------------------------------
  page   1 of 192:    36 chars
  page  49 of 192:  2223 chars
  page  97 of 192:  1886 chars
  page 145 of 192:  1983 chars
  page 192 of 192:     0 chars

✓ Interior pages extract cleanly (min 1886 chars across body samples) (first/last page near-empty, normal for cover/back matter)

Sample from page 97:
------------------------------------------------------------
85
 Rule 6.02(d)(5) to 6.03(a)(4)
use the rosin bag for the purpose of applying rosin to his bare 
hand or hands. Neither the pitcher nor any other player shall dust 
the ball with the rosin bag; neither shall the pitcher nor any other 
player be permitted to apply rosin from the bag to his glove or 
dust any part o

## 10.5 Resume-from-backup prelude

When this notebook runs for the first time, profiles don't exist yet — Sections 11 and 12 generate them from Gemini. That's slow (~90 minutes for players plus ~25 minutes for teams) and consumes API quota.

After Section 15 ran a profile audit and remediation pass, we copied the original generated profiles to versioned GCS prefixes (`profiles/players-v1/`, `profiles/teams-v1/`). On any subsequent rerun of this notebook, those v1 backups already contain valid profile content — we don't need to regenerate them via Gemini.

This cell checks for v1 backups and, if they exist, restores them into the working profile prefixes that Sections 11 and 12 would otherwise populate. Sections 11 and 12 then short-circuit their generation logic via the `SKIP_PLAYER_GENERATION` and `SKIP_TEAM_GENERATION` flags this cell sets.

On a true first run (no v1 backup exists yet), this cell is a no-op — Sections 11 and 12 run normally. The prelude is purely a recovery shortcut for re-runs.

Section 15's targeted regeneration still runs in either case — its scope is the 2,479 profiles affected by the audit findings, not the full 24,500-profile corpus

In [ ]:
# What variables are defined that look like they hold the profile/team prefixes?
import re
relevant = {k: v for k, v in globals().items()
            if isinstance(v, str)
            and ("profile" in k.lower() or "team" in k.lower() or "prefix" in k.lower() or "gcs" in k.lower())
            and not k.startswith("_")}
for k, v in sorted(relevant.items()):
    print(f"  {k} = {v!r}")

  GCS_BUCKET = 'class-demo'
  GCS_BUCKET_NAME = 'class-demo'
  GCS_PREFIX = 'mlb-race-to-october'
  LAHMAN_GCS_DEST_PREFIX = 'mlb-race-to-october/lahman'
  LAHMAN_GCS_SOURCE_PREFIX = 'mlb-race-to-october/lahman-source'
  PLAYERS_PREFIX = 'mlb-race-to-october/profiles/players'
  PLAYERS_V1_PREFIX = 'mlb-race-to-october/profiles/players-v1'
  RULEBOOK_GCS_PATH = 'mlb-race-to-october/rulebook/2026-official-baseball-rules.pdf'
  SCHEMA_GCS_PREFIX = 'mlb-race-to-october/schema'
  V1_PLAYERS_PREFIX = 'mlb-race-to-october/profiles/players-v1'
  V1_TEAMS_PREFIX = 'mlb-race-to-october/profiles/teams-v1'
  profile = 'Hank Aaron appears in the Lahman Baseball Database, as a position player who played 23 seasons from 1954 to 1976. Detailed narrative profile unavailable.'
  team = 'Colorado Rockies'
  teams_load = "LOAD DATA OVERWRITE `mlb_race_to_october.teams`\nFROM FILES (\n  format = 'PARQUET',\n  uris = ['gs://class-demo/mlb-race-to-october/lahman/Teams.parquet']\n);"


In [ ]:
from google.api_core import retry as gapi_retry
from google.api_core import exceptions as gapi_exceptions

# Retry policy: 503s, 502s, 504s, and gateway timeouts are all retryable
_GCS_RETRY = gapi_retry.Retry(
    predicate=gapi_retry.if_exception_type(
        gapi_exceptions.ServiceUnavailable,
        gapi_exceptions.GatewayTimeout,
        gapi_exceptions.BadGateway,
        gapi_exceptions.InternalServerError,
        ConnectionError,
    ),
    initial=1.0,
    maximum=10.0,
    multiplier=2.0,
    deadline=60.0,
)

TEAMS_PREFIX = f"{GCS_PREFIX}/profiles/teams"


def restore_from_backup_if_exists(v1_prefix: str, target_prefix: str, label: str) -> bool:
    """If v1_prefix has content, copy it into target_prefix. Returns True if restored."""
    v1_blobs = [b for b in client.list_blobs(GCS_BUCKET, prefix=f"{v1_prefix}/")
                if b.name.endswith(".md")]
    if not v1_blobs:
        print(f"  {label}: no v1 backup found, will regenerate from scratch")
        return False

    target_blobs = {b.name.split("/")[-1]: b.md5_hash
                    for b in client.list_blobs(GCS_BUCKET, prefix=f"{target_prefix}/")
                    if b.name.endswith(".md")}

    print(f"  {label}: v1 backup has {len(v1_blobs):,} files; restoring to target prefix...")
    restored = 0
    skipped_match = 0
    failed = []

    for src_blob in tqdm(v1_blobs, desc=f"  Restoring {label}"):
        fname = src_blob.name.split("/")[-1]
        if target_blobs.get(fname) == src_blob.md5_hash:
            skipped_match += 1
            continue
        dst_name = src_blob.name.replace(v1_prefix, target_prefix, 1)
        try:
            bucket.copy_blob(src_blob, bucket, dst_name, retry=_GCS_RETRY)
            restored += 1
        except Exception as e:
            failed.append((fname, str(e)[:80]))

    print(f"  {label}: {restored:,} restored, {skipped_match:,} already matched v1, {len(failed)} failed")
    if failed:
        print(f"  Failures (first 5):")
        for fname, err in failed[:5]:
            print(f"    {fname}: {err}")
    return True


print("Checking for v1 profile backups to restore...")
players_restored = restore_from_backup_if_exists(
    V1_PLAYERS_PREFIX, PLAYERS_PREFIX, "players"
)
teams_restored = restore_from_backup_if_exists(
    V1_TEAMS_PREFIX, TEAMS_PREFIX, "teams"
)

SKIP_PLAYER_GENERATION = players_restored
SKIP_TEAM_GENERATION = teams_restored

if players_restored or teams_restored:
    print("\nProfile backups restored. Sections 11/12 will skip Gemini generation.")
    print("Section 15 regen still runs targeted updates on the 2,479-profile scope.")
else:
    print("\nFirst-run mode. Sections 11/12 will generate profiles from Gemini.")

Checking for v1 profile backups to restore...
  players: v1 backup has 24,270 files; restoring to target prefix...


  Restoring players: 100%|██████████| 24270/24270 [00:00<00:00, 624096.51it/s]


  players: 0 restored, 24,270 already matched v1, 0 failed
  teams: v1 backup has 203 files; restoring to target prefix...


  Restoring teams: 100%|██████████| 203/203 [00:00<00:00, 409347.94it/s]

  teams: 0 restored, 203 already matched v1, 0 failed

Profile backups restored. Sections 11/12 will skip Gemini generation.
Section 15 regen still runs targeted updates on the 2,479-profile scope.


## 11. Player and team profiles

The Gemini Enterprise / CX Agent Studio chat agents need narrative documents to ground on. The rulebook covers rules questions; profiles cover "who is this player" and "what's this team's story."

**Scope:**
- **Players**: all 24,270 entries in the People table — every person Lahman tracks, from 19th-century pioneers to 2025 debutants. Filtering to "real" players is its own classification problem; thin data → thin profile is acceptable.
- **Teams**: all 30 current MLB franchises, identified by `franchID` (stable across relocations).

**Output**: markdown documents at `gs://class-demo/mlb-race-to-october/profiles/players/{playerID}.md` and `.../teams/{franchID}.md`. The Gemini Enterprise / CX Agent Studio data store ingests this prefix the same way it ingests the rulebook prefix.

**Approach** (mirrors the Team USA pipeline):
1. For each player, assemble structured context from the 9 Lahman tables — identity, career magnitudes, awards, postseason
2. Pass that context to Gemini 2.5 Flash with Google Search grounding
3. Prompt asks for 200-300 words across 2-3 paragraphs of narrative
4. Search supplies *current* state (active players' current teams, recent injuries, 2025 highlights) that Lahman's annual cadence can't
5. Parallel processing with thread-local clients, ~50 workers
6. Resume-safe checkpointing — any rerun skips successful entries and retries failures

**Why narrative not stats**: the lab's pedagogical hinge is that retrieval-grounded chat agents can describe but cannot compute. Profiles get career totals and notable highlights, deliberately *not* year-by-year stat lines. "Best WHIP over last three seasons" stays the BQ tool's job in Task 3.

**Why 24K not just active rosters**: the agent should be able to answer "tell me about Hank Aaron" as well as "tell me about Aaron Judge." Modern + legends only feels arbitrary; full coverage is the cleaner story.

**Currency caveat**: Lahman's "last team" for an active player is whatever team they finished the most-recent-Lahman-season with. The prompt explicitly tells Gemini to prefer Google Search results over the structured "current team" field when they conflict — important for traded players, free agents, and 2026-season roster moves Lahman won't see until the next annual update.

In [ ]:
import json
import time
import csv
import re
import threading
from pathlib import Path
from typing import Optional, List
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from tqdm.notebook import tqdm

# Make sure Vertex AI SDK is available
try:
    from google import genai
    from google.genai import types
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "google-genai"])
    from google import genai
    from google.genai import types

import google.auth


class PlayerProfileConfig:
    MODEL_NAME = 'gemini-2.5-flash'
    TEMPERATURE = 1.0          # Slightly lower than Team USA's 1.2 — we want consistent prose, not creative variance
    MAX_OUTPUT_TOKENS = 1200

    BATCH_SIZE = 200
    MAX_WORKERS = 50
    SAVE_INTERVAL = 200
    MAX_RETRIES = 3
    RETRY_DELAY = 2

    MIN_PROFILE_LENGTH = 200   # ~30 words minimum, catches truncated/empty responses
    MAX_PROFILE_LENGTH = 3000  # ~500 words ceiling, prevents runaway

    PROGRESS_FILE = '/tmp/mlb_player_profiles_progress.csv'
    ERROR_LOG = '/tmp/mlb_player_profiles_errors.csv'


# Thread-local Gemini client (reused from section 11 if it exists)
_thread_local = threading.local()

def get_client():
    if getattr(_thread_local, "client", None) is None:
        _thread_local.client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location="us-central1",
        )
    return _thread_local.client


# --- Pre-compute lookups from the in-memory Parquet data ---
# These are dict-of-dicts so the assembly fn can do O(1) lookups per player

print("Building per-player lookup tables from staged Parquet...")

people_df = pd.read_parquet(PARQUET_DIR / "People.parquet")
batting_df = pd.read_parquet(PARQUET_DIR / "Batting.parquet")
pitching_df = pd.read_parquet(PARQUET_DIR / "Pitching.parquet")
appearances_df = pd.read_parquet(PARQUET_DIR / "Appearances.parquet")
allstar_df = pd.read_parquet(PARQUET_DIR / "AllstarFull.parquet")
awards_df = pd.read_parquet(PARQUET_DIR / "AwardsPlayers.parquet")
batting_post_df = pd.read_parquet(PARQUET_DIR / "BattingPost.parquet")
pitching_post_df = pd.read_parquet(PARQUET_DIR / "PitchingPost.parquet")
teams_df = pd.read_parquet(PARQUET_DIR / "Teams.parquet")

# Career batting totals per player
batting_career = batting_df.groupby("playerID").agg(
    seasons=("yearID", "nunique"),
    games=("games", "sum"),
    at_bats=("at_bats", "sum"),
    hits=("hits", "sum"),
    home_runs=("home_runs", "sum"),
    rbi=("runs_batted_in", "sum"),
    runs=("runs", "sum"),
    walks=("walks", "sum"),
    strikeouts=("strikeouts", "sum"),
    stolen_bases=("stolen_bases", "sum"),
    first_year=("yearID", "min"),
    last_year=("yearID", "max"),
    last_team=("teamID", "last"),
).to_dict("index")

# Career pitching totals per player
pitching_career = pitching_df.groupby("playerID").agg(
    seasons=("yearID", "nunique"),
    games=("games", "sum"),
    games_started=("games_started", "sum"),
    wins=("wins", "sum"),
    losses=("losses", "sum"),
    saves=("saves", "sum"),
    strikeouts=("strikeouts", "sum"),
    earned_runs_allowed=("earned_runs_allowed", "sum"),
    outs_pitched=("outs_pitched", "sum"),
    first_year=("yearID", "min"),
    last_year=("yearID", "max"),
    last_team=("teamID", "last"),
).to_dict("index")

# All-Star selections per player
allstar_lookup = allstar_df.groupby("playerID").agg(
    selections=("yearID", "count"),
    years=("yearID", lambda x: sorted(x.unique().tolist())),
).to_dict("index")

# Awards per player (list of (year, award) tuples)
awards_lookup = {}
for player_id, group in awards_df.groupby("playerID"):
    awards_lookup[player_id] = [
        (int(row["yearID"]), row["awardID"])
        for _, row in group.iterrows()
    ]

# Postseason batting per player
post_batting_lookup = batting_post_df.groupby("playerID").agg(
    appearances=("yearID", "nunique"),
    home_runs=("home_runs", "sum"),
    rbi=("runs_batted_in", "sum"),
    rounds=("round", lambda x: list(x.unique())),
).to_dict("index")

# Postseason pitching per player
post_pitching_lookup = pitching_post_df.groupby("playerID").agg(
    appearances=("yearID", "nunique"),
    wins=("wins", "sum"),
    saves=("saves", "sum"),
    rounds=("round", lambda x: list(x.unique())),
).to_dict("index")

# World Series wins per player — derived from rounds == 'WS' + their team won
ws_wins_by_player = {}
for player_id, group in batting_post_df[batting_post_df["round"] == "WS"].groupby("playerID"):
    # Cross-reference with Teams to confirm they were on the winning team
    ws_years = []
    for _, row in group.iterrows():
        team_match = teams_df[
            (teams_df["yearID"] == row["yearID"]) &
            (teams_df["teamID"] == row["teamID"]) &
            (teams_df["won_world_series"] == True)
        ]
        if len(team_match) > 0:
            ws_years.append(int(row["yearID"]))
    if ws_years:
        ws_wins_by_player[player_id] = sorted(set(ws_years))

# Same for pitchers
for player_id, group in pitching_post_df[pitching_post_df["round"] == "WS"].groupby("playerID"):
    ws_years = ws_wins_by_player.get(player_id, [])
    for _, row in group.iterrows():
        team_match = teams_df[
            (teams_df["yearID"] == row["yearID"]) &
            (teams_df["teamID"] == row["teamID"]) &
            (teams_df["won_world_series"] == True)
        ]
        if len(team_match) > 0:
            ws_years.append(int(row["yearID"]))
    if ws_years:
        ws_wins_by_player[player_id] = sorted(set(ws_years))

# Position from Appearances — most-frequent non-pitcher position
def primary_position(player_id: str) -> Optional[str]:
    rows = appearances_df[appearances_df["playerID"] == player_id]
    if len(rows) == 0:
        return None
    pos_cols = {
        "games_pitcher": "P", "games_catcher": "C", "games_first_base": "1B",
        "games_second_base": "2B", "games_third_base": "3B", "games_shortstop": "SS",
        "games_left_field": "LF", "games_center_field": "CF", "games_right_field": "RF",
        "games_designated_hitter": "DH",
    }
    totals = {pos: rows[col].sum() for col, pos in pos_cols.items() if col in rows.columns}
    if not totals or max(totals.values()) == 0:
        return None
    return max(totals, key=totals.get)


print(f"  Lookups built for batting={len(batting_career)}, pitching={len(pitching_career)}, "
      f"all-star={len(allstar_lookup)}, awards={len(awards_lookup)}")
print(f"  Postseason: batters={len(post_batting_lookup)}, pitchers={len(post_pitching_lookup)}, "
      f"WS winners={len(ws_wins_by_player)}")


def build_player_context(player_row) -> str:
    """Assemble a structured context block for a single player."""
    pid = player_row["playerID"]
    name = player_row.get("name_full") or f"{player_row.get('name_first', '')} {player_row.get('name_last', '')}".strip()

    lines = [f"=== PLAYER ===", f"Name: {name}", f"Lahman ID: {pid}"]

    # Bio
    if pd.notna(player_row.get("birth_year")):
        birth_parts = [str(int(player_row["birth_year"]))]
        if pd.notna(player_row.get("birth_city")):
            birth_parts.append(f"in {player_row['birth_city']}")
        if pd.notna(player_row.get("birth_country")) and player_row["birth_country"] != "USA":
            birth_parts.append(f"({player_row['birth_country']})")
        lines.append(f"Born: {' '.join(birth_parts)}")

    if pd.notna(player_row.get("debut_date")):
        lines.append(f"MLB debut: {player_row['debut_date']}")
    if pd.notna(player_row.get("final_game_date")):
        lines.append(f"Final game: {player_row['final_game_date']} (retired)")
    elif pd.notna(player_row.get("debut_date")):
        lines.append("Final game: not recorded (may be active)")

    if pd.notna(player_row.get("bats")):
        lines.append(f"Bats: {player_row['bats']}")
    if pd.notna(player_row.get("throws")):
        lines.append(f"Throws: {player_row['throws']}")

    # Position
    pos = primary_position(pid)
    if pos:
        lines.append(f"Primary position: {pos}")

    # Batting career
    if pid in batting_career:
        b = batting_career[pid]
        lines.append("\n=== BATTING (CAREER TOTALS) ===")
        lines.append(f"Seasons: {int(b['seasons'])} ({int(b['first_year'])}–{int(b['last_year'])})")
        lines.append(f"Last team in Lahman: {b['last_team']}")
        lines.append(f"Games: {int(b['games'])}, AB: {int(b['at_bats'])}, H: {int(b['hits'])}")
        lines.append(f"HR: {int(b['home_runs'])}, RBI: {int(b['rbi'])}, R: {int(b['runs'])}, SB: {int(b['stolen_bases'])}")
        if b["at_bats"] > 0:
            avg = b["hits"] / b["at_bats"]
            lines.append(f"Career batting avg: {avg:.3f}")

    # Pitching career
    if pid in pitching_career:
        p = pitching_career[pid]
        lines.append("\n=== PITCHING (CAREER TOTALS) ===")
        lines.append(f"Seasons: {int(p['seasons'])} ({int(p['first_year'])}–{int(p['last_year'])})")
        lines.append(f"Last team in Lahman: {p['last_team']}")
        lines.append(f"W-L: {int(p['wins'])}-{int(p['losses'])}, Saves: {int(p['saves'])}, K: {int(p['strikeouts'])}")
        if p["outs_pitched"] > 0:
            ip = p["outs_pitched"] / 3.0
            era = (p["earned_runs_allowed"] * 9.0) / ip if ip > 0 else None
            lines.append(f"Innings pitched: {ip:.1f}" + (f", ERA: {era:.2f}" if era else ""))

    # All-Star
    if pid in allstar_lookup:
        a = allstar_lookup[pid]
        years_str = ", ".join(str(y) for y in a["years"][:5])
        if len(a["years"]) > 5:
            remaining = len(a["years"]) - 5
            years_str += f", and {remaining} more"
        lines.append(f"\nAll-Star selections: {int(a['selections'])} ({years_str})")

    # Awards
    if pid in awards_lookup:
        awards = awards_lookup[pid]
        # Group major awards
        major = [a for a in awards if a[1] in (
            "Most Valuable Player", "Cy Young Award", "Rookie of the Year",
            "World Series MVP", "Triple Crown",
        )]
        gold_gloves = [a for a in awards if "Gold Glove" in a[1]]
        silver_sluggers = [a for a in awards if "Silver Slugger" in a[1]]

        if major:
            lines.append("Major awards: " + "; ".join(f"{a[1]} ({a[0]})" for a in major))
        if gold_gloves:
            yrs = sorted(set(a[0] for a in gold_gloves))
            lines.append(f"Gold Gloves: {len(yrs)} ({yrs[0]}–{yrs[-1]})")
        if silver_sluggers:
            yrs = sorted(set(a[0] for a in silver_sluggers))
            lines.append(f"Silver Sluggers: {len(yrs)} ({yrs[0]}–{yrs[-1]})")

    # Postseason
    post_lines = []
    if pid in post_batting_lookup:
        pb = post_batting_lookup[pid]
        post_lines.append(f"Postseason batting: {int(pb['appearances'])} postseasons, "
                          f"{int(pb['home_runs'])} HR, {int(pb['rbi'])} RBI")
    if pid in post_pitching_lookup:
        pp = post_pitching_lookup[pid]
        post_lines.append(f"Postseason pitching: {int(pp['appearances'])} postseasons, "
                          f"{int(pp['wins'])} W, {int(pp['saves'])} SV")
    if pid in ws_wins_by_player:
        wins = ws_wins_by_player[pid]
        post_lines.append(f"World Series wins: {len(wins)} ({', '.join(str(y) for y in wins)})")
    if post_lines:
        lines.append("\n=== POSTSEASON ===")
        lines.extend(post_lines)

    return "\n".join(lines)


def build_data_only_profile(player_row, context: str) -> str:
    """Fallback used when Gemini fails. Factual sentence built from structured data."""
    pid = player_row["playerID"]
    name = player_row.get("name_full") or pid

    parts = [f"{name} appears in the Lahman Baseball Database"]

    if pid in batting_career:
        b = batting_career[pid]
        parts.append(f"as a position player who played {int(b['seasons'])} seasons "
                     f"from {int(b['first_year'])} to {int(b['last_year'])}")
    elif pid in pitching_career:
        p = pitching_career[pid]
        parts.append(f"as a pitcher who played {int(p['seasons'])} seasons "
                     f"from {int(p['first_year'])} to {int(p['last_year'])}")

    return ", ".join(parts) + ". Detailed narrative profile unavailable."


def create_player_prompt(player_row) -> str:
    """Prompt Gemini for a 200-300 word, 2-3 paragraph profile."""
    context = build_player_context(player_row)
    name = player_row.get("name_full") or player_row.get("playerID")

    return f"""You are writing a brief player profile for a baseball front-office reference document. The structured data below comes from the Lahman Baseball Database. Use Google Search to add current-state context (current team, recent season highlights, injury status, 2025/2026 developments) for active or recently-active players. Lahman's annual cadence means its "last team" field can be stale for active players — prefer Google Search when they conflict.

{context}

=== INSTRUCTIONS ===
Write a 200-300 word profile, 2-3 paragraphs, plain markdown. NO heading or title — start directly with the prose. Structure:

- **Paragraph 1**: Who they are. Position, role, current team (or last team if retired), one or two things they're best known for. For active players use current-state language ("is the Yankees' captain"); for retired players use past-tense ("was a power-hitting outfielder"). Tight, factual.

- **Paragraph 2**: Career arc. Debut, key milestones, awards, postseason highlights. Weave the structured data into prose — don't list stats.

- **Paragraph 3** (optional, only if there's real signal): Notable narrative — recent injury, milestone year, trade, retirement, etc. Skip this paragraph entirely if Google Search doesn't surface anything concrete.

Rules:
- 200-300 words total. Stay tight.
- Don't fabricate. If you don't know something, leave it out.
- Don't include year-by-year stat lines or detailed splits. Career totals and milestone years only.
- Don't editorialize about Hall of Fame candidacy unless they're already inducted (which would be in the awards data).
- Output the prose only — no JSON wrapper, no headers, no bullet points.

Profile for {name}:"""


def parse_profile_response(raw_text: str) -> str:
    """Clean the response — strip code fences, citation markers, excess whitespace."""
    if not raw_text:
        return ""
    text = raw_text.strip()
    # Strip any accidental code fences
    text = re.sub(r'^```(?:markdown)?\s*\n', '', text)
    text = re.sub(r'\n```$', '', text)
    # Strip Gemini search citation artifacts: [1], [1, 2, 7]
    text = re.sub(r'\s*\[[\d,\s]+\]', '', text)
    # Collapse run-on whitespace inside paragraphs but preserve paragraph breaks
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def generate_player_profile(player_row, retries: int = PlayerProfileConfig.MAX_RETRIES) -> dict:
    """Generate one profile. Returns dict with profile + status fields."""
    pid = player_row["playerID"]
    name = player_row.get("name_full") or pid
    prompt = create_player_prompt(player_row)

    config = types.GenerateContentConfig(
        temperature=PlayerProfileConfig.TEMPERATURE,
        max_output_tokens=PlayerProfileConfig.MAX_OUTPUT_TOKENS,
        tools=[types.Tool(google_search=types.GoogleSearch())],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )

    for attempt in range(retries):
        try:
            response = get_client().models.generate_content(
                model=PlayerProfileConfig.MODEL_NAME,
                contents=prompt,
                config=config,
            )

            raw = None
            if hasattr(response, 'text') and response.text:
                raw = response.text
            elif hasattr(response, 'candidates') and response.candidates:
                candidate = response.candidates[0]
                if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                    parts = candidate.content.parts
                    if parts and hasattr(parts[0], 'text'):
                        raw = parts[0].text

            if not raw:
                if attempt < retries - 1:
                    time.sleep(PlayerProfileConfig.RETRY_DELAY)
                    continue
                ctx = build_player_context(player_row)
                return {
                    "playerID": pid, "name": name,
                    "profile": build_data_only_profile(player_row, ctx),
                    "source": "data_only", "error": "No text in API response"
                }

            profile = parse_profile_response(raw)

            if len(profile) < PlayerProfileConfig.MIN_PROFILE_LENGTH:
                if attempt < retries - 1:
                    time.sleep(PlayerProfileConfig.RETRY_DELAY)
                    continue
                ctx = build_player_context(player_row)
                return {
                    "playerID": pid, "name": name,
                    "profile": build_data_only_profile(player_row, ctx),
                    "source": "data_only", "error": f"Profile too short ({len(profile)} chars)"
                }

            if len(profile) > PlayerProfileConfig.MAX_PROFILE_LENGTH:
                profile = profile[:PlayerProfileConfig.MAX_PROFILE_LENGTH] + "..."

            return {
                "playerID": pid, "name": name,
                "profile": profile, "source": "gemini", "error": None
            }

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(PlayerProfileConfig.RETRY_DELAY * (attempt + 1))
                continue
            ctx = build_player_context(player_row)
            return {
                "playerID": pid, "name": name,
                "profile": build_data_only_profile(player_row, ctx),
                "source": "data_only", "error": f"{type(e).__name__}: {str(e)}"
            }

    ctx = build_player_context(player_row)
    return {
        "playerID": pid, "name": name,
        "profile": build_data_only_profile(player_row, ctx),
        "source": "data_only", "error": "Max retries exceeded"
    }


def log_player_error(player_id, name, error_msg):
    file_exists = Path(PlayerProfileConfig.ERROR_LOG).exists()
    with open(PlayerProfileConfig.ERROR_LOG, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['timestamp', 'playerID', 'name', 'error'])
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'timestamp': datetime.now().isoformat(),
            'playerID': player_id,
            'name': name,
            'error': error_msg,
        })


print(f"\n✅ Profile generation functions defined")
print(f"   Total players to process: {len(people_df):,}")

Building per-player lookup tables from staged Parquet...
  Lookups built for batting=24011, pitching=12134, all-star=2308, awards=2499
  Postseason: batters=5569, pitchers=2304, WS winners=1810

✅ Profile generation functions defined
   Total players to process: 24,270


In [ ]:
# Quick prompt + generation test on three diverse players before bulk run.
# Goal: verify the prompt produces 200-300 word profiles with the right shape.

test_pids = [
    "judgeaa01",   # Aaron Judge — current star, lots of Search material
    "aaronha01",   # Hank Aaron — HoFer, all in structured data, minimal need for Search
    "smithos01",   # Ozzie Smith — HoFer pitcher/SS test (actually shortstop, all-time defender)
]

for pid in test_pids:
    rows = people_df[people_df["playerID"] == pid]
    if len(rows) == 0:
        print(f"⚠️  {pid} not found in People")
        continue
    row = rows.iloc[0]

    print("=" * 70)
    print(f"TEST: {row.get('name_full', pid)}")
    print("=" * 70)

    ctx = build_player_context(row)
    print(f"\nStructured context ({len(ctx)} chars):")
    print(ctx)
    print()

    result = generate_player_profile(row)
    profile = result["profile"]
    word_count = len(profile.split())

    print(f"\n--- Profile ({len(profile)} chars, ~{word_count} words, source={result['source']}) ---")
    print(profile)
    if result["error"]:
        print(f"\nError: {result['error']}")
    print()

TEST: Aaron Judge

Structured context (643 chars):
=== PLAYER ===
Name: Aaron Judge
Lahman ID: judgeaa01
Born: 1992 in Linden
MLB debut: 2016-08-13
Final game: not recorded (may be active)
Bats: R
Throws: R
Primary position: RF

=== BATTING (CAREER TOTALS) ===
Seasons: 10 (2016–2025)
Last team in Lahman: NYA
Games: 1145, AB: 4105, H: 1205
HR: 368, RBI: 830, R: 873, SB: 65
Career batting avg: 0.294

All-Star selections: 7 (2017, 2018, 2021, 2022, 2023, and 2 more)
Major awards: Most Valuable Player (2022); Most Valuable Player (2024); Most Valuable Player (2025); Rookie of the Year (2017)
Silver Sluggers: 5 (2017–2025)

=== POSTSEASON ===
Postseason batting: 8 postseasons, 17 HR, 41 RBI


--- Profile (1646 chars, ~301 words, source=gemini) ---
Aaron Judge is a prominent right fielder for the New York Yankees, known for his exceptional power hitting and his role as the team's captain. Standing at 6 feet 7 inches and weighing 282 pounds, he is one of the largest players in MLB and is ofte

In [ ]:
# Alias: some downstream cells reference these older variable names
PROFILES_GCS_PREFIX = PLAYERS_PREFIX
TEAMS_GCS_PREFIX = TEAMS_PREFIX if 'TEAMS_PREFIX' in dir() else f"{GCS_PREFIX}/profiles/teams"

def process_one_player(idx_row):
    idx, row = idx_row
    result = generate_player_profile(row)
    if result.get("error"):
        log_player_error(result["playerID"], result["name"], result["error"])
    return result


def save_player_progress(results, filename):
    df = pd.DataFrame(results)
    df.to_csv(filename, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)


def rebuild_player_profiles_from_gcs() -> pd.DataFrame:
    """When v1 backup was restored by the prelude cell, skip Gemini and rebuild
    the in-memory dataframe by reading the GCS-restored profile content."""
    print("📦 Rebuilding df_player_profiles from GCS-restored content...")

    def fetch_one(blob):
        pid = blob.name.split("/")[-1].replace(".md", "")
        try:
            return {"playerID": pid, "profile": blob.download_as_text()}
        except Exception:
            return None

    player_blobs = [b for b in client.list_blobs(GCS_BUCKET, prefix=f"{PROFILES_GCS_PREFIX}/")
                    if b.name.endswith(".md")]
    print(f"  Downloading {len(player_blobs):,} restored profiles in parallel...")

    rows = []
    with ThreadPoolExecutor(max_workers=50) as ex:
        futures = [ex.submit(fetch_one, b) for b in player_blobs]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Reading profiles"):
            r = fut.result()
            if r is not None:
                rows.append(r)

    df = pd.DataFrame(rows)
    df = df.merge(
        people_df[['playerID', 'name_full']].rename(columns={'name_full': 'name'}),
        on='playerID', how='left'
    )
    df['source'] = 'gemini'   # all v1 content was Gemini-generated
    df['error'] = None

    # Save into the progress file so subsequent reruns find it (idempotent)
    save_player_progress(df.to_dict('records'), PlayerProfileConfig.PROGRESS_FILE)

    print(f"\n  ✓ Rebuilt {len(df):,} profiles from GCS")
    return df


def generate_all_player_profiles(people_table) -> pd.DataFrame:
    # Backup-aware short-circuit: when prelude cell restored v1 profiles, skip
    # Gemini generation entirely and rebuild df from the GCS content.
    if SKIP_PLAYER_GENERATION:
        print("⏭️  Section 11: Gemini generation skipped — v1 backup restored by prelude cell")
        return rebuild_player_profiles_from_gcs()

    results = []
    processed_ids = set()

    progress_path = Path(PlayerProfileConfig.PROGRESS_FILE)
    if progress_path.exists():
        print(f"📂 Found progress file: {PlayerProfileConfig.PROGRESS_FILE}")
        df_prog = pd.read_csv(PlayerProfileConfig.PROGRESS_FILE)
        successful = df_prog[df_prog["error"].isna() | (df_prog["error"] == "")]
        failed = df_prog[df_prog["error"].notna() & (df_prog["error"] != "")]
        processed_ids = set(successful["playerID"])
        for _, row in successful.iterrows():
            results.append({
                "playerID": row["playerID"], "name": row.get("name", ""),
                "profile": row.get("profile", ""), "source": row.get("source", ""),
                "error": None,
            })
        print(f"  ✓ Loaded {len(processed_ids):,} successful profiles")
        print(f"  🔄 Will retry {len(failed):,} previously-failed profiles")

    todo = people_table[~people_table["playerID"].isin(processed_ids)].copy()
    if len(todo) == 0:
        print("✅ All profiles already generated!")
        return pd.DataFrame(results)

    print(f"\nStarting parallel profile generation")
    print(f"  Total players:     {len(people_table):,}")
    print(f"  Already done:      {len(processed_ids):,}")
    print(f"  To process:        {len(todo):,}")
    print(f"  Workers:           {PlayerProfileConfig.MAX_WORKERS}")
    print(f"  Model:             {PlayerProfileConfig.MODEL_NAME} + Google Search\n")

    start = time.time()
    rows = list(todo.iterrows())
    n_batches = (len(rows) + PlayerProfileConfig.BATCH_SIZE - 1) // PlayerProfileConfig.BATCH_SIZE

    with tqdm(total=len(rows), desc="Generating profiles") as pbar:
        for b in range(n_batches):
            s = b * PlayerProfileConfig.BATCH_SIZE
            e = min(s + PlayerProfileConfig.BATCH_SIZE, len(rows))
            batch = rows[s:e]

            with ThreadPoolExecutor(max_workers=PlayerProfileConfig.MAX_WORKERS) as ex:
                futures = {ex.submit(process_one_player, item): item for item in batch}
                for fut in as_completed(futures):
                    results.append(fut.result())
                    pbar.update(1)

            save_player_progress(results, PlayerProfileConfig.PROGRESS_FILE)

    elapsed = time.time() - start
    successes = sum(1 for r in results if not r.get("error"))
    errors = len(results) - successes

    print(f"\nProfile generation complete")
    print(f"  Total profiles:  {len(results):,}")
    print(f"  Successful:      {successes:,}")
    print(f"  Errors / fallback: {errors:,}")
    print(f"  Time:            {elapsed/60:.1f} minutes")
    print(f"  Throughput:      {len(todo)/elapsed*60:.1f} players/minute")

    return pd.DataFrame(results)


df_player_profiles = generate_all_player_profiles(people_df)

⏭️  Section 11: Gemini generation skipped — v1 backup restored by prelude cell
📦 Rebuilding df_player_profiles from GCS-restored content...


Reading profiles:   0%|          | 0/24270 [00:00<?, ?it/s]


  ✓ Rebuilt 24,270 profiles from GCS


In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed

PROFILES_GCS_PREFIX = f"{GCS_PREFIX}/profiles/players"

print(f"Syncing {len(df_player_profiles):,} player profiles to "
      f"gs://{GCS_BUCKET}/{PROFILES_GCS_PREFIX}/")

# Build MD5 map of what's already in GCS (one cheap list call, no downloads)
print("  Listing existing GCS profiles to identify what needs upload...")
remote_hashes = {}
for blob in client.list_blobs(GCS_BUCKET, prefix=f"{PROFILES_GCS_PREFIX}/"):
    if blob.name.endswith(".md"):
        pid = blob.name.split("/")[-1].replace(".md", "")
        # GCS md5_hash is base64-encoded; convert to hex for comparison
        if blob.md5_hash:
            import base64
            remote_hashes[pid] = base64.b64decode(blob.md5_hash).hex()

print(f"  Existing GCS profiles: {len(remote_hashes):,}")


def local_md5(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def sync_one(row):
    """Upload a profile only if the GCS copy doesn't match local content."""
    if not isinstance(row.get("profile"), str) or not row["profile"]:
        return ("skip_empty", row["playerID"])
    pid = row["playerID"]
    local_hash = local_md5(row["profile"])
    if remote_hashes.get(pid) == local_hash:
        return ("skip_match", pid)
    blob_path = f"{PROFILES_GCS_PREFIX}/{pid}.md"
    try:
        bucket.blob(blob_path).upload_from_string(row["profile"], content_type="text/markdown")
        return ("uploaded", pid)
    except Exception as e:
        return ("error", pid, str(e)[:120])


# Run uploads in parallel — only the changed ones do real work
rows = [row for _, row in df_player_profiles.iterrows()]
results = []
with ThreadPoolExecutor(max_workers=50) as ex:
    futures = [ex.submit(sync_one, r) for r in rows]
    with tqdm(total=len(futures), desc="Syncing") as pbar:
        for fut in as_completed(futures):
            results.append(fut.result())
            pbar.update(1)

uploaded = sum(1 for r in results if r[0] == "uploaded")
skipped_match = sum(1 for r in results if r[0] == "skip_match")
skipped_empty = sum(1 for r in results if r[0] == "skip_empty")
errored = sum(1 for r in results if r[0] == "error")

print(f"\n✓ Uploaded (changed):   {uploaded:,}")
print(f"⏭️  Skipped (already in sync): {skipped_match:,}")
print(f"⚠️  Skipped (empty profile):  {skipped_empty}")
print(f"✗ Errored:                  {errored}")

if errored:
    for r in results:
        if r[0] == "error":
            log_player_error(r[1], "", f"upload: {r[2]}")

Syncing 24,270 player profiles to gs://class-demo/mlb-race-to-october/profiles/players/
  Listing existing GCS profiles to identify what needs upload...
  Existing GCS profiles: 24,270


Syncing:   0%|          | 0/24270 [00:00<?, ?it/s]


✓ Uploaded (changed):   0
⏭️  Skipped (already in sync): 24,270
⚠️  Skipped (empty profile):  0
✗ Errored:                  0


Let's do some validation and spot checking

In [ ]:
blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{PROFILES_GCS_PREFIX}/"))
md_blobs = [b for b in blobs if b.name.endswith(".md")]
print(f"GCS profile files: {len(md_blobs):,}")
print(f"Expected:          {len(df_player_profiles):,}")
print(f"Local successful:  {(df_player_profiles['source'] == 'gemini').sum():,}")
print(f"Local data-only:   {(df_player_profiles['source'] == 'data_only').sum():,}")

GCS profile files: 24,270
Expected:          24,270
Local successful:  24,270
Local data-only:   0


In [ ]:
df_player_profiles['word_count'] = df_player_profiles['profile'].fillna('').apply(lambda s: len(s.split()))
df_player_profiles['char_count'] = df_player_profiles['profile'].fillna('').str.len()

print("Word count distribution (Gemini-sourced only):")
gemini_only = df_player_profiles[df_player_profiles['source'] == 'gemini']
print(gemini_only['word_count'].describe().to_string())

print(f"\nUnusually short (<150 words): {(gemini_only['word_count'] < 150).sum()}")
print(f"Unusually long (>400 words):  {(gemini_only['word_count'] > 400).sum()}")

Word count distribution (Gemini-sourced only):
count    24270.000000
mean       173.969345
std         47.297492
min         48.000000
25%        140.000000
50%        175.000000
75%        206.000000
max        470.000000

Unusually short (<150 words): 7513
Unusually long (>400 words):  14


In [ ]:
for pid in ["judgeaa01", "aaronha01", "ruthba01"]:
    blob = bucket.blob(f"{PROFILES_GCS_PREFIX}/{pid}.md")
    if blob.exists():
        content = blob.download_as_text()
        print(f"=== {pid} ({len(content)} chars) ===")
        print(content[:300] + ("..." if len(content) > 300 else ""))
        print()
    else:
        print(f"⚠️  {pid}.md not found in GCS")

=== judgeaa01 (1325 chars) ===
Aaron Judge is a prominent right fielder and the captain for the New York Yankees, known for his exceptional power-hitting and towering home runs.

Judge made his MLB debut in 2016 and quickly established himself as a force in the league, earning the American League Rookie of the Year award in 2017....

=== aaronha01 (1563 chars) ===
Hank Aaron, famously known as "Hammerin' Hank," was a prolific right fielder who spent 21 seasons with the Milwaukee/Atlanta Braves and his final two seasons with the Milwaukee Brewers before retiring in 1976. He is best known for his remarkable power-hitting, holding the MLB record for most career ...

=== ruthba01 (1279 chars) ===
George Herman "Babe" Ruth was an iconic American baseball player who spent 22 seasons in Major League Baseball from 1914 to 1935. Known as "the Bambino" and "the Sultan of Swat," he began his career as a star left-handed pitcher, but achieved his greatest fame as a power-hitting outfielder. He las

Check some short profiles

In [ ]:
short = df_player_profiles[(df_player_profiles['source'] == 'gemini') & (df_player_profiles['word_count'] < 100)].sample(3, random_state=42)
for _, r in short.iterrows():
    print(f"=== {r['playerID']} ({r['word_count']} words) ===")
    print(r['profile'])
    print()


=== battala01 (82 words) ===
Larry Battam was a third baseman who played briefly for the New York Giants in 1895. Born in Brooklyn in 1876, he was 19 years old when he made his Major League Baseball debut.

Battam's MLB career spanned a single season in 1895. He played in two games, accumulating four at-bats, one hit, and a career batting average of .250. He retired from Major League Baseball after his final game on September 30, 1895, though his minor league career continued until 1903.

=== norto01 (61 words) ===
Norton, identified as "norto01" in the Lahman Baseball Database, had a brief career spanning a single season in 1942. Playing for "NYC," Norton appeared in only one game. During this limited time, Norton registered no at-bats, hits, home runs, runs batted in, runs scored, or stolen bases. This player appears in the database with only a surname, suggesting incomplete historical records.

=== wilsoto01 (89 words) ===
Tom Wilson was a catcher who played briefly for the Washingt

And some long profiles

In [ ]:
long = df_player_profiles[df_player_profiles['word_count'] > 400][['playerID', 'name', 'word_count']]
print(long.sort_values('word_count', ascending=False).to_string(index=False))

 playerID              name  word_count
 datzje01         Jeff Datz         470
zambrca01   Carlos Zambrano         457
blancgr01     Gregor Blanco         434
gibbojo02      John Gibbons         432
mazeipa01   Patrick Mazeika         422
glanvdo01    Doug Glanville         420
brownan02      Andrew Brown         416
huntehe01       Herb Hunter         412
wellsve01      Vernon Wells         410
lopezal01          Al Lopez         410
guttedo01    Don Gutteridge         409
standja01  Jason Standridge         405
terwiwa01 Wayne Terwilliger         403
degotal01      Alex De Goti         402


In [ ]:
for pid in ["smith15", "perry01"]:
    blob = bucket.blob(f"{PROFILES_GCS_PREFIX}/{pid}.md")
    if blob.exists():
        print(f"=== {pid} ===")
        print(blob.download_as_text())
        print()

=== smith15 ===
Smith, identified as "smith15" in the Lahman Baseball Database, played one season in 1926. Primarily a pitcher, Smith appeared in one game for the team designated as DM. During this brief career, Smith recorded no at-bats, hits, home runs, runs batted in, runs, or stolen bases. On the mound, Smith finished with a win-loss record of 0-1 and registered no saves or strikeouts.

=== perry01 ===
Perry, identified as perry01 in the Lahman Baseball Database, played one season in 1944. Primarily a pitcher, Perry appeared in one game, pitching 9.0 innings with an ERA of 1.00, recording 7 strikeouts and a 0-1 win-loss record. He also had limited plate appearances, with 3 at-bats, 2 hits, 2 runs, and a .667 career batting average. His last recorded team was PS.



Crud. So some of the profiles are single names where Gemini essentially did a best guess. Not ideal.

In [ ]:
# Cross-reference profile word count against the actual data we passed Gemini
def lahman_data_richness(pid):
    """Score how much real data Lahman has for this player. Higher = richer."""
    score = 0
    if pid in batting_career:
        score += min(batting_career[pid]['games'] / 100, 10)  # cap contribution
    if pid in pitching_career:
        score += min(pitching_career[pid]['games'] / 100, 10)
    if pid in allstar_lookup:
        score += allstar_lookup[pid]['selections']
    if pid in awards_lookup:
        score += len(awards_lookup[pid])
    return score

df_player_profiles['lahman_richness'] = df_player_profiles['playerID'].apply(lahman_data_richness)

# The suspects: long profile (>250 words) but thin Lahman data (<2 score)
suspects = df_player_profiles[
    (df_player_profiles['word_count'] > 250) &
    (df_player_profiles['lahman_richness'] < 2)
].sort_values('word_count', ascending=False)

print(f"Suspect profiles (long output, thin Lahman data): {len(suspects)}")
print(suspects[['playerID', 'name', 'word_count', 'lahman_richness']].head(20).to_string(index=False))

Suspect profiles (long output, thin Lahman data): 472
 playerID             name  word_count  lahman_richness
 datzje01        Jeff Datz         470             0.07
gibbojo02     John Gibbons         432             0.18
mazeipa01  Patrick Mazeika         422             0.61
brownan02     Andrew Brown         416             1.44
huntehe01      Herb Hunter         412             0.39
standja01 Jason Standridge         405             1.60
degotal01     Alex De Goti         402             0.02
hessmmi01     Mike Hessman         391             1.09
howarch01     Chris Howard         383             0.22
gaineja01       Jay Gainer         375             0.23
garcimi02    Miguel Garcia         374             0.28
laforty01      Ty LaForest         369             0.52
coopeda01     David Cooper         365             0.72
birkbmi01    Mike Birkbeck         363             1.08
rickeda01    Dave Ricketts         362             1.30
  foxer01         Eric Fox         359            

In [ ]:
# Spot-check 5 random suspects from the middle of the distribution
sample = suspects.iloc[5:25].sample(5, random_state=42)
for _, r in sample.iterrows():
    blob = bucket.blob(f"{PROFILES_GCS_PREFIX}/{r['playerID']}.md")
    print(f"=== {r['playerID']} | {r['name']} | {r['word_count']} words | richness={r['lahman_richness']:.2f} ===")
    print(blob.download_as_text()[:600])
    print()

=== standja01 | Jason Standridge | 405 words | richness=1.60 ===
Jason Standridge was a right-handed pitcher who played in Major League Baseball for parts of seven seasons between 2001 and 2007, primarily known for his stints with the Tampa Bay Devil Rays and the Kansas City Royals. Born in Birmingham in 1978, Standridge was a starting pitcher and reliever throughout his career.

Standridge made his MLB debut on July 29, 2001. Over his career, he appeared in 80 games, pitching 127.3 innings with a 3-9 win-loss record and 80 strikeouts. His career ERA was 5.80. He concluded his MLB career with the Kansas City Royals, playing his final game on May 19, 2007.



=== montara01 | Rafael Montalvo | 333 words | richness=0.02 ===
Rafael Montalvo was a right-handed pitcher who made a solitary appearance for the Houston Astros in 1986. Born in Rio Piedras, Puerto Rico, in 1964, Montalvo's major league career was notably brief, lasting just a single inning. He is known for having one of the shorte

In [ ]:
# How many of the 24,270 are single-name Lahman entries?
single_name = people_df[people_df['name_first'].isna() | people_df['name_last'].isna()]
print(f"Single-name Lahman entries: {len(single_name)}")

# How many of those got long profiles (the ones at risk of conflation)?
single_name_pids = set(single_name['playerID'])
single_name_profiles = df_player_profiles[df_player_profiles['playerID'].isin(single_name_pids)]
print(f"Single-name profile word count: mean={single_name_profiles['word_count'].mean():.0f}, "
      f"median={single_name_profiles['word_count'].median():.0f}")
print(f"  Long ones (>250 words): {(single_name_profiles['word_count'] > 250).sum()}")

Single-name Lahman entries: 413
Single-name profile word count: mean=145, median=148
  Long ones (>250 words): 0


Let's see what we can do to fix the single name entries.

In [ ]:
def create_partial_name_prompt(player_row) -> str:
    """Prompt variant for Lahman entries with only a surname (or only a first name).
    Forbids writing about other players who share the partial name."""
    context = build_player_context(player_row)
    name = player_row.get("name_full") or player_row.get("playerID")

    return f"""You are writing a brief player profile for a baseball front-office reference document. The structured data below comes from the Lahman Baseball Database.

IMPORTANT: This Lahman entry has only a partial name (no first name, or no last name). This is common for 19th- and early 20th-century players whose full names were never fully recorded. There is almost certainly no information available on this specific individual via web search.

Do NOT write about other, more famous players who share this surname or partial name. Do NOT speculate about who this player might have been. Do NOT mention namesakes or note that other players exist with similar names.

{context}

=== INSTRUCTIONS ===
Write a 60-120 word factual profile, plain markdown. No heading. Single paragraph. Use only the structured data above. Acknowledge in tone that limited information exists for this player ("appears in Lahman as...", "played briefly in...", etc.). Do not editorialize.

Profile for {name}:"""

In [ ]:
def generate_partial_name_profile(player_row, retries: int = PlayerProfileConfig.MAX_RETRIES) -> dict:
    pid = player_row["playerID"]
    name = player_row.get("name_full") or pid
    prompt = create_partial_name_prompt(player_row)

    config = types.GenerateContentConfig(
        temperature=0.7,  # tighter than the main run — we want consistent factual output
        max_output_tokens=400,
        tools=[types.Tool(google_search=types.GoogleSearch())],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )

    for attempt in range(retries):
        try:
            response = get_client().models.generate_content(
                model=PlayerProfileConfig.MODEL_NAME,
                contents=prompt,
                config=config,
            )
            raw = response.text if hasattr(response, 'text') and response.text else None
            if not raw and hasattr(response, 'candidates') and response.candidates:
                parts = response.candidates[0].content.parts
                if parts and hasattr(parts[0], 'text'):
                    raw = parts[0].text

            if raw:
                profile = parse_profile_response(raw)
                if 50 <= len(profile.split()) <= 200:  # accept anything reasonable
                    return {"playerID": pid, "name": name, "profile": profile,
                            "source": "gemini_partial_name", "error": None}

            if attempt < retries - 1:
                time.sleep(PlayerProfileConfig.RETRY_DELAY)
                continue
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(PlayerProfileConfig.RETRY_DELAY * (attempt + 1))
                continue

    # Fallback: build from data only
    ctx = build_player_context(player_row)
    return {"playerID": pid, "name": name,
            "profile": build_data_only_profile(player_row, ctx),
            "source": "data_only", "error": "Partial-name regen failed"}

In [ ]:
# Identify the 67: single-name AND currently long
single_name_pids = set(people_df[
    people_df['name_first'].isna() | people_df['name_last'].isna()
]['playerID'])
long_single_name = df_player_profiles[
    (df_player_profiles['playerID'].isin(single_name_pids)) &
    (df_player_profiles['word_count'] > 250)
]
print(f"Regenerating {len(long_single_name)} partial-name profiles...")

regenerated = []
for _, prof_row in tqdm(long_single_name.iterrows(), total=len(long_single_name)):
    pid = prof_row['playerID']
    player_row = people_df[people_df['playerID'] == pid].iloc[0]
    new = generate_partial_name_profile(player_row)
    regenerated.append(new)

    # Update in-memory df
    idx = df_player_profiles[df_player_profiles['playerID'] == pid].index[0]
    df_player_profiles.at[idx, 'profile'] = new['profile']
    df_player_profiles.at[idx, 'source'] = new['source']
    df_player_profiles.at[idx, 'error'] = new['error']

    # Overwrite in GCS
    blob_path = f"{PROFILES_GCS_PREFIX}/{pid}.md"
    bucket.blob(blob_path).upload_from_string(new['profile'], content_type="text/markdown")

# Refresh word count
df_player_profiles['word_count'] = df_player_profiles['profile'].fillna('').apply(lambda s: len(s.split()))

# Save updated progress
df_player_profiles.to_csv(PlayerProfileConfig.PROGRESS_FILE, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

# Spot-check
print("\nSample regenerated:")
for new in regenerated[:3]:
    print(f"=== {new['playerID']} ({len(new['profile'].split())} words, source={new['source']}) ===")
    print(new['profile'])
    print()

Regenerating 0 partial-name profiles...


0it [00:00, ?it/s]


Sample regenerated:


## 12. Team profiles

Same pattern as section 11, applied to franchises rather than players. The chat agents need to answer "tell me about the Yankees" or "did the St. Louis Browns ever win a pennant" with the same quality as questions about Aaron Judge or Hank Aaron.

**Scope**: every distinct franchise in Lahman's `Teams` table — approximately 120, including all 30 current MLB franchises plus historical ones (Brooklyn Tip-Tops, St. Louis Browns, Cleveland Spiders, Federal League, etc.). Symmetry with the player pass: we profile everyone Lahman tracks, not just the current era.

**Active vs. defunct branching**: the prompt detects from the data whether a franchise has played in the last 5 years and adjusts tone accordingly. Active franchises get current-state framing ("the Yankees are an AL East team..."); defunct franchises get past-tense historical framing ("the Cleveland Spiders were a National League team that operated from 1887 to 1899...").

**Output**: markdown documents at `gs://class-demo/mlb-race-to-october/profiles/teams/{franchID}.md`. Filename uses `franchID` (e.g., `NYY.md`, `LAD.md`, `BRO.md`) which is stable across team relocations — Brooklyn Dodgers and Los Angeles Dodgers share `LAD`.

**What goes in a team profile**:
- Para 1: identity — current league/division (or historical league for defunct), city/cities, founding year, summary of franchise character
- Para 2: notable history — championships, era-defining players or seasons, key milestones
- Para 3 (active only, optional): current state — recent playoff trajectory, notable current players, last World Series

200-300 words, plain markdown, same prose style as player profiles.

**Currency caveat carried over**: Lahman is current through 2025. Active-franchise profiles will reflect 2025 standings and roster as the most recent grounded data; Search supplements with anything from the 2025-26 offseason or current 2026 season.

In [ ]:
class TeamProfileConfig:
    MODEL_NAME = 'gemini-2.5-flash'
    TEMPERATURE = 1.0
    MAX_OUTPUT_TOKENS = 1500
    MAX_RETRIES = 3
    RETRY_DELAY = 2
    MIN_PROFILE_LENGTH = 200
    MAX_PROFILE_LENGTH = 3500
    PROGRESS_FILE = '/tmp/mlb_team_profiles_progress.csv'
    ERROR_LOG = '/tmp/mlb_team_profiles_errors.csv'


# Determine the most recent year in Lahman to gate "active" classification
LAHMAN_LAST_YEAR = int(teams_df['yearID'].max())
ACTIVE_THRESHOLD = LAHMAN_LAST_YEAR - 4  # franchise played within last 5 years = active

print(f"Lahman most recent year: {LAHMAN_LAST_YEAR}")
print(f"Active threshold: franchise must have a row in {ACTIVE_THRESHOLD} or later")


def build_team_context(franch_id: str) -> tuple[str, bool]:
    """Assemble structured context for a franchise. Returns (context_string, is_active)."""
    franch_rows = teams_df[teams_df['franchID'] == franch_id].sort_values('yearID')
    if len(franch_rows) == 0:
        return "", False

    first_year = int(franch_rows['yearID'].min())
    last_year = int(franch_rows['yearID'].max())
    is_active = last_year >= ACTIVE_THRESHOLD

    # Most recent name and city/park (helpful for both active and defunct)
    last_row = franch_rows.iloc[-1]
    most_recent_name = last_row.get('team_name', franch_id)

    # Names this franchise used historically (in case of relocations / rebrands)
    historical_names = sorted(franch_rows['team_name'].dropna().unique().tolist())

    # League history
    leagues = sorted(franch_rows['lgID'].dropna().unique().tolist())

    # Division (for years that have one)
    divs_recent = franch_rows[franch_rows['divID'].notna()].tail(5)
    current_div = divs_recent['divID'].iloc[-1] if len(divs_recent) > 0 else None

    # Championships
    ws_wins = franch_rows[franch_rows['won_world_series'] == True]
    ws_years = sorted(ws_wins['yearID'].astype(int).tolist())
    pennants = franch_rows[franch_rows['won_league'] == True]
    pennant_years = sorted(pennants['yearID'].astype(int).tolist())

    # Recent record (last 5 years available)
    recent = franch_rows.tail(5)
    recent_records = []
    for _, row in recent.iterrows():
        # Pandas nullable BOOLs return <NA> instead of False; coerce explicitly.
        def _truthy(val):
            return val is True  # treats NA, None, False all as False

        won_ws = _truthy(row.get('won_world_series'))
        won_lg = _truthy(row.get('won_league'))
        won_div = _truthy(row.get('won_division'))
        won_wc = _truthy(row.get('won_wild_card'))

        rec = f"{int(row['yearID'])}: {int(row['wins'])}-{int(row['losses'])}"
        if won_ws:
            rec += " (won World Series)"
        elif won_lg:
            rec += " (won pennant)"
        elif won_div or won_wc:
            rec += " (made postseason)"

        recent_records.append(rec)

    # All-time totals
    total_wins = int(franch_rows['wins'].sum())
    total_losses = int(franch_rows['losses'].sum())
    total_seasons = len(franch_rows)

    # Build context
    lines = [f"=== FRANCHISE ===", f"Franchise ID: {franch_id}", f"Most recent name: {most_recent_name}"]

    if len(historical_names) > 1:
        lines.append(f"Historical names: {'; '.join(historical_names)}")

    lines.append(f"Operated: {first_year}–{last_year} ({total_seasons} seasons)")
    lines.append(f"Status: {'ACTIVE' if is_active else 'DEFUNCT'} (last season in Lahman: {last_year})")

    lines.append(f"\nLeague(s): {', '.join(leagues)}")
    if is_active and current_div:
        lines.append(f"Current division: {current_div}")

    lines.append(f"\nAll-time regular season: {total_wins}-{total_losses} ({total_wins/(total_wins+total_losses):.3f} pct)" if total_wins + total_losses > 0 else "")

    if ws_years:
        lines.append(f"\nWorld Series titles: {len(ws_years)} ({', '.join(str(y) for y in ws_years)})")
    if pennant_years:
        lines.append(f"Pennants (league championships): {len(pennant_years)}")

    if recent_records:
        lines.append(f"\nMost recent {len(recent_records)} seasons:")
        for r in recent_records:
            lines.append(f"  {r}")

    return "\n".join(lines), is_active


def create_team_prompt(franch_id: str) -> str:
    context, is_active = build_team_context(franch_id)

    if is_active:
        directive = """Use Google Search to add current-state context: 2026 season standing, key recent transactions, notable current players, recent injuries. Lahman is current through 2025; anything 2026 comes from Search.

Structure:
- Paragraph 1: Identity. Current league/division, city, summary of franchise character ("the Yankees are an AL East franchise based in New York, known for their championship pedigree and large payroll"). Present tense.
- Paragraph 2: Notable history. World Series wins, era-defining players or dynasties, key milestones. Past tense for historical claims.
- Paragraph 3 (optional): Current state. Recent playoff trajectory, current notable players, recent moves. Skip if Search doesn't surface anything substantive.

Tone: factual, front-office reference. No editorializing, no sentimentality."""
    else:
        directive = """This is a DEFUNCT franchise. Do not write in present tense. Do not speculate about a "current" team — there is none. Search may surface historical material but should not be used to fabricate current relevance.

Structure:
- Paragraph 1: Identity. Past-tense framing ("the Cleveland Spiders were a National League team based in Cleveland from 1887 to 1899"). League, city, era of operation, summary of franchise character.
- Paragraph 2: Notable history. Championships if any, notable players, why the franchise is remembered, why it folded if known.
- Optionally a third short paragraph for legacy/historical significance, only if Search surfaces something concrete.

Tone: historical reference, past tense throughout."""

    return f"""You are writing a brief team profile for a baseball front-office reference document. The structured data below comes from the Lahman Baseball Database.

{context}

=== INSTRUCTIONS ===
{directive}

Length: 200-300 words. Plain markdown. NO heading or title — start directly with prose.

Rules:
- Don't fabricate. If you don't know something, leave it out.
- Don't list year-by-year records. Reference key seasons by year only when narratively meaningful.
- Don't editorialize about Hall of Fame players unless they're in this franchise's history.
- Output the prose only — no JSON wrapper, no bullets.

Profile for franchise {franch_id}:"""


def generate_team_profile(franch_id: str, retries: int = TeamProfileConfig.MAX_RETRIES) -> dict:
    prompt = create_team_prompt(franch_id)
    context, is_active = build_team_context(franch_id)

    config = types.GenerateContentConfig(
        temperature=TeamProfileConfig.TEMPERATURE,
        max_output_tokens=TeamProfileConfig.MAX_OUTPUT_TOKENS,
        tools=[types.Tool(google_search=types.GoogleSearch())],
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )

    for attempt in range(retries):
        try:
            response = get_client().models.generate_content(
                model=TeamProfileConfig.MODEL_NAME,
                contents=prompt,
                config=config,
            )
            raw = None
            if hasattr(response, 'text') and response.text:
                raw = response.text
            elif hasattr(response, 'candidates') and response.candidates:
                parts = response.candidates[0].content.parts
                if parts and hasattr(parts[0], 'text'):
                    raw = parts[0].text

            if raw:
                profile = parse_profile_response(raw)
                if TeamProfileConfig.MIN_PROFILE_LENGTH <= len(profile) <= TeamProfileConfig.MAX_PROFILE_LENGTH:
                    return {"franchID": franch_id, "profile": profile,
                            "is_active": is_active, "source": "gemini", "error": None}
                if len(profile) > TeamProfileConfig.MAX_PROFILE_LENGTH:
                    profile = profile[:TeamProfileConfig.MAX_PROFILE_LENGTH] + "..."
                    return {"franchID": franch_id, "profile": profile,
                            "is_active": is_active, "source": "gemini", "error": None}

            if attempt < retries - 1:
                time.sleep(TeamProfileConfig.RETRY_DELAY)
                continue
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(TeamProfileConfig.RETRY_DELAY * (attempt + 1))
                continue

    return {"franchID": franch_id, "profile": f"Profile generation failed for franchise {franch_id}.",
            "is_active": is_active, "source": "data_only", "error": "Generation failed"}


# Identify all franchises
all_franch_ids = sorted(teams_df['franchID'].dropna().unique().tolist())
active_count = sum(1 for f in all_franch_ids if build_team_context(f)[1])
defunct_count = len(all_franch_ids) - active_count
print(f"\nTotal franchises in Lahman: {len(all_franch_ids)}")
print(f"  Active:  {active_count}")
print(f"  Defunct: {defunct_count}")

Lahman most recent year: 2025
Active threshold: franchise must have a row in 2021 or later

Total franchises in Lahman: 203
  Active:  30
  Defunct: 173


In [ ]:
# Test the prompt on one active and one defunct franchise before bulk run
test_franchises = ["NYY", "BRO", "CLV"]  # Yankees (active), Brooklyn Dodgers franchise (now LAD — careful, this is the historical Brooklyn-only entry), Cleveland Spiders (defunct)

# Note: depending on Lahman's encoding, BRO might map differently.
# Adjust to whatever's in the actual data.
for fid in test_franchises:
    if fid not in all_franch_ids:
        print(f"⚠️  {fid} not found in franchises\n")
        continue

    context, is_active = build_team_context(fid)
    print("=" * 70)
    print(f"TEST: {fid} ({'ACTIVE' if is_active else 'DEFUNCT'})")
    print("=" * 70)
    print(f"\nStructured context:")
    print(context)
    print()

    result = generate_team_profile(fid)
    profile = result["profile"]
    word_count = len(profile.split())

    print(f"--- Profile ({len(profile)} chars, ~{word_count} words, source={result['source']}) ---")
    print(profile)
    if result.get("error"):
        print(f"\nError: {result['error']}")
    print()

TEST: NYY (ACTIVE)

Structured context:
=== FRANCHISE ===
Franchise ID: NYY
Most recent name: New York Yankees
Historical names: Baltimore Orioles; New York Highlanders; New York Yankees
Operated: 1901–2025 (125 seasons)
Status: ACTIVE (last season in Lahman: 2025)

League(s): AL
Current division: E

All-time regular season: 10990-8369 (0.568 pct)

World Series titles: 27 (1923, 1927, 1928, 1932, 1936, 1937, 1938, 1939, 1941, 1943, 1947, 1949, 1950, 1951, 1952, 1953, 1956, 1958, 1961, 1962, 1977, 1978, 1996, 1998, 1999, 2000, 2009)
Pennants (league championships): 41

Most recent 5 seasons:
  2021: 92-70 (made postseason)
  2022: 99-63 (made postseason)
  2023: 82-80
  2024: 94-68 (won pennant)
  2025: 94-68 (made postseason)

--- Profile (1341 chars, ~218 words, source=gemini) ---
The New York Yankees are an active AL East franchise based in New York, known for their championship pedigree and large payroll. The team has been in operation for 125 seasons, from 1901–2025, and plays in t

In [ ]:
def rebuild_team_profiles_from_gcs() -> pd.DataFrame:
    """When v1 backup was restored by the prelude cell, skip Gemini and rebuild
    the in-memory dataframe by reading the GCS-restored profile content."""
    print("📦 Rebuilding df_team_profiles from GCS-restored content...")

    team_blobs = [b for b in client.list_blobs(GCS_BUCKET, prefix=f"{TEAMS_GCS_PREFIX}/")
                  if b.name.endswith(".md")]
    print(f"  Downloading {len(team_blobs)} restored team profiles...")

    rows = []
    for blob in tqdm(team_blobs, desc="Reading team profiles"):
        fid = blob.name.split("/")[-1].replace(".md", "")
        try:
            rows.append({"franchID": fid, "profile": blob.download_as_text()})
        except Exception:
            continue

    df = pd.DataFrame(rows)

    # Recompute is_active using the same rule as build_team_context:
    # franchise is active if its last Lahman year is within 4 of the most recent year
    LAHMAN_LAST_YEAR = int(teams_df['yearID'].max())
    franch_last_year = teams_df.groupby('franchID')['yearID'].max().to_dict()
    df['is_active'] = df['franchID'].apply(
        lambda fid: franch_last_year.get(fid, 0) >= (LAHMAN_LAST_YEAR - 4)
    )

    df['source'] = 'gemini'   # all v1 content was Gemini-generated
    df['error'] = None
    df['word_count'] = df['profile'].apply(lambda s: len(s.split()) if isinstance(s, str) else 0)

    print(f"\n  ✓ Rebuilt {len(df)} team profiles from GCS")
    return df


if SKIP_TEAM_GENERATION:
    print("⏭️  Section 12: Gemini generation skipped — v1 backup restored by prelude cell")
    df_team_profiles = rebuild_team_profiles_from_gcs()

    # Save into the progress file so subsequent reruns find it (idempotent)
    df_team_profiles.to_csv(TeamProfileConfig.PROGRESS_FILE, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

else:
    team_results = []
    print(f"Generating profiles for {len(all_franch_ids)} franchises...")

    for fid in tqdm(all_franch_ids):
        result = generate_team_profile(fid)
        team_results.append(result)
        if result.get("error"):
            log_player_error(fid, fid, result["error"])  # reuse the error logger

    df_team_profiles = pd.DataFrame(team_results)
    df_team_profiles['word_count'] = df_team_profiles['profile'].apply(lambda s: len(s.split()))

    # Save progress locally
    df_team_profiles.to_csv(TeamProfileConfig.PROGRESS_FILE, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

# Summary stats — runs in both branches
successes = (df_team_profiles['source'] == 'gemini').sum()
fallbacks = (df_team_profiles['source'] == 'data_only').sum() if 'data_only' in df_team_profiles['source'].values else 0

print(f"\nGenerated: {len(df_team_profiles)} franchise profiles")
print(f"  Successful: {successes}")
print(f"  Fallback:   {fallbacks}")
print(f"  Active:     {df_team_profiles['is_active'].sum()}")
print(f"  Defunct:    {(~df_team_profiles['is_active']).sum()}")
print(f"\nWord count: mean={df_team_profiles['word_count'].mean():.0f}, "
      f"median={df_team_profiles['word_count'].median():.0f}")

⏭️  Section 12: Gemini generation skipped — v1 backup restored by prelude cell
📦 Rebuilding df_team_profiles from GCS-restored content...


Reading team profiles:   0%|          | 0/203 [00:00<?, ?it/s]


  ✓ Rebuilt 203 team profiles from GCS

Generated: 203 franchise profiles
  Successful: 203
  Fallback:   0
  Active:     30
  Defunct:    173

Word count: mean=207, median=204


In [ ]:
TEAMS_GCS_PREFIX = f"{GCS_PREFIX}/profiles/teams"

print(f"Uploading {len(df_team_profiles)} team profiles to gs://{GCS_BUCKET}/{TEAMS_GCS_PREFIX}/")

uploaded = 0
errors = 0
for _, row in df_team_profiles.iterrows():
    if not isinstance(row.get("profile"), str) or not row["profile"]:
        errors += 1
        continue
    blob_path = f"{TEAMS_GCS_PREFIX}/{row['franchID']}.md"
    try:
        bucket.blob(blob_path).upload_from_string(row["profile"], content_type="text/markdown")
        uploaded += 1
    except Exception as e:
        errors += 1

print(f"✓ Uploaded: {uploaded}")
print(f"✗ Errors:   {errors}")

# Verify
team_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{TEAMS_GCS_PREFIX}/"))
print(f"\nGCS team profile files: {len([b for b in team_blobs if b.name.endswith('.md')])}")

Uploading 203 team profiles to gs://class-demo/mlb-race-to-october/profiles/teams/
✓ Uploaded: 203
✗ Errors:   0

GCS team profile files: 203


Quick validation check

In [ ]:
# 1. GCS object count matches
team_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{TEAMS_GCS_PREFIX}/"))
md_blobs = [b for b in team_blobs if b.name.endswith(".md")]
print(f"GCS team profile files: {len(md_blobs)}")
print(f"Expected:               {len(df_team_profiles)}")
print(f"Active / Defunct:       {df_team_profiles['is_active'].sum()} / {(~df_team_profiles['is_active']).sum()}")

# 2. Length sanity — anything truly degenerate?
print(f"\nWord count: min={df_team_profiles['word_count'].min()}, "
      f"max={df_team_profiles['word_count'].max()}, "
      f"median={df_team_profiles['word_count'].median():.0f}")
print(f"  Under 100 words: {(df_team_profiles['word_count'] < 100).sum()}")
print(f"  Over 400 words:  {(df_team_profiles['word_count'] > 400).sum()}")

# 3. Spot-check three franchises by fetching from GCS
for fid in ["LAD", "BOS", "FLA"]:  # active dynasty, active long-history, defunct-ish (Marlins were renamed)
    blob = bucket.blob(f"{TEAMS_GCS_PREFIX}/{fid}.md")
    if blob.exists():
        content = blob.download_as_text()
        word_count = len(content.split())
        print(f"\n=== {fid} ({word_count} words) ===")
        print(content[:400] + ("..." if len(content) > 400 else ""))
    else:
        print(f"\n⚠️  {fid}.md not found")

GCS team profile files: 203
Expected:               203
Active / Defunct:       30 / 173

Word count: min=89, max=476, median=204
  Under 100 words: 2
  Over 400 words:  4

=== LAD (476 words) ===
The Los Angeles Dodgers are an active National League franchise based in Los Angeles, California, currently competing in the NL West division. Known for their consistent success and strong fan base, the Dodgers have been a prominent team in Major League Baseball for decades.

The Dodgers franchise boasts a rich history dating back to 1884, operating for 142 seasons. They have secured nine World Se...

=== BOS (200 words) ===
The Boston Red Sox are an active Major League Baseball franchise based in Boston, Massachusetts, competing in the American League East Division. They are known for their long history and dedicated fanbase, consistently fielding competitive teams within the AL. The team's all-time regular season record stands at 10044 wins and 9336 losses, a winning percentage of .518.

Th

## 13. Update cadence and the API-backfill decision

Documenting the data-freshness architecture so future-you (or whoever runs the next refresh) understands what gets updated when, and why we made the architectural choices we did.

### Lahman refresh

Lahman is now maintained by SABR (https://sabr.org/lahman-database/) and releases annually, typically in early January after the prior MLB season concludes. The release we're using covers data through the 2025 season.

**Annual refresh procedure** (instructor-only, before each event year):

1. Download the latest `lahman_csv` archive from SABR's distribution page (Box.com, manual download)
2. Extract and upload the 9 tables we use to `gs://class-demo/mlb-race-to-october/lahman-source/` (overwriting the previous year's CSVs)
3. Rerun this notebook end-to-end. The cells download from `lahman-source/`, transform to Parquet, and write to `lahman/`. The Task 1 SQL is regenerated automatically.
4. Validate the section 8 SQL still loads cleanly against a throwaway dataset
5. Validate the section 9 BQML training still produces an AUC in the 0.75–0.99 range
6. Regenerate profiles (sections 11–12) — this is the slow part, ~90 minutes for players plus ~25 minutes for teams. Profiles capture the prior season's results and are the most date-sensitive output.

A skipped refresh year is survivable. Profiles will be one season stale, the BQML model will be one season stale, and the chat agents will give answers that are slightly behind the current standings — but nothing breaks.

### The API-backfill decision

The original lab spec asked us to evaluate three options:
- **(a)** Live API backfill in the lab — students see the API used both for backfill and live stats
- **(b)** Instructor-only API backfill in this notebook
- **(c)** No backfill — accept that BigQuery is "as of last Lahman release"

**Decision: option (c).** Here's why.

The freshness gap turned out to be smaller than the original spec assumed. SABR's January 2026 release covers data through the 2025 regular season. The lab will run during the 2026 season, so at any point in 2026 the BigQuery data is at most a few months stale (current season in progress) and at minimum a few weeks stale (right after refresh).

A few months of staleness doesn't justify the architectural cost. The lab's pedagogical hinge in Task 3 is the clean dichotomy:

> **BigQuery is for historical / analytical / computational queries**
> **The MLB Stats API is for live / current / real-time queries**

Backfilling BigQuery from the API blurs that line. The ADK agent's "live stats" tool would have to explain why it exists when BigQuery already covers up to "yesterday." Students would reasonably ask "why are we using two data sources for current data?" and the answer "because we ran a backfill script" is unsatisfying compared to the cleaner story.

Option (c) preserves the dichotomy. Profile narratives use Google Search grounding to surface 2026 in-season context (current team, recent injuries, milestone games) where it matters most — at the chat-agent layer, not at the warehouse layer.

### What's live vs. what's annual

| Data | Source | Refresh cadence |
|---|---|---|
| Historical stats (1871–2025) | Lahman → BigQuery | Annual, instructor-driven |
| Rules | MLB rulebook PDF | Annual, when MLB releases new edition |
| Player and team profile narratives | Lahman + Gemini + Search | Annual, regenerated with Lahman refresh |
| Current season stats | MLB Stats API (live) | Real-time, no caching |
| Current rosters, injuries, transactions | MLB Stats API + chat-agent Search grounding | Real-time |

The split is clean: anything that gets refreshed annually lives in GCS or BigQuery. Anything that requires real-time accuracy lives in the API and the agent's Search grounding.

## 14. Final QC — all deliverables staged

This is the closing audit. Walks every output prefix, confirms presence and counts, and prints a deliverable-by-deliverable readout that should match the lab spec from the conversation header.

In [ ]:
# Walk every output we should have produced and confirm it's where the lab expects it.

print("=" * 70)
print("MLB RACE TO OCTOBER — DATA FOUNDATION FINAL QC")
print("=" * 70)
print(f"\nBucket:    gs://{GCS_BUCKET}/")
print(f"Prefix:    {GCS_PREFIX}/")
print(f"Run date:  {datetime.now().strftime('%Y-%m-%d %H:%M %Z')}")

# ---------- Deliverable 1: Lahman in GCS ----------
print("\n" + "─" * 70)
print("DELIVERABLE 1: Lahman staged in GCS (Parquet, agent-friendly schema)")
print("─" * 70)

source_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/lahman-source/"))
parquet_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/lahman/"))

source_csvs = [b for b in source_blobs if b.name.endswith(".csv")]
parquet_files = [b for b in parquet_blobs if b.name.endswith(".parquet")]

print(f"  Source CSVs (lahman-source/): {len(source_csvs)} files")
print(f"  Parquet  (lahman/):           {len(parquet_files)} files, "
      f"{sum(b.size for b in parquet_files) / 1024 / 1024:.2f} MB total")
print(f"  Status: {'✓' if len(parquet_files) == len(LAHMAN_TABLES) else '✗'} "
      f"all {len(LAHMAN_TABLES)} target tables present")

# ---------- Deliverable 2: Rulebook ----------
print("\n" + "─" * 70)
print("DELIVERABLE 2: MLB Rulebook PDF")
print("─" * 70)

rulebook_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/rulebook/"))
rulebook_pdfs = [b for b in rulebook_blobs if b.name.endswith(".pdf")]
if rulebook_pdfs:
    rb = rulebook_pdfs[0]
    print(f"  File: {rb.name}")
    print(f"  Size: {rb.size / 1024 / 1024:.2f} MB")
    print(f"  Status: ✓ staged")
else:
    print(f"  Status: ✗ not found")

# ---------- Deliverable 3: Profiles ----------
print("\n" + "─" * 70)
print("DELIVERABLE 3: Player and team profiles (markdown)")
print("─" * 70)

player_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/profiles/players/"))
team_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/profiles/teams/"))
player_mds = [b for b in player_blobs if b.name.endswith(".md")]
team_mds = [b for b in team_blobs if b.name.endswith(".md")]

print(f"  Players: {len(player_mds):,} files, "
      f"{sum(b.size for b in player_mds) / 1024 / 1024:.2f} MB total")
print(f"  Teams:   {len(team_mds)} files, "
      f"{sum(b.size for b in team_mds) / 1024:.1f} KB total")
print(f"  Total:   {len(player_mds) + len(team_mds):,} markdown files")

# ---------- Deliverable 4: Task 1 SQL ----------
print("\n" + "─" * 70)
print("DELIVERABLE 4: Task 1 SQL (LOAD + ALTER) — students execute in lab")
print("─" * 70)

schema_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/schema/"))
sql_files = [b for b in schema_blobs if b.name.endswith(".sql")]
for sf in sql_files:
    print(f"  {sf.name.split('/')[-1]}: {sf.size:,} bytes")
print(f"  Status: {'✓' if len(sql_files) >= 2 else '✗'} "
      f"({len(sql_files)} SQL file(s) staged)")

# ---------- Deliverable 5: BQML model SQL ----------
# Same prefix as #4; called out separately because it's a distinct deliverable.
bqml_files = [b for b in sql_files if "playoff" in b.name or "bqml" in b.name or "model" in b.name]
print(f"\n  BQML model SQL: {len(bqml_files)} file(s)")
print(f"  Validated AUC:  ~0.987 (in expected 0.75–0.99 range)")
print(f"  Status: {'✓' if bqml_files else '✗'} captured for Task 2")

# ---------- Deliverable 6: MLB Stats API ----------
print("\n" + "─" * 70)
print("DELIVERABLE 6: MLB Stats API validation")
print("─" * 70)
print(f"  Status: ✓ skipped — validated in earlier lab iteration, no changes needed")

# ---------- Deliverable 7: Update cadence + backfill writeup ----------
print("\n" + "─" * 70)
print("DELIVERABLE 7: Update cadence + API backfill decision")
print("─" * 70)
print(f"  Decision: option (c) — no API backfill; BigQuery as-of last Lahman release")
print(f"  Documented: section 13 markdown")
print(f"  Status: ✓ rationale captured in notebook")

# ---------- Final summary ----------
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
total_objects = (
    len(source_csvs) + len(parquet_files) + len(rulebook_pdfs)
    + len(player_mds) + len(team_mds) + len(sql_files)
)
total_size_mb = (
    sum(b.size for b in source_csvs)
    + sum(b.size for b in parquet_files)
    + sum(b.size for b in rulebook_pdfs)
    + sum(b.size for b in player_mds)
    + sum(b.size for b in team_mds)
    + sum(b.size for b in sql_files)
) / 1024 / 1024

print(f"  Total objects in gs://{GCS_BUCKET}/{GCS_PREFIX}/: {total_objects:,}")
print(f"  Total size:                                     {total_size_mb:.2f} MB")
print(f"  Lab data foundation: COMPLETE")
print()
print(f"  Next: lab content authoring uses these staged assets")
print(f"        - Task 1 embeds {sql_files[0].name.split('/')[-1] if sql_files else 'load_lahman.sql'}")
print(f"        - Task 2 embeds the BQML training SQL")
print(f"        - Task 3 ADK agent reads from BigQuery + MLB Stats API")
print(f"        - Gemini Enterprise data store ingests profiles/ + rulebook/")
print(f"        - CX Agent Studio shares the same data store for fan-facing chat")
print("=" * 70)

MLB RACE TO OCTOBER — DATA FOUNDATION FINAL QC

Bucket:    gs://class-demo/
Prefix:    mlb-race-to-october/
Run date:  2026-05-06 21:52 

──────────────────────────────────────────────────────────────────────
DELIVERABLE 1: Lahman staged in GCS (Parquet, agent-friendly schema)
──────────────────────────────────────────────────────────────────────
  Source CSVs (lahman-source/): 27 files
  Parquet  (lahman/):           9 files, 7.14 MB total
  Status: ✓ all 9 target tables present

──────────────────────────────────────────────────────────────────────
DELIVERABLE 2: MLB Rulebook PDF
──────────────────────────────────────────────────────────────────────
  File: mlb-race-to-october/rulebook/2026-official-baseball-rules.pdf
  Size: 1.26 MB
  Status: ✓ staged

──────────────────────────────────────────────────────────────────────
DELIVERABLE 3: Player and team profiles (markdown)
──────────────────────────────────────────────────────────────────────
  Players: 24,270 files, 23.56 MB total
 

## 15. Profile audit and remediation

After the central planning conversation reviewed sample profiles, four issues surfaced that need investigation before the data foundation is genuinely complete:

1. **Duplication.** The LAD profile contained its full content twice. Possibly a one-off generation glitch, possibly systemic — needs programmatic check across all 203 team profiles.

2. **Historical-arc coverage gaps.** Relocated franchises (LAD, ATL, SFG observed) frame their entire history through the current city, omitting prior-city eras and era-defining figures (no Brooklyn / Jackie Robinson in LAD, no Boston-Milwaukee / Hank Aaron in ATL). Fan-facing queries about "the Brooklyn Dodgers" or "Hank Aaron's Milwaukee era" return either nothing or wrong-city framing.

3. **Current-state staleness.** Profiles dedicate 40-60% of content to "as of May 6, 2026" material — current standings, injuries, roster moves. This information will be wrong by July, badly wrong by September, and the lab runs through fall 2026 and possibly into 2027. One profile cited "Spencer Strider's October 2022 contract" as recent, suggesting Search grounding pulled stale material the model treated as current.

4. **UTF-8 encoding glitches.** Confirmed across all three sampled team profiles: `IbÃ¡Ã±ez`, `KikÃ© HernÃ¡ndez`, `AcuÃ±a`. UTF-8 bytes being decoded as Latin-1 somewhere in the pipeline.

### The architectural decision driving the fixes

Profiles are **historical and biographical reference documents**, not current-state snapshots. The MLB Stats API exists for current-state queries — current standings, rosters, injuries, in-season transactions. The chat agents that ground on profiles can pair them with API calls at query time. Profile content that mixes the two surfaces collapses the architectural wall the lab is built around.

This means the regeneration prompt must:
- **Strip all in-season / current-state content** — no current standings, no current injury status, no recent transactions, no "as of {date}" framing
- **Player profiles may state current team affiliation** as a slow-changing biographical fact, but no in-season highlights or injury status
- **Team profiles cover the full historical arc** — for relocated franchises, name prior cities and era-defining figures explicitly
- **Fix encoding** at the write layer so Spanish names and other non-ASCII content round-trip correctly

### Plan

1. Audit duplication, encoding, and historical-arc coverage programmatically (not anecdotally) — measure before deciding regeneration scope
2. Sample player profiles for the same issues
3. Make a documented decision on regeneration scope based on findings
4. Revise prompts (team profile prompt gets the historical-arc requirement; both prompts strip current-state)
5. Targeted regeneration of affected files only
6. Re-run final QC

Audit findings drive the regeneration scope, not anecdote.

In [ ]:
import hashlib
from collections import Counter

def find_duplicated_content(text: str, min_chunk_words: int = 30) -> dict:
    """Detect if a document contains substantial repeated content.

    Returns a dict with diagnostic info. The simplest signal is: split the
    document in half and check if the halves are very similar.
    """
    words = text.split()
    if len(words) < min_chunk_words * 2:
        return {"duplicated": False, "reason": "too short to check"}

    # Simple half-vs-half comparison
    half = len(words) // 2
    first_half = " ".join(words[:half])
    second_half = " ".join(words[half:])

    # Count overlapping 5-grams
    def ngrams(s, n=5):
        toks = s.split()
        return set(" ".join(toks[i:i+n]) for i in range(len(toks) - n + 1))

    g1 = ngrams(first_half)
    g2 = ngrams(second_half)
    if not g1 or not g2:
        return {"duplicated": False, "reason": "ngram extraction empty"}

    overlap = len(g1 & g2) / min(len(g1), len(g2))
    return {
        "duplicated": overlap > 0.3,  # >30% 5-gram overlap = strong duplication signal
        "overlap_ratio": round(overlap, 3),
        "word_count": len(words),
    }


# Audit all team profiles
print("Checking 203 team profiles for duplication...")
team_dupes = []
for blob in client.list_blobs(GCS_BUCKET, prefix=f"{TEAMS_GCS_PREFIX}/"):
    if not blob.name.endswith(".md"):
        continue
    fid = blob.name.split("/")[-1].replace(".md", "")
    content = blob.download_as_text()
    result = find_duplicated_content(content)
    if result["duplicated"]:
        team_dupes.append({
            "franchID": fid,
            "overlap": result["overlap_ratio"],
            "words": result["word_count"],
        })

print(f"\nTeam profiles with duplication signal: {len(team_dupes)} / 203")
if team_dupes:
    for d in sorted(team_dupes, key=lambda x: -x["overlap"]):
        print(f"  {d['franchID']:6s}  overlap={d['overlap']:.2f}  words={d['words']}")

# Sample 100 random player profiles for the same check
import random
random.seed(42)
all_player_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{PROFILES_GCS_PREFIX}/"))
all_player_blobs = [b for b in all_player_blobs if b.name.endswith(".md")]
sample_blobs = random.sample(all_player_blobs, 100)

print(f"\nSpot-checking 100 random player profiles for duplication...")
player_dupes = []
for blob in sample_blobs:
    pid = blob.name.split("/")[-1].replace(".md", "")
    content = blob.download_as_text()
    result = find_duplicated_content(content)
    if result["duplicated"]:
        player_dupes.append({"playerID": pid, "overlap": result["overlap_ratio"]})

print(f"Player profiles with duplication signal (sample of 100): {len(player_dupes)}")
if player_dupes:
    for d in player_dupes:
        print(f"  {d['playerID']}  overlap={d['overlap']:.2f}")

Checking 203 team profiles for duplication...

Team profiles with duplication signal: 4 / 203
  MIL     overlap=0.72  words=290
  BAL     overlap=0.69  words=408
  LAD     overlap=0.60  words=476
  PHI     overlap=0.44  words=422

Spot-checking 100 random player profiles for duplication...
Player profiles with duplication signal (sample of 100): 1
  sharrge01  overlap=0.55


In [ ]:
# Two encoding signals to look for:
#   - The "Ã" sequence — the canonical mojibake artifact when UTF-8 bytes are decoded as Latin-1
#   - At the bytes level: properly-encoded UTF-8 should produce 0xC3 0xA1 for á; mojibake produces 0xC3 0x83 0xC2 0xA1

print("Auditing 203 team profiles for UTF-8 encoding issues...")
team_encoding_bad = []
for blob in client.list_blobs(GCS_BUCKET, prefix=f"{TEAMS_GCS_PREFIX}/"):
    if not blob.name.endswith(".md"):
        continue
    fid = blob.name.split("/")[-1].replace(".md", "")

    # Read as text (default UTF-8) and look for the Ã artifact
    text = blob.download_as_text()
    if "Ã" in text:
        # Count occurrences for severity
        team_encoding_bad.append({
            "franchID": fid,
            "occurrences": text.count("Ã"),
        })

print(f"\nTeam profiles showing 'Ã' mojibake: {len(team_encoding_bad)} / 203")
for d in sorted(team_encoding_bad, key=lambda x: -x["occurrences"])[:10]:
    print(f"  {d['franchID']:6s}  occurrences={d['occurrences']}")

# Same check on the player sample (reuse the 100 from cell A)
print(f"\nPlayer profiles showing 'Ã' mojibake (sample of 100):")
player_encoding_bad = []
for blob in sample_blobs:
    pid = blob.name.split("/")[-1].replace(".md", "")
    text = blob.download_as_text()
    if "Ã" in text:
        player_encoding_bad.append({"playerID": pid, "occurrences": text.count("Ã")})

print(f"  Affected: {len(player_encoding_bad)} / 100")
for d in sorted(player_encoding_bad, key=lambda x: -x["occurrences"])[:10]:
    print(f"  {d['playerID']:12s}  occurrences={d['occurrences']}")

# Diagnostic: pull bytes from a known-affected file (LAD) to determine
# whether the corruption is in the file content itself or only in the read path
print(f"\nByte-level diagnostic on LAD.md:")
lad_bytes = bucket.blob(f"{TEAMS_GCS_PREFIX}/LAD.md").download_as_bytes()
# Look for either properly-encoded UTF-8 (0xC3 0xA1 for á) or mojibake (0xC3 0x83 0xC2 0xA1)
proper_a_acute = lad_bytes.count(b'\xc3\xa1')
mojibake_a_acute = lad_bytes.count(b'\xc3\x83\xc2\xa1')
print(f"  Proper UTF-8 'á' bytes (\\xc3\\xa1):       {proper_a_acute}")
print(f"  Mojibake 'á' bytes (\\xc3\\x83\\xc2\\xa1):  {mojibake_a_acute}")
print(f"  → Corruption is in the file" if mojibake_a_acute > 0 else
      f"  → File bytes are clean; mojibake was a read-side artifact")

Auditing 203 team profiles for UTF-8 encoding issues...

Team profiles showing 'Ã' mojibake: 0 / 203

Player profiles showing 'Ã' mojibake (sample of 100):
  Affected: 0 / 100

Byte-level diagnostic on LAD.md:
  Proper UTF-8 'á' bytes (\xc3\xa1):       2
  Mojibake 'á' bytes (\xc3\x83\xc2\xa1):  0
  → File bytes are clean; mojibake was a read-side artifact


In [ ]:
# Identify franchises that have lived in multiple cities by checking team_name history
relocated = []
for fid in teams_df["franchID"].dropna().unique():
    franch_rows = teams_df[teams_df["franchID"] == fid]
    names = franch_rows["team_name"].dropna().unique()
    # Extract distinct city tokens — heuristic: first word(s) of team name
    cities = set()
    for name in names:
        # Most team names are "{City} {Mascot}" — take the first word as city
        first_word = str(name).split()[0]
        cities.add(first_word)
    if len(cities) >= 2:  # franchise has had 2+ city names
        relocated.append({
            "franchID": fid,
            "cities": sorted(cities),
            "names": sorted(names.tolist()),
        })

print(f"Found {len(relocated)} franchises with multiple city names in their history")
print(f"(Sampling those with the most distinct cities for audit)\n")

# Sort by interesting-ness — franchises with most city changes
relocated_sorted = sorted(relocated, key=lambda x: -len(x["cities"]))
to_check = relocated_sorted[:20]  # top 20 most-relocated

# For each, check whether the profile mentions each prior city
print(f"{'franchID':10s} {'cities':50s} {'mentioned':12s} {'missing'}")
print("-" * 100)
arc_issues = []
for r in to_check:
    fid = r["franchID"]
    blob = bucket.blob(f"{TEAMS_GCS_PREFIX}/{fid}.md")
    if not blob.exists():
        continue
    content = blob.download_as_text().lower()
    mentioned = [c for c in r["cities"] if c.lower() in content]
    missing = [c for c in r["cities"] if c.lower() not in content]

    cities_str = ", ".join(r["cities"])[:48]
    mention_str = f"{len(mentioned)}/{len(r['cities'])}"
    missing_str = ", ".join(missing) if missing else "(none)"
    print(f"{fid:10s} {cities_str:50s} {mention_str:12s} {missing_str}")

    if missing:
        arc_issues.append({"franchID": fid, "missing_cities": missing, "all_cities": r["cities"]})

print(f"\nFranchises with missing prior-city mentions: {len(arc_issues)} / {len(to_check)}")

Found 27 franchises with multiple city names in their history
(Sampling those with the most distinct cities for audit)

franchID   cities                                             mentioned    missing
----------------------------------------------------------------------------------------------------
BEG        Baltimore, Cleveland, Columbus, Nashville, Washi   4/5          Cleveland
OAK        Athletics, Kansas, Oakland, Philadelphia           3/4          Kansas
ATL        Atlanta, Boston, Milwaukee                         1/3          Boston, Milwaukee
BAL        Baltimore, Milwaukee, St.                          3/3          (none)
PC         Pittsburgh, Toledo, Toledo-Indianapolis            3/3          (none)
SNH        Harrisburg-St., Indianapolis, St.                  3/3          (none)
IC         Cincinnati, Cincinnati-Indianapolis, Indianapoli   2/3          Cincinnati-Indianapolis
ANA        Anaheim, California, Los                           3/3          (none)
PIT      

In [ ]:
# Sample players across era buckets to check for staleness / current-state contamination
# Era buckets: 19th c (pre-1900 debut), Dead-ball (1900-1919), Live-ball (1920-1945),
# Integration era (1946-1968), Free-agency era (1969-1994), Modern (1995-2015), Active (2016-2025)

people_with_debut = people_df[people_df["debut_date"].notna()].copy()
people_with_debut["debut_year"] = pd.to_datetime(people_with_debut["debut_date"]).dt.year

eras = {
    "19th c. (pre-1900)":    people_with_debut[people_with_debut["debut_year"] < 1900],
    "Dead-ball (1900-1919)": people_with_debut[(people_with_debut["debut_year"] >= 1900) & (people_with_debut["debut_year"] < 1920)],
    "Live-ball (1920-1945)": people_with_debut[(people_with_debut["debut_year"] >= 1920) & (people_with_debut["debut_year"] < 1946)],
    "Integration (1946-68)": people_with_debut[(people_with_debut["debut_year"] >= 1946) & (people_with_debut["debut_year"] < 1969)],
    "Free agency (1969-94)": people_with_debut[(people_with_debut["debut_year"] >= 1969) & (people_with_debut["debut_year"] < 1995)],
    "Modern (1995-2015)":    people_with_debut[(people_with_debut["debut_year"] >= 1995) & (people_with_debut["debut_year"] < 2016)],
    "Active (2016-2025)":    people_with_debut[people_with_debut["debut_year"] >= 2016],
}

# Sample 3 players per era
random.seed(7)
sample_players = []
for era, df in eras.items():
    if len(df) >= 3:
        picks = df.sample(3, random_state=7)
        for _, row in picks.iterrows():
            sample_players.append({"era": era, "playerID": row["playerID"], "name": row["name_full"]})

print(f"Auditing {len(sample_players)} sampled players across 7 eras\n")

# Look for in-season / current-state contamination markers
staleness_markers = [
    "as of may", "as of april", "as of march", "as of february",
    "as of 2026", "in 2026", "currently leading", "currently",
    "this season", "this week", "this month",
    "is currently on the injured list", "injured list",
    "is leading", "ranks first",
]

problems = []
for sp in sample_players:
    blob = bucket.blob(f"{PROFILES_GCS_PREFIX}/{sp['playerID']}.md")
    if not blob.exists():
        continue
    content = blob.download_as_text()
    content_lower = content.lower()

    # Check for current-state markers
    found_markers = [m for m in staleness_markers if m in content_lower]

    # Check for encoding issues
    encoding_issue = "Ã" in content

    # Check for duplication
    dup_result = find_duplicated_content(content)

    if found_markers or encoding_issue or dup_result["duplicated"]:
        problems.append({
            "era": sp["era"],
            "playerID": sp["playerID"],
            "name": sp["name"],
            "current_state_markers": found_markers,
            "encoding_issue": encoding_issue,
            "duplicated": dup_result["duplicated"],
        })

print(f"Players with at least one issue: {len(problems)} / {len(sample_players)}\n")
for p in problems:
    flags = []
    if p["current_state_markers"]:
        flags.append(f"current-state ({len(p['current_state_markers'])} markers)")
    if p["encoding_issue"]:
        flags.append("encoding")
    if p["duplicated"]:
        flags.append("duplicated")
    print(f"  [{p['era']:22s}] {p['playerID']:12s} {p['name']:30s} → {', '.join(flags)}")

Auditing 21 sampled players across 7 eras

Players with at least one issue: 4 / 21

  [Modern (1995-2015)    ] severat01    Atahualpa Severino             → current-state (2 markers)
  [Active (2016-2025)    ] dominse01    Seranthony Dominguez           → current-state (1 markers)
  [Active (2016-2025)    ] marloca01    Cade Marlowe                   → current-state (3 markers)
  [Active (2016-2025)    ] azocajo01    Jose Azocar                    → current-state (1 markers)


In [ ]:
print("Running full-population dedup check on all player profiles (in-memory)...")
print(f"Total player profiles to check: {len(df_player_profiles):,}\n")

# Apply the dedup heuristic to each profile in the dataframe
def check_profile(text):
    if not isinstance(text, str):
        return None
    return find_duplicated_content(text)

# Add columns for the audit
audit = df_player_profiles[['playerID', 'name', 'profile']].copy()
audit['dup_result'] = audit['profile'].apply(check_profile)
audit['duplicated'] = audit['dup_result'].apply(lambda r: r and r.get('duplicated', False))
audit['overlap'] = audit['dup_result'].apply(lambda r: r.get('overlap_ratio', 0) if r else 0)
audit['words'] = audit['dup_result'].apply(lambda r: r.get('word_count', 0) if r else 0)

duplicate_players = audit[audit['duplicated']].sort_values('overlap', ascending=False)
print(f"=== Player profiles with duplication signal ===")
print(f"Total: {len(duplicate_players):,} of {len(df_player_profiles):,} "
      f"({len(duplicate_players)/len(df_player_profiles)*100:.2f}%)")
print(f"\nTop 20 by overlap:")
for _, r in duplicate_players.head(20).iterrows():
    print(f"  {r['playerID']:14s} {r['name']:30s} overlap={r['overlap']:.2f}  words={r['words']}")

# Save the playerIDs that need regen for the regen cells later
DUPLICATE_PLAYER_IDS = set(duplicate_players['playerID'].tolist())
print(f"\nStashed {len(DUPLICATE_PLAYER_IDS)} player IDs for targeted regeneration.")

Running full-population dedup check on all player profiles (in-memory)...
Total player profiles to check: 24,270

=== Player profiles with duplication signal ===
Total: 110 of 24,270 (0.45%)

Top 20 by overlap:
  ciasda01       Darryl Cias                    overlap=0.99  words=325
  finneha01      Joseph Finneran                overlap=0.99  words=261
  jacobme01      Merwin Jacobson                overlap=0.99  words=239
  kingle01       Lee King                       overlap=0.99  words=197
  armstbo01      Robert Armstrong               overlap=0.99  words=153
  brittza01      Zack Britton                   overlap=0.93  words=369
  griffke02      Ken Griffey                    overlap=0.92  words=399
  primra01       Ray Prim                       overlap=0.92  words=283
  caskecr01      Craig Caskey                   overlap=0.90  words=229
  snydere01      Emanuel Snyder                 overlap=0.90  words=220
  drewst01       Stephen Drew                   overlap=0.89  words=3

In [ ]:
# Hand-curated list of the 30 current MLB franchises that have meaningfully
# played under prior-city identities. franchID is the stable identifier.

CURRENT_FRANCHISES_WITH_HISTORY = {
    "LAD": {
        "current": "Los Angeles Dodgers",
        "prior_eras": ["Brooklyn"],
        "era_defining_figures": ["Jackie Robinson", "Sandy Koufax"],
    },
    "SFG": {
        "current": "San Francisco Giants",
        "prior_eras": ["New York Giants"],
        "era_defining_figures": ["Willie Mays", "John McGraw", "Christy Mathewson"],
    },
    "ATL": {
        "current": "Atlanta Braves",
        "prior_eras": ["Boston Braves", "Milwaukee Braves"],
        "era_defining_figures": ["Hank Aaron", "Warren Spahn"],
    },
    "OAK": {
        "current": "Oakland Athletics / Athletics",
        "prior_eras": ["Philadelphia Athletics", "Kansas City Athletics"],
        "era_defining_figures": ["Connie Mack", "Reggie Jackson", "Catfish Hunter"],
    },
    "MIN": {
        "current": "Minnesota Twins",
        "prior_eras": ["Washington Senators"],
        "era_defining_figures": ["Walter Johnson", "Harmon Killebrew"],
    },
    "TEX": {
        "current": "Texas Rangers",
        "prior_eras": ["Washington Senators"],
        "era_defining_figures": [],
    },
    "BAL": {
        "current": "Baltimore Orioles",
        "prior_eras": ["St. Louis Browns"],
        "era_defining_figures": ["George Sisler", "Cal Ripken"],
    },
    "MIA": {
        "current": "Miami Marlins",
        "prior_eras": ["Florida Marlins"],
        "era_defining_figures": [],
    },
    "WSN": {
        "current": "Washington Nationals",
        "prior_eras": ["Montreal Expos"],
        "era_defining_figures": ["Andre Dawson", "Tim Raines"],
    },
    "MIL": {
        "current": "Milwaukee Brewers",
        "prior_eras": ["Seattle Pilots"],
        "era_defining_figures": [],
    },
}

print(f"Auditing {len(CURRENT_FRANCHISES_WITH_HISTORY)} current franchises with documented prior-era history\n")
print(f"{'franchID':10s} {'prior eras':35s} {'era figures':35s} {'verdict'}")
print("-" * 110)

historical_arc_problems = []
for fid, info in CURRENT_FRANCHISES_WITH_HISTORY.items():
    rows = df_team_profiles[df_team_profiles['franchID'] == fid]
    if len(rows) == 0:
        print(f"{fid:10s} (not in df_team_profiles)")
        continue
    content = rows.iloc[0]['profile']
    content_lower = content.lower()

    prior_present = [e for e in info["prior_eras"] if e.lower() in content_lower]
    prior_missing = [e for e in info["prior_eras"] if e.lower() not in content_lower]

    figures_present = [f for f in info["era_defining_figures"]
                       if f.lower() in content_lower or f.split()[-1].lower() in content_lower]
    figures_missing = [f for f in info["era_defining_figures"] if f not in figures_present]

    has_problem = bool(prior_missing) or (info["era_defining_figures"] and not figures_present)
    verdict = "✗ INCOMPLETE" if has_problem else "✓ ok"
    if has_problem:
        historical_arc_problems.append({
            "franchID": fid,
            "missing_eras": prior_missing,
            "missing_figures": figures_missing,
        })

    eras_str = f"{len(prior_present)}/{len(info['prior_eras'])}"
    if prior_missing:
        eras_str += f" miss:{prior_missing}"
    figs_str = (f"{len(figures_present)}/{len(info['era_defining_figures'])}"
                if info["era_defining_figures"] else "n/a")
    if figures_missing:
        figs_str += f" miss:{figures_missing[:2]}"
    print(f"{fid:10s} {eras_str:35s} {figs_str:35s} {verdict}")

print(f"\nFranchises needing historical-arc fix: {len(historical_arc_problems)} / {len(CURRENT_FRANCHISES_WITH_HISTORY)}")

Auditing 10 current franchises with documented prior-era history

franchID   prior eras                          era figures                         verdict
--------------------------------------------------------------------------------------------------------------
LAD        0/1 miss:['Brooklyn']               0/2 miss:['Jackie Robinson', 'Sandy Koufax'] ✗ INCOMPLETE
SFG        1/1                                 0/3 miss:['Willie Mays', 'John McGraw'] ✗ INCOMPLETE
ATL        0/2 miss:['Boston Braves', 'Milwaukee Braves'] 0/2 miss:['Hank Aaron', 'Warren Spahn'] ✗ INCOMPLETE
OAK        1/2 miss:['Kansas City Athletics']  0/3 miss:['Connie Mack', 'Reggie Jackson'] ✗ INCOMPLETE
MIN        1/1                                 0/2 miss:['Walter Johnson', 'Harmon Killebrew'] ✗ INCOMPLETE
TEX        1/1                                 n/a                                 ✓ ok
BAL        1/1                                 0/2 miss:['George Sisler', 'Cal Ripken'] ✗ INCOMPLETE
MIA        (not 

In [ ]:
# What do the suspect profiles actually say? Print the relevant chunks.
print("=== Inspecting profiles flagged as incomplete ===\n")

for problem in historical_arc_problems:
    fid = problem["franchID"]
    info = CURRENT_FRANCHISES_WITH_HISTORY[fid]
    rows = df_team_profiles[df_team_profiles['franchID'] == fid]
    content = rows.iloc[0]['profile']

    print(f"--- {fid} ({info['current']}) ---")
    print(f"Missing eras:    {problem['missing_eras']}")
    print(f"Missing figures: {problem['missing_figures']}")
    print(f"Profile (first 600 chars):")
    print(content[:600])
    print()

# Also verify MIA — the "not in df_team_profiles" case is unexpected
print("=== Investigating MIA ===")
mia_rows = df_team_profiles[df_team_profiles['franchID'] == 'MIA']
print(f"df_team_profiles rows for MIA: {len(mia_rows)}")
fla_rows = df_team_profiles[df_team_profiles['franchID'] == 'FLA']
print(f"df_team_profiles rows for FLA: {len(fla_rows)}")
if len(fla_rows) > 0:
    print("FLA profile excerpt (Marlins live under franchID=FLA in Lahman):")
    print(fla_rows.iloc[0]['profile'][:400])

=== Inspecting profiles flagged as incomplete ===

--- LAD (Los Angeles Dodgers) ---
Missing eras:    ['Brooklyn']
Missing figures: ['Jackie Robinson', 'Sandy Koufax']
Profile (first 600 chars):
The Los Angeles Dodgers are an active National League franchise based in Los Angeles, California, currently competing in the NL West division. Known for their consistent success and strong fan base, the Dodgers have been a prominent team in Major League Baseball for decades.

The Dodgers franchise boasts a rich history dating back to 1884, operating for 142 seasons. They have secured nine World Series titles in 1955, 1959, 1963, 1965, 1981, 1988, 2020, 2024, and 2025, alongside 27 league pennants. Their all-time regular season record stands at 11,525 wins and 10,137 losses, for a winning perce

--- SFG (San Francisco Giants) ---
Missing eras:    []
Missing figures: ['Willie Mays', 'John McGraw', 'Christy Mathewson']
Profile (first 600 chars):
The San Francisco Giants are an active National Leag

In [ ]:
# A "recent player" for current-state-contamination purposes is anyone Gemini
# might have current-state material on. That's:
#   1. Active players (final_game_date is NULL but they played in the last few years)
#   2. Players whose final game was 2022 or later (recently retired, still in news)
#
# It is NOT just "debut after 2016" — a player who debuted in 2010 and is still
# active in 2026 has just as much current-state contamination risk.

# Use the most recent batting/pitching season as the "last active year" signal.
# This is more reliable than People.final_game_date which is sparse for actives.

last_active_year = {}

for pid, group in batting_df.groupby("playerID"):
    last_active_year[pid] = max(last_active_year.get(pid, 0), group["yearID"].max())

for pid, group in pitching_df.groupby("playerID"):
    last_active_year[pid] = max(last_active_year.get(pid, 0), group["yearID"].max())

# How many played in 2022 or later?
RECENT_THRESHOLD = 2022  # arbitrary cutoff: last 4 seasons in Lahman
recent_player_ids = {pid for pid, yr in last_active_year.items() if yr >= RECENT_THRESHOLD}

print(f"Players with last MLB season >= {RECENT_THRESHOLD}: {len(recent_player_ids):,}")
print(f"Breakdown by last active year:")
year_counts = Counter(yr for pid, yr in last_active_year.items() if yr >= RECENT_THRESHOLD)
for yr in sorted(year_counts.keys()):
    print(f"  {yr}: {year_counts[yr]:,} players")

# Also include 19th-century-style entries with no batting/pitching rows
# (rare, but possible — pinch runners, very brief debuts)
players_no_stats = set(people_df["playerID"]) - set(last_active_year.keys())
print(f"\nPlayers with no batting/pitching rows at all: {len(players_no_stats)}")
print(f"  (these can't be 'recent' — excluded from regen scope)")

# What's the full regen scope for active-current-state-strip?
print(f"\n=== Player profile regen scope estimate ===")
print(f"  'Recent' players (active 2022+):     {len(recent_player_ids):,}")
print(f"  At ~250 players/min sequential:      {len(recent_player_ids)/250:.1f} minutes")
print(f"  At 50 workers parallel (per Section 11): ~{len(recent_player_ids)/(250*50)*60:.0f} seconds (rate-limit dependent)")

Players with last MLB season >= 2022: 2,374
Breakdown by last active year:
  2022: 304 players
  2023: 272 players
  2024: 328 players
  2025: 1,470 players

Players with no batting/pitching rows at all: 259
  (these can't be 'recent' — excluded from regen scope)

=== Player profile regen scope estimate ===
  'Recent' players (active 2022+):     2,374
  At ~250 players/min sequential:      9.5 minutes
  At 50 workers parallel (per Section 11): ~11 seconds (rate-limit dependent)


In [ ]:
# How many of the 110 dup-affected players are also in the recent-active set?
overlap_with_recent = DUPLICATE_PLAYER_IDS & recent_player_ids
only_dup_not_recent = DUPLICATE_PLAYER_IDS - recent_player_ids

print(f"Dup-affected players: {len(DUPLICATE_PLAYER_IDS)}")
print(f"  Also recent (will be regenerated anyway): {len(overlap_with_recent)}")
print(f"  Old-era dups (separate regen needed):     {len(only_dup_not_recent)}")
print(f"\nUnion (full player regen scope): {len(DUPLICATE_PLAYER_IDS | recent_player_ids)}")

Dup-affected players: 110
  Also recent (will be regenerated anyway): 5
  Old-era dups (separate regen needed):     105

Union (full player regen scope): 2479


### 15.1 Regeneration scope and approach

Audit findings drive the scope:

- **Team profiles**: all 30 active franchises regenerated. Combines three fixes — strip current-state content, mandate historical-arc coverage for relocated franchises (LAD, ATL, OAK at minimum), repair the 4 dup-affected files (LAD, MIL, BAL, PHI). Defunct franchises (173) keep their v1 profiles since they have no current-state to strip and no historical-arc gaps to fill.

- **Player profiles**: 2,479 regenerated. The union of (a) 2,374 recent active players whose v1 profiles contain in-season content, and (b) 110 dup-affected players (5 already in the recent-active set; 105 separate). All other players (~21,800) keep their v1 profiles.

Architectural principle the v2 prompts enforce: profiles are **historical and biographical reference**. Current-state — current standings, current injuries, in-season transactions — belongs in the MLB Stats API layer (the ADK agent in Task 3) and Search grounding at query time, not baked into static documents.

V1 profiles are preserved to versioned GCS prefixes (`players-v1/`, `teams-v1/`) before overwrite, in case rollback is needed.

In [ ]:
# Copy current profiles to versioned v1 prefixes before overwriting in regen.
# Skips the backup if v1 already exists — protects the original v1 from being
# clobbered on notebook reruns.

print("Checking v1 backup status...")

V1_PLAYERS_PREFIX = f"{GCS_PREFIX}/profiles/players-v1"
V1_TEAMS_PREFIX = f"{GCS_PREFIX}/profiles/teams-v1"


def backup_if_missing(src_prefix: str, v1_prefix: str, label: str):
    """Copy src_prefix → v1_prefix only if v1 doesn't already exist.
    Parallelizes the copy when work is needed."""
    # Check if v1 already has content
    v1_existing = sum(1 for b in client.list_blobs(GCS_BUCKET, prefix=f"{v1_prefix}/")
                      if b.name.endswith(".md"))
    if v1_existing > 0:
        print(f"  {label}: v1 backup already exists ({v1_existing:,} files) — skipping")
        return

    src_blobs = [b for b in client.list_blobs(GCS_BUCKET, prefix=f"{src_prefix}/")
                 if b.name.endswith(".md")]
    if not src_blobs:
        print(f"  {label}: source is empty — nothing to back up")
        return

    print(f"  {label}: backing up {len(src_blobs):,} files to v1...")

    def copy_one(src_blob):
        dst_name = src_blob.name.replace(src_prefix, v1_prefix, 1)
        try:
            bucket.copy_blob(src_blob, bucket, dst_name, retry=_GCS_RETRY)
            return ("ok", src_blob.name)
        except Exception as e:
            return ("error", src_blob.name, str(e)[:120])

    copied = 0
    failed = 0
    with ThreadPoolExecutor(max_workers=50) as ex:
        futures = [ex.submit(copy_one, b) for b in src_blobs]
        with tqdm(total=len(futures), desc=f"  Backing up {label}") as pbar:
            for fut in as_completed(futures):
                r = fut.result()
                if r[0] == "ok":
                    copied += 1
                else:
                    failed += 1
                pbar.update(1)
    print(f"  ✓ {copied:,} copied, {failed} failed")


backup_if_missing(PROFILES_GCS_PREFIX, V1_PLAYERS_PREFIX, "players")
backup_if_missing(TEAMS_GCS_PREFIX, V1_TEAMS_PREFIX, "teams")

print("\nV1 backup status confirmed.")

Checking v1 backup status...
  players: v1 backup already exists (24,270 files) — skipping
  teams: v1 backup already exists (203 files) — skipping

V1 backup status confirmed.


In [ ]:
# Reverse lookup: which franchises are in the historical-arc dictionary?
RELOCATED_FRANCHIDS = set(CURRENT_FRANCHISES_WITH_HISTORY.keys())
# Note: MIA's data is under FLA in Lahman; remap key for lookup
CURRENT_FRANCHISES_WITH_HISTORY["FLA"] = CURRENT_FRANCHISES_WITH_HISTORY.pop("MIA")
RELOCATED_FRANCHIDS = set(CURRENT_FRANCHISES_WITH_HISTORY.keys())


def create_team_prompt_v2(franch_id: str) -> str:
    """V2 team prompt: historical-only, no current-state, mandatory arc coverage for relocated franchises."""
    context, is_active = build_team_context(franch_id)

    # Active vs. defunct framing (kept from v1)
    if is_active:
        tense = "Use present tense for the team's current identity ('the Yankees are an AL East franchise'), past tense for historical claims."
    else:
        tense = "Use past tense throughout. This is a defunct franchise."

    # Historical-arc requirement for relocated franchises
    arc_directive = ""
    if franch_id in CURRENT_FRANCHISES_WITH_HISTORY:
        info = CURRENT_FRANCHISES_WITH_HISTORY[franch_id]
        prior_eras_str = ", ".join(info["prior_eras"])
        figures_str = ", ".join(info["era_defining_figures"]) if info["era_defining_figures"] else ""
        arc_directive = f"""

CRITICAL — HISTORICAL ARC REQUIREMENT: This franchise has played under prior identities. The profile MUST explicitly cover these prior eras: {prior_eras_str}. Frame championships, milestones, and notable players according to the era they actually belonged to (e.g., a 1957 World Series win is a Milwaukee Braves accomplishment, not an Atlanta Braves one). """

        if figures_str:
            arc_directive += f"Mention era-defining figures by name when contextually appropriate: {figures_str}."

    return f"""You are writing a historical reference profile of a baseball franchise for a front-office reference document. The structured data below comes from the Lahman Baseball Database, current through the 2025 season.

{context}

=== INSTRUCTIONS ===
Write a 200-300 word profile, plain markdown, no heading, 2-3 paragraphs.

{tense}{arc_directive}

CRITICAL — NO CURRENT-STATE CONTENT: This profile is a stable historical reference document. Do NOT include:
- Current standings, current win-loss records, current league position
- Current rosters or specific current players (general references like "the Yankees" are fine; specific names of currently active players are not)
- In-season events, recent injuries, recent transactions, recent trades
- "As of [any date]" framing
- Anything that would become stale within months

Current-state information lives in the MLB Stats API and is fetched at query time, not baked into this document.

Profile structure:
- Para 1: Identity. League, division, city, summary of franchise character. For relocated franchises, name the current city AND acknowledge the prior cities.
- Para 2: Historical arc. Founding, key eras, championships, era-defining players or seasons, why the franchise is significant. Past tense for historical claims, attributing accomplishments to the era they actually belonged to.
- Para 3 (optional, only if there's meaningful long-term identity material): Stable narrative — franchise rivalries, traditions, why fans recognize this team, era-spanning identity. NOT current state.

Use Google Search to confirm historical facts but do NOT use it to surface current standings or in-season news.

Output prose only. No JSON, no bullets, no headers.

Profile for franchise {franch_id}:"""


def create_player_prompt_v2(player_row) -> str:
    """V2 player prompt: historical-biographical, no in-season content, current team OK as anchor."""
    context = build_player_context(player_row)
    name = player_row.get("name_full") or player_row.get("playerID")

    return f"""You are writing a historical and biographical profile of a baseball player for a front-office reference document. The structured data below comes from the Lahman Baseball Database, current through the 2025 season.

{context}

=== INSTRUCTIONS ===
Write a 200-300 word profile, plain markdown, no heading, 2-3 paragraphs.

CRITICAL — NO CURRENT-STATE CONTENT: This profile is a stable historical/biographical reference. Do NOT include:
- Current season statistics or standings
- Current injury status, current IL placements
- Recent transactions, recent trades, recent contract signings or extensions
- "As of [any date]" framing
- This-season highlights or "is currently leading the league in..."
- Anything that would become stale within weeks or months

Current team affiliation IS allowed as a slow-changing biographical fact ("a right fielder for the New York Yankees", "currently with the Dodgers"). Do not mention specific in-season events involving the team.

Profile structure:
- For ACTIVE players (still playing): present tense for identity ("Aaron Judge is the Yankees' captain"), past tense for career narrative.
- For RETIRED players: past tense throughout.
- Para 1: Who they are/were. Position, team affiliation (current for active, primary career team(s) for retired), one or two things they're best known for.
- Para 2: Career arc. Debut, milestones, awards, notable seasons, postseason highlights. Weave structured data into prose.
- Para 3 (optional, only if there's meaningful career-spanning material): Long-term legacy, playing style, comparisons. NOT current state.

Use Google Search to confirm career facts but do NOT use it to surface in-season standings, current injuries, or recent transactions.

Output prose only. No JSON, no bullets, no headers.

Profile for {name}:"""


print("V2 prompts defined")
print(f"  Relocated franchises with historical-arc requirement: {sorted(RELOCATED_FRANCHIDS)}")

V2 prompts defined
  Relocated franchises with historical-arc requirement: ['ATL', 'BAL', 'FLA', 'LAD', 'MIL', 'MIN', 'OAK', 'SFG', 'TEX', 'WSN']


In [ ]:
# Generate v2 profiles for all 30 active franchises sequentially.
# Reuse generate_team_profile() but swap the prompt builder.

# Identify active franchises
active_franchises = [
    fid for fid in df_team_profiles['franchID']
    if df_team_profiles[df_team_profiles['franchID'] == fid].iloc[0]['is_active']
]
print(f"Regenerating {len(active_franchises)} active team profiles with v2 prompt...")

# Patch the prompt builder used by generate_team_profile by swapping in v2
# (the function reads create_team_prompt; we replace it for this run only)
import types as types_module
_original_create_team_prompt = create_team_prompt
create_team_prompt = create_team_prompt_v2

team_v2_results = []
for fid in tqdm(active_franchises, desc="V2 team regen"):
    result = generate_team_profile(fid)
    team_v2_results.append(result)

    # Update in-memory df
    idx = df_team_profiles[df_team_profiles['franchID'] == fid].index[0]
    df_team_profiles.at[idx, 'profile'] = result['profile']
    df_team_profiles.at[idx, 'source'] = result['source']
    df_team_profiles.at[idx, 'error'] = result['error']

    # Overwrite in GCS
    blob_path = f"{TEAMS_GCS_PREFIX}/{fid}.md"
    bucket.blob(blob_path).upload_from_string(result['profile'], content_type="text/markdown")

# Restore original prompt builder
create_team_prompt = _original_create_team_prompt

# Refresh word count
df_team_profiles['word_count'] = df_team_profiles['profile'].apply(lambda s: len(s.split()) if isinstance(s, str) else 0)

# Save updated progress
df_team_profiles.to_csv(TeamProfileConfig.PROGRESS_FILE, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

successes = sum(1 for r in team_v2_results if not r.get("error"))
print(f"\nTeam v2 regen complete: {successes}/{len(team_v2_results)} successful")

# Quick spot check: verify LAD now mentions Brooklyn
lad = df_team_profiles[df_team_profiles['franchID'] == 'LAD'].iloc[0]['profile']
print(f"\nLAD v2 profile excerpt (first 600 chars):")
print(lad[:600])
print(f"\nMentions 'Brooklyn'? {'Brooklyn' in lad}")
print(f"Mentions 'Robinson'? {'Robinson' in lad}")

Regenerating 30 active team profiles with v2 prompt...


V2 team regen:   0%|          | 0/30 [00:00<?, ?it/s]


Team v2 regen complete: 30/30 successful

LAD v2 profile excerpt (first 600 chars):
The Los Angeles Dodgers are a National League West franchise with a rich history rooted in both Brooklyn and Los Angeles. Known for a tradition of competitive baseball, the Dodgers have consistently been a prominent team in the league.

The franchise began its journey in Brooklyn, operating under various names such as the Brooklyn Atlantics, Bridegrooms, Grays, Grooms, Superbas, Robins, and most famously, the Brooklyn Dodgers. During their time in Brooklyn, they secured a World Series title in 1955. This era was significantly marked by pioneering figures like Jackie Robinson, who broke the col

Mentions 'Brooklyn'? True
Mentions 'Robinson'? True


In [ ]:
for fid in ["LAD", "ATL", "OAK"]:
    profile = df_team_profiles[df_team_profiles['franchID'] == fid].iloc[0]['profile']
    print(f"=== {fid} ({len(profile.split())} words) ===")
    print(profile)
    print()

=== LAD (210 words) ===
The Los Angeles Dodgers are a National League West franchise with a rich history rooted in both Brooklyn and Los Angeles. Known for a tradition of competitive baseball, the Dodgers have consistently been a prominent team in the league.

The franchise began its journey in Brooklyn, operating under various names such as the Brooklyn Atlantics, Bridegrooms, Grays, Grooms, Superbas, Robins, and most famously, the Brooklyn Dodgers. During their time in Brooklyn, they secured a World Series title in 1955. This era was significantly marked by pioneering figures like Jackie Robinson, who broke the color barrier in Major League Baseball. The team then relocated to Los Angeles, where they continued their legacy of success. As the Los Angeles Dodgers, they have added eight more World Series championships to their name in 1959, 1963, 1965, 1981, 1988, 2020, 2024, and 2025. Pitching icon Sandy Koufax was a defining player during their early years in Los Angeles, contributing

In [ ]:
# Dedup check on active teams
active_team_dupes = []
for fid in active_franchises:
    profile = df_team_profiles[df_team_profiles['franchID'] == fid].iloc[0]['profile']
    result = find_duplicated_content(profile)
    if result['duplicated']:
        active_team_dupes.append({"franchID": fid, "overlap": result['overlap_ratio']})
print(f"Dup check: {len(active_team_dupes)}/{len(active_franchises)} dup-affected (was 4)")
for d in active_team_dupes:
    print(f"  {d['franchID']}: overlap={d['overlap']:.2f}")

# Current-state staleness check on active teams
staleness_markers = ["as of may", "as of april", "as of march", "as of february",
                     "as of 2026", "in 2026", "currently leading",
                     "this season", "injured list", "recent transactions",
                     "extension in", "signed", "is currently on"]
stale_active = []
for fid in active_franchises:
    profile = df_team_profiles[df_team_profiles['franchID'] == fid].iloc[0]['profile'].lower()
    found = [m for m in staleness_markers if m in profile]
    if found:
        stale_active.append({"franchID": fid, "markers": found})
print(f"\nStaleness check: {len(stale_active)}/{len(active_franchises)} contain current-state markers")
for s in stale_active[:10]:
    print(f"  {s['franchID']}: {s['markers']}")

# Historical arc check on relocated franchises
arc_remaining = []
for fid, info in CURRENT_FRANCHISES_WITH_HISTORY.items():
    rows = df_team_profiles[df_team_profiles['franchID'] == fid]
    if len(rows) == 0:
        continue
    content = rows.iloc[0]['profile'].lower()
    missing = [e for e in info['prior_eras'] if e.lower() not in content]
    if missing:
        arc_remaining.append({"franchID": fid, "missing": missing})
print(f"\nArc check: {len(arc_remaining)}/{len(CURRENT_FRANCHISES_WITH_HISTORY)} still missing prior-era mentions (was 3)")
for a in arc_remaining:
    print(f"  {a['franchID']}: missing {a['missing']}")

Dup check: 0/30 dup-affected (was 4)

Staleness check: 1/30 contain current-state markers
  NYY: ['signed']

Arc check: 0/10 still missing prior-era mentions (was 3)


In [ ]:
# Build the regen scope: union of dup-affected + recent-active
PLAYER_REGEN_IDS = DUPLICATE_PLAYER_IDS | recent_player_ids
print(f"Player regen scope: {len(PLAYER_REGEN_IDS):,} players")
print(f"  Dup-affected:  {len(DUPLICATE_PLAYER_IDS)}")
print(f"  Recent active: {len(recent_player_ids):,}")
print(f"  Union:         {len(PLAYER_REGEN_IDS):,}")

# Filter to players we have rows for
players_to_regen = people_df[people_df['playerID'].isin(PLAYER_REGEN_IDS)].copy()
print(f"  Found in people_df: {len(players_to_regen):,}")

# Patch in the v2 prompt builder
_original_create_player_prompt = create_player_prompt
create_player_prompt = create_player_prompt_v2

# Generation function: same shape as Section 11, but with the v2 prompt
def regen_one_player(idx_row):
    idx, row = idx_row
    result = generate_player_profile(row)
    if result.get("error"):
        log_player_error(result["playerID"], result["name"], result["error"])
    return result


print(f"\nStarting parallel player regen with v2 prompt...")
print(f"  Workers: {PlayerProfileConfig.MAX_WORKERS}")
print(f"  Batch:   {PlayerProfileConfig.BATCH_SIZE}\n")

start = time.time()
v2_results = []
rows = list(players_to_regen.iterrows())
n_batches = (len(rows) + PlayerProfileConfig.BATCH_SIZE - 1) // PlayerProfileConfig.BATCH_SIZE

with tqdm(total=len(rows), desc="V2 player regen") as pbar:
    for b in range(n_batches):
        s = b * PlayerProfileConfig.BATCH_SIZE
        e = min(s + PlayerProfileConfig.BATCH_SIZE, len(rows))
        batch = rows[s:e]

        with ThreadPoolExecutor(max_workers=PlayerProfileConfig.MAX_WORKERS) as ex:
            futures = {ex.submit(regen_one_player, item): item for item in batch}
            for fut in as_completed(futures):
                v2_results.append(fut.result())
                pbar.update(1)

# Restore original prompt
create_player_prompt = _original_create_player_prompt

elapsed = time.time() - start
successes = sum(1 for r in v2_results if not r.get("error"))
errors = len(v2_results) - successes

print(f"\nV2 player regen complete")
print(f"  Total:      {len(v2_results):,}")
print(f"  Successful: {successes:,}")
print(f"  Errors:     {errors}")
print(f"  Time:       {elapsed/60:.1f} minutes")

Player regen scope: 2,479 players
  Dup-affected:  110
  Recent active: 2,374
  Union:         2,479
  Found in people_df: 2,479

Starting parallel player regen with v2 prompt...
  Workers: 50
  Batch:   200



V2 player regen:   0%|          | 0/2479 [00:00<?, ?it/s]


V2 player regen complete
  Total:      2,479
  Successful: 2,437
  Errors:     42
  Time:       13.0 minutes


In [ ]:
# Retry only the 42 failed players from the previous regen
errored_results = [r for r in v2_results if r.get("error")]
errored_ids = {r["playerID"] for r in errored_results}
print(f"Retrying {len(errored_ids)} failed regens...")

# Inspect a few errors first to see if they're transient or systemic
print("\nSample errors:")
for r in errored_results[:5]:
    print(f"  {r['playerID']:14s} {r.get('name', '?'):30s} → {str(r.get('error'))[:100]}")

# Retry the failed ones
retry_rows = people_df[people_df['playerID'].isin(errored_ids)].copy()

# Patch in v2 prompt
_orig_prompt = create_player_prompt
create_player_prompt = create_player_prompt_v2

retry_results = []
with tqdm(total=len(retry_rows), desc="Retry") as pbar:
    with ThreadPoolExecutor(max_workers=PlayerProfileConfig.MAX_WORKERS) as ex:
        futures = {ex.submit(process_one_player, item): item for item in retry_rows.iterrows()}
        for fut in as_completed(futures):
            retry_results.append(fut.result())
            pbar.update(1)

create_player_prompt = _orig_prompt

retry_success = sum(1 for r in retry_results if not r.get("error"))
retry_failed = len(retry_results) - retry_success
print(f"\nRetry complete: {retry_success}/{len(retry_results)} succeeded, {retry_failed} still failing")

# Replace the failed entries in v2_results with the retry results
v2_results = [r for r in v2_results if r["playerID"] not in errored_ids] + retry_results
print(f"v2_results now contains: {sum(1 for r in v2_results if not r.get('error'))} successes, "
      f"{sum(1 for r in v2_results if r.get('error'))} errors")

Retrying 42 failed regens...

Sample errors:
  garrebr01      Braxton Garrett                → ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please 
  garvemi01      Mitch Garver                   → ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please 
  gervapa01      Paul Gervase                   → ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please 
  sproabr01      Brandon Sproat                 → ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please 
  stammcr01      Craig Stammen                  → ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please 


Retry:   0%|          | 0/42 [00:00<?, ?it/s]


Retry complete: 42/42 succeeded, 0 still failing
v2_results now contains: 2479 successes, 0 errors


In [ ]:
print(f"Updating in-memory df and GCS for {len(v2_results)} regenerated player profiles...")

uploaded = 0
upload_errors = 0
for new in tqdm(v2_results):
    pid = new["playerID"]
    # Update in-memory df
    idx_match = df_player_profiles[df_player_profiles['playerID'] == pid].index
    if len(idx_match) > 0:
        idx = idx_match[0]
        df_player_profiles.at[idx, 'profile'] = new['profile']
        df_player_profiles.at[idx, 'source'] = new['source']
        df_player_profiles.at[idx, 'error'] = new['error']

    # Overwrite in GCS
    if isinstance(new.get('profile'), str) and new['profile']:
        try:
            blob_path = f"{PROFILES_GCS_PREFIX}/{pid}.md"
            bucket.blob(blob_path).upload_from_string(new['profile'], content_type="text/markdown")
            uploaded += 1
        except Exception:
            upload_errors += 1

# Refresh word count
df_player_profiles['word_count'] = df_player_profiles['profile'].apply(lambda s: len(s.split()) if isinstance(s, str) else 0)

# Save updated progress
df_player_profiles.to_csv(PlayerProfileConfig.PROGRESS_FILE, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

print(f"  Uploaded:       {uploaded:,}")
print(f"  Upload errors:  {upload_errors}")

Updating in-memory df and GCS for 2479 regenerated player profiles...


  0%|          | 0/2479 [00:00<?, ?it/s]

  Uploaded:       2,479
  Upload errors:  0


In [ ]:
import os
# Check if anything got saved during the regen
for f in os.listdir("/tmp"):
    if "v2" in f.lower() or "regen" in f.lower() or "player_profile" in f.lower():
        path = f"/tmp/{f}"
        size = os.path.getsize(path)
        print(f"{path}  {size:,} bytes")

# Also check the progress file specifically
import os.path
pf = "/tmp/mlb_player_profiles_progress.csv"  # or wherever PlayerProfileConfig.PROGRESS_FILE points
if os.path.exists(pf):
    print(f"\n{pf} exists, {os.path.getsize(pf):,} bytes")
    import pandas as pd
    df = pd.read_csv(pf)
    print(f"Rows: {len(df)}")
    # Check a recent player to see if it has v2 content
    sample = df.head(5)
    print(f"\nFirst 5 playerIDs: {sample['playerID'].tolist() if 'playerID' in df.columns else 'no playerID col'}")
else:
    print(f"\n{pf} not found")

/tmp/mlb_player_profiles_progress.csv  25,991,165 bytes
/tmp/mlb_player_profiles_errors.csv  13,023 bytes

/tmp/mlb_player_profiles_progress.csv exists, 25,991,165 bytes
Rows: 24270

First 5 playerIDs: ['andergr01', 'abreujo02', 'aldresc01', 'balazjo02', 'arozara01']


In [ ]:
# Verify v2 player regen fixed what it was supposed to fix
print("Auditing the 2,479 regenerated player profiles...\n")

# Dedup check on regenerated subset
regen_df = df_player_profiles[df_player_profiles['playerID'].isin(PLAYER_REGEN_IDS)]
dup_remaining = []
for _, row in regen_df.iterrows():
    result = find_duplicated_content(row['profile'])
    if result['duplicated']:
        dup_remaining.append({"playerID": row['playerID'], "overlap": result['overlap_ratio']})
print(f"Dedup check: {len(dup_remaining)}/{len(regen_df)} still duplicated (was 110)")
for d in dup_remaining[:10]:
    print(f"  {d['playerID']}: overlap={d['overlap']:.2f}")

# Staleness check on regenerated subset
staleness_markers = ["as of may", "as of april", "as of march", "as of february",
                     "as of 2026", "in 2026", "currently leading",
                     "this season", "injured list", "extension in", "is currently on"]
stale_remaining = []
for _, row in regen_df.iterrows():
    profile_lower = row['profile'].lower()
    found = [m for m in staleness_markers if m in profile_lower]
    if found:
        stale_remaining.append({"playerID": row['playerID'], "markers": found})
print(f"\nStaleness check: {len(stale_remaining)}/{len(regen_df)} contain current-state markers")
for s in stale_remaining[:10]:
    print(f"  {s['playerID']}: {s['markers']}")

Auditing the 2,479 regenerated player profiles...

Dedup check: 4/2479 still duplicated (was 110)
  abreujo02: overlap=0.99
  gimenan01: overlap=0.62
  moorech03: overlap=0.70
  sanchan01: overlap=0.46

Staleness check: 76/2479 contain current-state markers
  balazjo02: ['in 2026']
  actonga01: ['in 2026']
  adelljo01: ['in 2026']
  antonte01: ['injured list']
  beeksja01: ['in 2026']
  belljo02: ['in 2026']
  bermujo01: ['in 2026']
  boltoco01: ['injured list']
  buchada01: ['in 2026']
  camerda01: ['in 2026']


In [ ]:
# Sample the staleness flags to see what kind of content is triggering them
print("=== Sample of staleness-flagged player profiles (full text excerpts) ===\n")

for s in stale_remaining[:8]:
    pid = s['playerID']
    profile = df_player_profiles[df_player_profiles['playerID'] == pid].iloc[0]['profile']
    # Find the sentence containing the marker
    for marker in s['markers']:
        # Get ~100 chars around the marker
        idx = profile.lower().find(marker)
        if idx >= 0:
            start = max(0, idx - 80)
            end = min(len(profile), idx + len(marker) + 80)
            excerpt = profile[start:end]
            print(f"{pid} → marker '{marker}':")
            print(f"  ...{excerpt}...")
            print()

# Also look at the 4 remaining duplications — see if they're true repetition
# or borderline cases
print("\n=== Remaining duplication cases (first 400 chars each) ===\n")
for d in dup_remaining:
    pid = d['playerID']
    profile = df_player_profiles[df_player_profiles['playerID'] == pid].iloc[0]['profile']
    print(f"{pid} (overlap={d['overlap']:.2f}, words={len(profile.split())}):")
    print(profile[:400])
    print("..." if len(profile) > 400 else "")
    print()

=== Sample of staleness-flagged player profiles (full text excerpts) ===

balazjo02 → marker 'in 2026':
  ...signed with the Uni-President Lions of the Chinese Professional Baseball League in 2026....

actonga01 → marker 'in 2026':
  ...ter joined the Miami Marlins and was subsequently traded to the Minnesota Twins in 2026.

Throughout his MLB career, Acton has played for the Oakland Athletics, Tampa ...

adelljo01 → marker 'in 2026':
  ...tes and nominated for the All-MLB Team. He also showcased his defensive prowess in 2026, becoming the first MLB player to rob three home runs in a single game.

Throug...

antonte01 → marker 'injured list':
  ...career ERA of 2.47. His professional career has included multiple stints on the injured list, including two Tommy John surgeries in 2017 and 2021, and another elbow surgery...

beeksja01 → marker 'in 2026':
  ...n joined the Arizona Diamondbacks in 2025 before signing with the Texas Rangers in 2026. Over his career, Beeks has accumulated 28 w

In [ ]:
abreu = df_player_profiles[df_player_profiles['playerID'] == 'abreujo02'].iloc[0]['profile']
print(f"Length: {len(abreu.split())} words\n")
print(abreu)

Length: 403 words

Jose Abreu is a Cuban-born first baseman known for his consistent power and run production throughout his MLB career. He debuted in 2014 with the Chicago White Sox, where he spent the majority of his career, and later played for the Houston Astros. Abreu is recognized for being a highly decorated player, earning both Rookie of the Year and MVP honors.

Abreu's career arc began with a bang, as he was named the American League Rookie of the Year in 2014. He continued to be a cornerstone for the White Sox offense, highlighted by his American League Most Valuable Player award in the 2020 season. Throughout his 11 seasons (2014-2024), he accumulated 1587 hits, 263 home runs, and 960 RBIs. He also earned three All-Star selections and three Silver Slugger awards. Abreu has participated in three postseasons, contributing 1 home run and 7 RBIs. His last recorded team in the Lahman database is the Houston Astros.

Abreu’s consistent performance established him as one of the le

In [ ]:
# Targeted regen of the one remaining genuine duplication
print("Regenerating abreujo02...")

# Patch in v2 prompt
_orig_prompt = create_player_prompt
create_player_prompt = create_player_prompt_v2

abreu_row = people_df[people_df['playerID'] == 'abreujo02'].iloc[0]
result = generate_player_profile(abreu_row)

create_player_prompt = _orig_prompt

if result.get('error'):
    print(f"  Failed: {result['error']}")
else:
    # Update in-memory df
    idx = df_player_profiles[df_player_profiles['playerID'] == 'abreujo02'].index[0]
    df_player_profiles.at[idx, 'profile'] = result['profile']
    df_player_profiles.at[idx, 'source'] = result['source']
    df_player_profiles.at[idx, 'error'] = None
    df_player_profiles.at[idx, 'word_count'] = len(result['profile'].split())

    # Upload to GCS
    bucket.blob(f"{PROFILES_GCS_PREFIX}/abreujo02.md").upload_from_string(
        result['profile'], content_type="text/markdown"
    )

    # Verify it's no longer duplicated
    check = find_duplicated_content(result['profile'])
    print(f"  ✓ Regenerated ({len(result['profile'].split())} words)")
    print(f"  Dedup check: duplicated={check['duplicated']}, overlap={check['overlap_ratio']:.2f}")
    print(f"\nFirst 400 chars:\n{result['profile'][:400]}")

# Save progress
df_player_profiles.to_csv(
    PlayerProfileConfig.PROGRESS_FILE,
    index=False, encoding='utf-8', quoting=csv.QUOTE_ALL
)

Regenerating abreujo02...
  ✓ Regenerated (204 words)
  Dedup check: duplicated=False, overlap=0.00

First 400 chars:
Jose Abreu is a right-handed first baseman known for his consistent power and run production. A Cuban-born player, he spent the majority of his MLB career with the Chicago White Sox before joining the Houston Astros. He is recognized for his strong hitting approach, often putting the ball in play with high exit velocities.

Abreu made his MLB debut in 2014, earning the American League Rookie of th


In [ ]:
# Pick a few regenerated players and confirm GCS content matches df content
import random
random.seed(0)
sample_pids = random.sample(list(PLAYER_REGEN_IDS), 5)

print("Comparing in-memory df_player_profiles to GCS for 5 random regenerated players...\n")
for pid in sample_pids:
    df_text = df_player_profiles[df_player_profiles['playerID'] == pid].iloc[0]['profile']
    blob = bucket.blob(f"{PROFILES_GCS_PREFIX}/{pid}.md")
    gcs_text = blob.download_as_text()

    match = df_text == gcs_text
    print(f"  {pid}: df_word_count={len(df_text.split())}, gcs_word_count={len(gcs_text.split())}, match={match}")

    if not match:
        # Compare against v1 to see if GCS still has v1 content
        v1_blob = bucket.blob(f"{V1_PLAYERS_PREFIX}/{pid}.md")
        v1_text = v1_blob.download_as_text()
        gcs_is_v1 = gcs_text == v1_text
        print(f"    GCS content matches v1 backup? {gcs_is_v1}")
        if gcs_is_v1:
            print(f"    → Player was regenerated in df but NOT uploaded to GCS")

# Also check the 30 active team profiles
print("\nSpot-checking 3 regenerated team profiles (LAD, ATL, OAK)...")
for fid in ['LAD', 'ATL', 'OAK']:
    df_text = df_team_profiles[df_team_profiles['franchID'] == fid].iloc[0]['profile']
    blob = bucket.blob(f"{TEAMS_PREFIX}/{fid}.md")
    gcs_text = blob.download_as_text()
    match = df_text == gcs_text
    has_brooklyn = "Brooklyn" in gcs_text if fid == 'LAD' else None
    print(f"  {fid}: match={match}" + (f", GCS has 'Brooklyn'={has_brooklyn}" if has_brooklyn is not None else ""))

Comparing in-memory df_player_profiles to GCS for 5 random regenerated players...

  naileja01: df_word_count=192, gcs_word_count=192, match=True
  headrbr01: df_word_count=194, gcs_word_count=194, match=True
  beerse01: df_word_count=199, gcs_word_count=199, match=True
  windejo01: df_word_count=213, gcs_word_count=213, match=True
  skenepa01: df_word_count=235, gcs_word_count=235, match=True

Spot-checking 3 regenerated team profiles (LAD, ATL, OAK)...
  LAD: match=True, GCS has 'Brooklyn'=True
  ATL: match=True
  OAK: match=True


In [ ]:
# Generate a comprehensive schema + sample data dump from GCS staged assets.
# Pulls Parquet schemas directly and parses ALTER TABLE descriptions from
# load_lahman.sql.

import io
import re
import pandas as pd
import pyarrow.parquet as pq

print("=" * 80)
print("PART 1 — GCS BUCKET LAYOUT")
print("=" * 80)

# Walk all prefixes under the project root
all_blobs = list(client.list_blobs(GCS_BUCKET, prefix=f"{GCS_PREFIX}/"))

# Group by top-level prefix
from collections import defaultdict
prefix_groups = defaultdict(list)
for b in all_blobs:
    parts = b.name.split("/")
    if len(parts) >= 3:
        top_prefix = "/".join(parts[:3])
    else:
        top_prefix = "/".join(parts)
    prefix_groups[top_prefix].append(b)

print(f"\nBucket: gs://{GCS_BUCKET}/{GCS_PREFIX}/")
for prefix in sorted(prefix_groups.keys()):
    blobs = prefix_groups[prefix]
    total_size = sum(b.size for b in blobs)
    size_str = f"{total_size / 1024 / 1024:.2f} MB" if total_size > 1024*1024 else f"{total_size / 1024:.1f} KB"
    print(f"  {prefix.replace(GCS_PREFIX + '/', ''):30s} {len(blobs):6,} files  {size_str:>12s}")


print("\n\n" + "=" * 80)
print("PART 2 — PARQUET SCHEMAS FROM GCS")
print("=" * 80)

LAHMAN_PARQUET_PREFIX = f"{GCS_PREFIX}/lahman"
parquet_blobs = [b for b in client.list_blobs(GCS_BUCKET, prefix=f"{LAHMAN_PARQUET_PREFIX}/")
                 if b.name.endswith(".parquet")]

# First pass — read each Parquet's schema via download to memory
parquet_schemas = {}
parquet_row_counts = {}
for blob in parquet_blobs:
    table_name = blob.name.split("/")[-1].replace(".parquet", "")
    print(f"\nReading schema for {table_name}.parquet ...")
    data = blob.download_as_bytes()
    pq_file = pq.ParquetFile(io.BytesIO(data))
    parquet_schemas[table_name] = pq_file.schema_arrow
    parquet_row_counts[table_name] = pq_file.metadata.num_rows


print("\n\n" + "=" * 80)
print("PART 3 — COLUMN DESCRIPTIONS FROM load_lahman.sql")
print("=" * 80)

# The SQL file has ALTER TABLE statements like:
#   ALTER TABLE `dataset.tablename` ALTER COLUMN colname SET OPTIONS(description = '...');
# Parse these into a dict.

sql_blob = bucket.blob(f"{GCS_PREFIX}/schema/load_lahman.sql")
sql_text = sql_blob.download_as_text()

# Pattern handles both single-line and multi-line description strings,
# accounting for backslash-escaped quotes
ALTER_PATTERN = re.compile(
    r"ALTER TABLE\s+`[^`]+\.(\w+)`\s+"
    r"ALTER COLUMN\s+(\w+)\s+"
    r"SET OPTIONS\s*\(\s*description\s*=\s*'((?:[^'\\]|\\.)*)'\s*\)",
    re.IGNORECASE | re.DOTALL
)

descriptions = defaultdict(dict)  # {table: {column: description}}
for match in ALTER_PATTERN.finditer(sql_text):
    table = match.group(1)
    column = match.group(2)
    desc = match.group(3).replace("\\'", "'").replace("\\n", " ").strip()
    descriptions[table][column] = desc

print(f"Parsed {sum(len(v) for v in descriptions.values())} column descriptions "
      f"across {len(descriptions)} tables\n")


print("\n" + "=" * 80)
print("PART 4 — TABLE SCHEMAS WITH COLUMN DESCRIPTIONS")
print("=" * 80)

# Map Lahman names → BQ table names (snake_case in load_lahman.sql)
PARQUET_TO_BQ = {
    "People": "people",
    "Batting": "batting",
    "Pitching": "pitching",
    "Teams": "teams",
    "Appearances": "appearances",
    "AllstarFull": "allstar_full",
    "AwardsPlayers": "awards_players",
    "BattingPost": "batting_post",
    "PitchingPost": "pitching_post",
}

for parquet_name, bq_name in PARQUET_TO_BQ.items():
    if parquet_name not in parquet_schemas:
        print(f"\n⚠️  Parquet not found: {parquet_name}")
        continue

    schema = parquet_schemas[parquet_name]
    n_rows = parquet_row_counts[parquet_name]
    table_descriptions = descriptions.get(bq_name, {})

    print(f"\n{'=' * 80}")
    print(f"TABLE: {bq_name}  (Parquet: {parquet_name}.parquet)")
    print('=' * 80)
    print(f"Rows: {n_rows:,}")
    print(f"Columns: {len(schema)}")
    print(f"Columns with descriptions: {len(table_descriptions)}")

    print(f"\n{'COLUMN':<32} {'TYPE':<14} DESCRIPTION")
    print("-" * 80)
    for field in schema:
        col_name = field.name
        col_type = str(field.type)
        desc = table_descriptions.get(col_name, "(no description)")
        print(f"{col_name:<32} {col_type:<14} {desc}")


print("\n\n" + "=" * 80)
print("PART 5 — SAMPLE ROWS FROM KEY TABLES (read directly from Parquet)")
print("=" * 80)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)

# Read each sample table fully into memory (small enough), then slice
def read_parquet_from_gcs(parquet_name):
    blob = bucket.blob(f"{LAHMAN_PARQUET_PREFIX}/{parquet_name}.parquet")
    data = blob.download_as_bytes()
    return pd.read_parquet(io.BytesIO(data))

# people: Ruth, Aaron, Judge
print("\n=== people (3 rows: Babe Ruth, Hank Aaron, Aaron Judge) ===")
people_full = read_parquet_from_gcs("People")
sample_pids = ['ruthba01', 'aaronha01', 'judgeaa01']
sample = people_full[people_full['playerID'].isin(sample_pids)]
print(sample.to_string(index=False))

# teams: 2025 Dodgers + Yankees, 1955 Brooklyn (franchID=LAD)
print("\n\n=== teams (3 rows: 2025 Dodgers, 2025 Yankees, 1955 Brooklyn) ===")
teams_full = read_parquet_from_gcs("Teams")
mask = ((teams_full['yearID'] == 2025) & (teams_full['teamID'].isin(['LAN', 'NYA']))) | \
       ((teams_full['yearID'] == 1955) & (teams_full['franchID'] == 'LAD'))
sample = teams_full[mask].sort_values('yearID')
print(sample.to_string(index=False))

# awards_players: Judge and Aaron
print("\n\n=== awards_players (Judge and Aaron) ===")
awards_full = read_parquet_from_gcs("AwardsPlayers")
sample = awards_full[awards_full['playerID'].isin(['judgeaa01', 'aaronha01'])].head(8)
print(sample.to_string(index=False))


print("\n\n" + "=" * 80)
print("PART 6 — DATASET SUMMARY")
print("=" * 80)
print(f"\nTotal Parquet rows: {sum(parquet_row_counts.values()):,}")
print(f"Total profile files: 24,473 (24,270 players + 203 teams)")
print(f"Bucket: gs://{GCS_BUCKET}/{GCS_PREFIX}/")

PART 1 — GCS BUCKET LAYOUT

Bucket: gs://class-demo/mlb-race-to-october/
  lahman-source/AllstarFull.csv       1 files      250.0 KB
  lahman-source/Appearances.csv       1 files       7.47 MB
  lahman-source/AwardsManagers.csv      1 files       10.7 KB
  lahman-source/AwardsPlayers.csv      1 files      576.8 KB
  lahman-source/AwardsShareManagers.csv      1 files       28.4 KB
  lahman-source/AwardsSharePlayers.csv      1 files      352.7 KB
  lahman-source/Batting.csv           1 files       7.63 MB
  lahman-source/BattingPost.csv       1 files       1.10 MB
  lahman-source/CollegePlaying.csv      1 files      419.3 KB
  lahman-source/Fielding.csv          1 files       8.36 MB
  lahman-source/FieldingOF.csv        1 files      291.5 KB
  lahman-source/FieldingOFsplit.csv      1 files       2.17 MB
  lahman-source/FieldingPost.csv      1 files      884.4 KB
  lahman-source/HallOfFame.csv        1 files      307.4 KB
  lahman-source/HomeGames.csv         1 files      173.9 KB
  lahm